# Qwen2.5-VL 3B ChartQA QLoRA active notebook

本 notebook 当前只保留：

1. **模块 1**：Colab 环境、GitHub 仓库、依赖和 Drive 初始化。
2. **模块 20**：补跑 full-val 3B baseline 与 `standard_numeric_final` prompt-only control，并与已训练 adapters 汇总。

历史模块 13-19 已剪切归档到旧 notebook：

```text
C:/Users/90553/Downloads/qwen25vl_chartqa_qlora.ipynb
```

复现与断线恢复约束：

- 每次新 Colab runtime 先运行模块 1：至少 `1.1 -> 1.3 -> 1.4`。
- 模块 20 自带 full-val 数据恢复、helper 生成、Drive 结果恢复，不依赖历史模块 13/19。
- 默认推理仍使用 4-bit NF4，与此前 3B/full-val 结果保持可比。
- 若 Colab 断线，重新运行模块 1 和 `20.1` 后，可直接继续 `20.2`、`20.3` 或 `20.4`。

最小运行路径：

- 环境恢复：`1.1 -> 1.3 -> 1.4`
- 补跑 full-val baseline：`20.1 -> 20.2`
- 补跑 full-val standard_numeric_final：`20.1 -> 20.3`
- 汇总当前已有 full-val 结果：`20.1 -> 20.4`


## 模块 1：环境与仓库初始化

**功能**：连接 GitHub 仓库，安装依赖，挂载 Google Drive，并可选检查 Colab GPU 环境。  
**依赖**：无，建议最先运行本模块。  
**代码块**：1.1 仓库克隆/更新；1.2 仓库位置检查；1.3 安装依赖；1.4 挂载 Drive；1.5 可选环境检查。


In [1]:
# 1.1 仓库克隆/更新：获取 GitHub 代码；如果仓库已存在则进入目录并拉取最新提交。
from pathlib import Path
from google.colab import userdata

REPO_DIR = Path("/content/qwen25vl-chartqa-qlora")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = (
    f"https://{GITHUB_TOKEN}@github.com/lsdlBlueDragon/qwen25vl-chartqa-qlora.git"
    if GITHUB_TOKEN
    else "https://github.com/lsdlBlueDragon/qwen25vl-chartqa-qlora.git"
)

if REPO_DIR.exists():
    %cd /content/qwen25vl-chartqa-qlora
    !git pull
else:
    !git clone {REPO_URL} /content/qwen25vl-chartqa-qlora
    %cd /content/qwen25vl-chartqa-qlora

Cloning into '/content/qwen25vl-chartqa-qlora'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 126 (delta 62), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 54.52 KiB | 4.19 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/qwen25vl-chartqa-qlora


In [2]:
# 1.2 仓库位置检查：确认当前工作目录和仓库文件是否正常。
%cd /content/qwen25vl-chartqa-qlora
!pwd
!ls

/content/qwen25vl-chartqa-qlora
/content/qwen25vl-chartqa-qlora
AGENTS.md  configs  docs       outputs	  requirements.txt  src
app	   data     notebooks  README.md  scripts	    tests


In [3]:
# 1.3 安装依赖：安装 requirements.txt 中的项目依赖。
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 47.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [4]:
# 1.4 挂载 Drive：连接 Google Drive 并设置项目 Drive 根目录。
from google.colab import drive
import os

MOUNT_POINT = "/content/drive"

# 清理可能残留的旧挂载状态，注意不要 rm -rf /content/drive
!fusermount -u /content/drive 2>/dev/null || true
!umount /content/drive 2>/dev/null || true
!mkdir -p /content/drive

drive.mount(
    MOUNT_POINT,
    force_remount=True,
    timeout_ms=300000  # 5分钟
)

PROJECT_DRIVE = "/content/drive/MyDrive/qwen25vl-chartqa-qlora"
os.makedirs(PROJECT_DRIVE, exist_ok=True)

print("Drive mounted.")
print("PROJECT_DRIVE =", PROJECT_DRIVE)

ValueError: mount failed

In [ ]:
# 1.5 可选环境检查：需要时检查 Colab GPU、Python 和核心依赖状态。
#!python scripts/env_check.py --output outputs/env_check_colab.json

## 模块 20：全量 val1920 baseline 与 prompt-only control 补跑

**目的**：补齐模块 19 缺失的 full-val 3B baseline，以及 `standard_numeric_final` prompt-only control。这样才能在全量验证集上严谨比较 baseline、默认 prompt adapter、numeric_final prompt adapter、hardmix 和 F。

**依赖**：新 Colab runtime 先运行模块 1：至少 `1.1 -> 1.3 -> 1.4`。本模块自带 full-val 数据恢复/导出和推理 helper，不依赖已归档到旧 notebook 的模块 13/19。

**默认推理设置**：与此前 3B/full-val 结果保持一致，默认 4-bit NF4 量化加载、`min_pixels=50176`、`max_pixels=401408`、greedy decoding。若要做非量化消融，可在 helper 命令里加 `--no-load-in-4bit`，但这会改变可比性且显存压力更高。

**输出位置**：

- 本地：`outputs/chartqa_3b_full_val`
- Drive：`/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val`

**最小运行顺序**：

1. `1.1 -> 1.3 -> 1.4`
2. `20.1`：恢复/构造 full-val 数据，并写入本模块 helper
3. `20.2`：补跑 full-val 3B baseline
4. `20.3`：补跑 full-val `standard_numeric_final`
5. `20.4`：汇总 baseline/control/adapters，并重算 oracle

**断线恢复**：20.1 会从 Drive 恢复 full-val JSONL/图片；20.2/20.3/20.4 会从 Drive 恢复已有 prediction、metrics、errors、evaluated、analysis。若 prediction 行数不是 1920，会视为不完整并重新推理。


In [ ]:
# 20.1 准备 full-val 数据与 baseline/control helper：可断线后重复运行。
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from collections import Counter
import json
import shutil
import subprocess
import sys

# 4-bit inference needs a recent bitsandbytes; qwen-vl-utils is needed by the helper.
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
], check=True)

import bitsandbytes as bnb
from qwen_vl_utils import process_vision_info

print("bitsandbytes:", getattr(bnb, "__version__", "unknown"))
print("qwen_vl_utils import OK:", process_vision_info)

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
LOCAL_PROCESSED = Path("data/processed")
DRIVE_PROCESSED = PROJECT_DRIVE / "data" / "processed"
FULL_VAL_TOTAL = 1920
FULL_VAL_JSONL = LOCAL_PROCESSED / "chartqa_val_full_sft_1920.jsonl"
FULL_VAL_IMAGE_DIR = LOCAL_PROCESSED / "chartqa_val_full_sft_1920_images"
DRIVE_FULL_VAL_JSONL = DRIVE_PROCESSED / FULL_VAL_JSONL.name
DRIVE_FULL_VAL_IMAGE_DIR = DRIVE_PROCESSED / FULL_VAL_IMAGE_DIR.name

LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)
DRIVE_PROCESSED.mkdir(parents=True, exist_ok=True)


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def restore_full_val_from_drive() -> None:
    if not FULL_VAL_JSONL.exists() and DRIVE_FULL_VAL_JSONL.exists():
        shutil.copy2(DRIVE_FULL_VAL_JSONL, FULL_VAL_JSONL)
        print("Restored full-val JSONL from Drive:", DRIVE_FULL_VAL_JSONL)
    local_image_count = len(list(FULL_VAL_IMAGE_DIR.glob("*.png"))) if FULL_VAL_IMAGE_DIR.exists() else 0
    if local_image_count < FULL_VAL_TOTAL and DRIVE_FULL_VAL_IMAGE_DIR.exists():
        FULL_VAL_IMAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DRIVE_FULL_VAL_IMAGE_DIR, FULL_VAL_IMAGE_DIR, dirs_exist_ok=True)
        print("Restored full-val images from Drive:", DRIVE_FULL_VAL_IMAGE_DIR)


def full_val_complete() -> bool:
    if not FULL_VAL_JSONL.exists() or not FULL_VAL_IMAGE_DIR.exists():
        return False
    rows = read_jsonl(FULL_VAL_JSONL)
    if len(rows) != FULL_VAL_TOTAL:
        print(f"Existing full-val JSONL has {len(rows)} rows, expected {FULL_VAL_TOTAL}.")
        return False
    missing = []
    for row in rows:
        image_path = FULL_VAL_JSONL.parent / row["image"]
        if not image_path.exists():
            missing.append(str(image_path))
            if len(missing) >= 5:
                break
    if missing:
        print("Missing image examples:", missing)
        return False
    return True


restore_full_val_from_drive()
if not full_val_complete():
    print("Full-val artifacts missing/incomplete; exporting ChartQA val1920...")
    subprocess.run([
        sys.executable,
        "scripts/prepare_chartqa_sft.py",
        "--split", "val",
        "--n-samples", str(FULL_VAL_TOTAL),
        "--output", str(FULL_VAL_JSONL),
        "--drive-output-dir", str(DRIVE_PROCESSED),
    ], check=True)

rows = read_jsonl(FULL_VAL_JSONL)
if len(rows) != FULL_VAL_TOTAL:
    raise RuntimeError(f"Expected {FULL_VAL_TOTAL} full-val records, got {len(rows)}")

image_count = len(list(FULL_VAL_IMAGE_DIR.glob("*.png")))
if image_count < FULL_VAL_TOTAL:
    raise FileNotFoundError(f"Expected at least {FULL_VAL_TOTAL} full-val images, got {image_count}")

print("full_val_jsonl:", FULL_VAL_JSONL)
print("records:", len(rows))
print("image_dir:", FULL_VAL_IMAGE_DIR, "images:", image_count)
print("human_or_machine_counts:", dict(Counter(row.get("human_or_machine") for row in rows)))

helper_path = Path("scripts/run_chartqa_full_val_controls.py")
helper_path.write_text(r"""
import argparse
import json
import shutil
import subprocess
import sys
import time
from collections import deque
from dataclasses import asdict
from pathlib import Path

from PIL import Image

PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eval_chartqa import exact_match
from src.infer import DEFAULT_MODEL_ID, GenerationConfig, InferenceConfig, load_model_and_processor


PROMPTS = {
    "default": (
        "Answer the chart question with a concise answer. "
        "If the answer is numeric, return only the number and unit when needed.\n"
        "Question: {question}"
    ),
    "numeric_final": (
        "Answer the chart question with only the final answer. "
        "If the answer is numeric, keep the chart scale and unit consistent with the question and axis labels. "
        "Do any counting or arithmetic internally, but do not output reasoning.\n"
        "Question: {question}"
    ),
}


RUNS = {
    "baseline_default": {
        "run_name": "chartqa_val_full_3b_baseline_default_1920",
        "adapter": None,
        "prompt": "default",
    },
    "standard_numeric_final": {
        "run_name": "chartqa_val_full_3b_standard_steps100_numeric_final_1920",
        "adapter": "chartqa_qlora_train1k_steps100",
        "prompt": "numeric_final",
    },
}


def parse_args():
    parser = argparse.ArgumentParser(description="Run full ChartQA val1920 baseline/control for Qwen2.5-VL-3B.")
    parser.add_argument("--data-jsonl", type=Path, required=True)
    parser.add_argument("--labels", nargs="+", required=True, choices=sorted(RUNS))
    parser.add_argument("--output-dir", type=Path, default=Path("outputs/chartqa_3b_full_val"))
    parser.add_argument("--drive-output-dir", type=Path, required=True)
    parser.add_argument("--project-drive", type=Path, required=True)
    parser.add_argument("--model-id", default=DEFAULT_MODEL_ID)
    parser.add_argument("--expected-total", type=int, default=1920)
    parser.add_argument("--device-map", default="auto")
    parser.add_argument("--torch-dtype", default="auto")
    parser.add_argument("--min-pixels", type=int, default=50176)
    parser.add_argument("--max-pixels", type=int, default=401408)
    parser.add_argument("--max-new-tokens", type=int, default=64)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--top-p", type=float, default=1.0)
    parser.add_argument("--load-in-4bit", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--force-infer", action="store_true")
    return parser.parse_args()


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def run_logged(command: list[str], label: str) -> None:
    print()
    print(f"===== {label} =====")
    print("command:", " ".join(str(part) for part in command))
    tail = deque(maxlen=160)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line)
    returncode = process.wait()
    if returncode != 0:
        print()
        print(f"FAILED: {label}, returncode={returncode}")
        print("last_log_lines:")
        print("".join(tail))
        raise subprocess.CalledProcessError(returncode, command)
    print(f"DONE: {label}")


def adapter_complete(path: Path) -> bool:
    return (
        (path / "adapter_config.json").exists()
        and ((path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists())
    )


def restore_adapter(project_drive: Path, adapter_name: str | None) -> Path | None:
    if adapter_name is None:
        return None
    local = Path("outputs/adapters") / adapter_name
    drive = project_drive / "outputs" / "adapters" / adapter_name
    if not adapter_complete(local) and adapter_complete(drive):
        local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive, local, dirs_exist_ok=True)
        print("Restored adapter from Drive:", drive)
    if not adapter_complete(local):
        raise FileNotFoundError(f"Missing adapter {adapter_name}. Expected local {local} or Drive {drive}.")
    return local


def restore_from_drive(local_path: Path, drive_dir: Path) -> None:
    drive_path = drive_dir / local_path.name
    if not local_path.exists() and drive_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
        print("Restored from Drive:", drive_path)


def copy_to_drive(local_path: Path, drive_dir: Path) -> None:
    if local_path.exists():
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, drive_dir / local_path.name)
        print("Copied to Drive:", drive_dir / local_path.name)


def prediction_complete(path: Path, expected_total: int) -> bool:
    count = len(read_jsonl(path))
    if count == expected_total:
        return True
    if path.exists():
        print(f"Prediction exists but incomplete: {path} has {count}, expected {expected_total}.")
    return False


def build_messages(image, question: str, prompt_name: str) -> list[dict]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPTS[prompt_name].format(question=question)},
            ],
        }
    ]


def predict_with_prompt(image, question, model, processor, generation_config, prompt_name):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = build_messages(image, question, prompt_name)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    generate_kwargs = asdict(generation_config)
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)
    latency = time.perf_counter() - start

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids, strict=True)
    ]
    answer = processor.batch_decode(
        generated_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
    return answer, round(latency, 4), generate_kwargs, PROMPTS[prompt_name].format(question=question)


def run_inference(args, label: str, predictions: Path, adapter_path: Path | None) -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Switch Colab runtime to GPU before running inference.")

    cfg = RUNS[label]
    rows = read_jsonl(args.data_jsonl)
    inference_config = InferenceConfig(
        model_id=args.model_id,
        adapter_path=str(adapter_path) if adapter_path is not None else None,
        load_in_4bit=args.load_in_4bit,
        device_map=args.device_map,
        torch_dtype=args.torch_dtype,
        min_pixels=args.min_pixels,
        max_pixels=args.max_pixels,
    )
    generation_config = GenerationConfig(
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_p=args.top_p,
        do_sample=args.temperature > 0,
    )

    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0))
    print("Model:", args.model_id)
    print("Adapter:", adapter_path)
    print("Prompt:", cfg["prompt"])
    print("load_in_4bit:", args.load_in_4bit)
    print("min_pixels/max_pixels:", args.min_pixels, args.max_pixels)
    model, processor = load_model_and_processor(inference_config)
    model.eval()

    out_rows = []
    base_dir = args.data_jsonl.parent
    start_all = time.perf_counter()
    for position, row in enumerate(rows):
        image_path = base_dir / row["image"]
        image = Image.open(image_path).convert("RGB")
        question = row["query"]
        reference_answer = row.get("answer")
        print(f"[{position + 1}/{len(rows)}] source_idx={row.get('sample_index')} {question}")
        answer, latency, generation, prompt_text = predict_with_prompt(
            image=image,
            question=question,
            model=model,
            processor=processor,
            generation_config=generation_config,
            prompt_name=cfg["prompt"],
        )
        is_exact = exact_match(answer, reference_answer)
        out_rows.append({
            "sample_index": row.get("sample_index", position),
            "selected_index": row.get("selected_index", position),
            "question": question,
            "answer": answer,
            "reference_answer": reference_answer,
            "all_labels": row.get("all_labels"),
            "human_or_machine": row.get("human_or_machine"),
            "split": row.get("split"),
            "exact_match": is_exact,
            "model_id": args.model_id,
            "adapter_path": str(adapter_path) if adapter_path is not None else None,
            "prompt_name": cfg["prompt"],
            "prompt_text": prompt_text,
            "image_path": str(image_path),
            "latency_seconds": latency,
            "generation": generation,
            "min_pixels": args.min_pixels,
            "max_pixels": args.max_pixels,
            "load_in_4bit": args.load_in_4bit,
        })
        print("Prediction:", answer)
        print("Reference:", reference_answer)
        print("Exact match:", is_exact)

    predictions.parent.mkdir(parents=True, exist_ok=True)
    with predictions.open("w", encoding="utf-8") as handle:
        for row in out_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    elapsed = time.perf_counter() - start_all
    exact_count = sum(1 for row in out_rows if row["exact_match"])
    print(f"Wrote {len(out_rows)} predictions to {predictions}")
    print(f"Exact match: {exact_count}/{len(out_rows)} = {exact_count / len(out_rows):.2%}")
    print(f"Total elapsed seconds: {elapsed:.2f}")


def run_one(args, label: str) -> dict:
    cfg = RUNS[label]
    args.output_dir.mkdir(parents=True, exist_ok=True)
    args.drive_output_dir.mkdir(parents=True, exist_ok=True)

    predictions = args.output_dir / f"{cfg['run_name']}.jsonl"
    metrics = args.output_dir / f"{cfg['run_name']}_metrics.json"
    errors = args.output_dir / f"{cfg['run_name']}_errors.jsonl"
    evaluated = args.output_dir / f"{cfg['run_name']}_evaluated.jsonl"
    analysis = args.output_dir / f"{cfg['run_name']}_error_analysis.json"

    for path in [predictions, metrics, errors, evaluated, analysis]:
        restore_from_drive(path, args.drive_output_dir)

    adapter_path = restore_adapter(args.project_drive, cfg["adapter"])

    if args.force_infer or not prediction_complete(predictions, args.expected_total):
        run_inference(args, label, predictions, adapter_path)
        copy_to_drive(predictions, args.drive_output_dir)
    else:
        print(f"Using complete existing prediction for {label}: {predictions}")

    if not prediction_complete(predictions, args.expected_total):
        raise RuntimeError(f"Prediction for {label} is incomplete after inference: {predictions}")

    eval_command = [
        sys.executable,
        "scripts/evaluate_predictions.py",
        "--predictions", str(predictions),
        "--metrics-output", str(metrics),
        "--errors-output", str(errors),
        "--evaluated-output", str(evaluated),
    ]
    run_logged(eval_command, f"evaluate {label}")

    analysis_command = [
        sys.executable,
        "scripts/analyze_chartqa_errors.py",
        "--errors", str(errors),
        "--output", str(analysis),
    ]
    run_logged(analysis_command, f"analyze {label}")

    for path in [metrics, errors, evaluated, analysis]:
        copy_to_drive(path, args.drive_output_dir)

    with metrics.open("r", encoding="utf-8") as handle:
        metric_payload = json.load(handle)

    return {
        "label": label,
        "run_name": cfg["run_name"],
        "adapter": cfg["adapter"],
        "prompt": cfg["prompt"],
        "total": metric_payload.get("total"),
        "exact_accuracy": metric_payload.get("exact_accuracy"),
        "relaxed_accuracy": metric_payload.get("relaxed_accuracy"),
        "metrics_path": str(metrics),
        "evaluated_path": str(evaluated),
        "load_in_4bit": args.load_in_4bit,
    }


def main() -> int:
    args = parse_args()
    if not args.data_jsonl.exists():
        raise FileNotFoundError(f"Missing full-val JSONL: {args.data_jsonl}. Run module 20.1 first.")
    data_count = len(read_jsonl(args.data_jsonl))
    if data_count != args.expected_total:
        raise RuntimeError(f"Expected {args.expected_total} records in {args.data_jsonl}, got {data_count}.")

    summaries = []
    for label in args.labels:
        summaries.append(run_one(args, label))

    status_path = args.output_dir / "chartqa_val_full_3b_controls_run_status.json"
    existing = {}
    if status_path.exists():
        with status_path.open("r", encoding="utf-8") as handle:
            existing = json.load(handle)
    runs = {item["label"]: item for item in existing.get("runs", [])}
    for item in summaries:
        runs[item["label"]] = item
    payload = {
        "data_jsonl": str(args.data_jsonl),
        "expected_total": args.expected_total,
        "runs": [runs[label] for label in sorted(runs)],
    }
    status_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    copy_to_drive(status_path, args.drive_output_dir)

    print()
    print("full_val_control_status:")
    for item in payload["runs"]:
        print(item["label"], {
            "total": item["total"],
            "exact": f"{item['exact_accuracy']:.2%}",
            "relaxed": f"{item['relaxed_accuracy']:.2%}",
            "load_in_4bit": item["load_in_4bit"],
        })
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
""", encoding="utf-8")

print("Wrote helper:", helper_path)
subprocess.run([sys.executable, str(helper_path), "--help"], check=True)


/content/qwen25vl-chartqa-qlora
bitsandbytes: 0.49.2
qwen_vl_utils import OK: <function process_vision_info at 0x78eb2c80fba0>
full_val_jsonl: data/processed/chartqa_val_full_sft_1920.jsonl
records: 1920
image_dir: data/processed/chartqa_val_full_sft_1920_images images: 1920
human_or_machine_counts: {0: 960, 1: 960}
Wrote helper: scripts/run_chartqa_full_val_controls.py


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_chartqa_full_val_controls.py', '--help'], returncode=0)

In [ ]:
# 20.2 补跑 full-val 3B baseline：无 adapter，default prompt。
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import subprocess
import sys

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
FULL_VAL_JSONL = Path("data/processed/chartqa_val_full_sft_1920.jsonl")
FULL_VAL_TOTAL = 1920

FORCE_FULL_VAL_BASELINE = False

command = [
    sys.executable,
    "scripts/run_chartqa_full_val_controls.py",
    "--data-jsonl", str(FULL_VAL_JSONL),
    "--labels", "baseline_default",
    "--project-drive", str(PROJECT_DRIVE),
    "--drive-output-dir", str(PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"),
    "--expected-total", str(FULL_VAL_TOTAL),
]
if FORCE_FULL_VAL_BASELINE:
    command.append("--force-infer")

print("run:", "baseline_default")
print("command:", " ".join(command))

subprocess.run(command, check=True)


/content/qwen25vl-chartqa-qlora
run: baseline_default
command: /usr/bin/python3 scripts/run_chartqa_full_val_controls.py --data-jsonl data/processed/chartqa_val_full_sft_1920.jsonl --labels baseline_default --project-drive /content/drive/MyDrive/qwen25vl-chartqa-qlora --drive-output-dir /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val --expected-total 1920


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_chartqa_full_val_controls.py', '--data-jsonl', 'data/processed/chartqa_val_full_sft_1920.jsonl', '--labels', 'baseline_default', '--project-drive', '/content/drive/MyDrive/qwen25vl-chartqa-qlora', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val', '--expected-total', '1920'], returncode=0)

In [ ]:
# 20.3 补跑 full-val standard_numeric_final：steps100 adapter + numeric_final prompt。
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import subprocess
import sys

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
FULL_VAL_JSONL = Path("data/processed/chartqa_val_full_sft_1920.jsonl")
FULL_VAL_TOTAL = 1920

FORCE_FULL_VAL_NUMERIC_FINAL = False

command = [
    sys.executable,
    "scripts/run_chartqa_full_val_controls.py",
    "--data-jsonl", str(FULL_VAL_JSONL),
    "--labels", "standard_numeric_final",
    "--project-drive", str(PROJECT_DRIVE),
    "--drive-output-dir", str(PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"),
    "--expected-total", str(FULL_VAL_TOTAL),
]
if FORCE_FULL_VAL_NUMERIC_FINAL:
    command.append("--force-infer")

print("run:", "standard_numeric_final")
print("command:", " ".join(command))
subprocess.run(command, check=True)


/content/qwen25vl-chartqa-qlora
run: standard_numeric_final
command: /usr/bin/python3 scripts/run_chartqa_full_val_controls.py --data-jsonl data/processed/chartqa_val_full_sft_1920.jsonl --labels standard_numeric_final --project-drive /content/drive/MyDrive/qwen25vl-chartqa-qlora --drive-output-dir /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val --expected-total 1920


CalledProcessError: Command '['/usr/bin/python3', 'scripts/run_chartqa_full_val_controls.py', '--data-jsonl', 'data/processed/chartqa_val_full_sft_1920.jsonl', '--labels', 'standard_numeric_final', '--project-drive', '/content/drive/MyDrive/qwen25vl-chartqa-qlora', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val', '--expected-total', '1920']' returned non-zero exit status 1.

In [ ]:
# 20.3a 诊断 standard_numeric_final 已生成文件：只读检查，不修改文件。
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import json
import os
import traceback

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

RUN_NAME = "chartqa_val_full_3b_standard_steps100_numeric_final_1920"
EXPECTED_TOTAL = 1920

paths = {
    "local_predictions": FULL_VAL_DIR / f"{RUN_NAME}.jsonl",
    "local_metrics": FULL_VAL_DIR / f"{RUN_NAME}_metrics.json",
    "local_errors": FULL_VAL_DIR / f"{RUN_NAME}_errors.jsonl",
    "local_evaluated": FULL_VAL_DIR / f"{RUN_NAME}_evaluated.jsonl",
    "local_analysis": FULL_VAL_DIR / f"{RUN_NAME}_error_analysis.json",
    "drive_predictions": DRIVE_FULL_VAL_DIR / f"{RUN_NAME}.jsonl",
    "drive_metrics": DRIVE_FULL_VAL_DIR / f"{RUN_NAME}_metrics.json",
    "drive_errors": DRIVE_FULL_VAL_DIR / f"{RUN_NAME}_errors.jsonl",
    "drive_evaluated": DRIVE_FULL_VAL_DIR / f"{RUN_NAME}_evaluated.jsonl",
    "drive_analysis": DRIVE_FULL_VAL_DIR / f"{RUN_NAME}_error_analysis.json",
}

def file_info(path: Path) -> dict:
    if not path.exists():
        return {"exists": False}
    stat = path.stat()
    return {
        "exists": True,
        "size_bytes": stat.st_size,
        "mtime": stat.st_mtime,
    }

def read_jsonl_safely(path: Path, max_bad: int = 5):
    rows = []
    bad = []
    if not path.exists():
        return rows, bad
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue
            try:
                rows.append(json.loads(raw))
            except Exception as exc:
                bad.append({
                    "line_no": line_no,
                    "error": repr(exc),
                    "preview": raw[:500],
                })
                if len(bad) >= max_bad:
                    break
    return rows, bad

print("=== file existence / size ===")
for name, path in paths.items():
    print(name, path, file_info(path))

print("\n=== predictions row counts ===")
for label in ["local_predictions", "drive_predictions"]:
    path = paths[label]
    rows, bad = read_jsonl_safely(path)
    print(label, {
        "path": str(path),
        "rows_readable": len(rows),
        "expected_total": EXPECTED_TOTAL,
        "complete": len(rows) == EXPECTED_TOTAL,
        "bad_json_lines": bad,
    })
    if rows:
        print("first_row_keys:", sorted(rows[0].keys()))
        print("first_sample_index:", rows[0].get("sample_index"))
        print("last_sample_index:", rows[-1].get("sample_index"))
        print("last_selected_index:", rows[-1].get("selected_index"))
        print("last_question:", rows[-1].get("question"))
        print("last_answer:", rows[-1].get("answer"))
        print("last_reference:", rows[-1].get("reference_answer"))
        print("last_latency:", rows[-1].get("latency_seconds"))

print("\n=== existing metrics payloads ===")
for label in ["local_metrics", "drive_metrics"]:
    path = paths[label]
    print(label, path)
    if path.exists():
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps({
                "total": payload.get("total"),
                "exact_match": payload.get("exact_match"),
                "exact_accuracy": payload.get("exact_accuracy"),
                "relaxed_correct": payload.get("relaxed_correct"),
                "relaxed_accuracy": payload.get("relaxed_accuracy"),
                "numeric_reference_total": payload.get("numeric_reference_total"),
                "relaxed_numeric_match": payload.get("relaxed_numeric_match"),
                "relaxed_numeric_accuracy_on_numeric": payload.get("relaxed_numeric_accuracy_on_numeric"),
                "by_human_or_machine": payload.get("by_human_or_machine"),
            }, ensure_ascii=False, indent=2))
        except Exception:
            traceback.print_exc()
    else:
        print("missing")

print("\n=== compare local vs drive prediction progress ===")
local_rows, local_bad = read_jsonl_safely(paths["local_predictions"])
drive_rows, drive_bad = read_jsonl_safely(paths["drive_predictions"])

print({
    "local_rows": len(local_rows),
    "drive_rows": len(drive_rows),
    "local_bad": local_bad,
    "drive_bad": drive_bad,
})

if local_rows and drive_rows:
    local_last = local_rows[-1].get("sample_index")
    drive_last = drive_rows[-1].get("sample_index")
    print({
        "local_last_sample_index": local_last,
        "drive_last_sample_index": drive_last,
        "same_row_count": len(local_rows) == len(drive_rows),
        "same_last_sample_index": local_last == drive_last,
    })

print("\n=== partial evaluation attempt on readable local predictions ===")
if paths["local_predictions"].exists() and local_rows:
    partial_path = FULL_VAL_DIR / f"{RUN_NAME}_partial_{len(local_rows)}.jsonl"
    partial_metrics = FULL_VAL_DIR / f"{RUN_NAME}_partial_{len(local_rows)}_metrics.json"
    partial_errors = FULL_VAL_DIR / f"{RUN_NAME}_partial_{len(local_rows)}_errors.jsonl"
    partial_evaluated = FULL_VAL_DIR / f"{RUN_NAME}_partial_{len(local_rows)}_evaluated.jsonl"

    # 为避免破坏原文件，只写一个 partial 副本用于评估。
    with partial_path.open("w", encoding="utf-8") as handle:
        for row in local_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("partial_path:", partial_path)
    print("partial_rows:", len(local_rows))

    import subprocess, sys
    try:
        subprocess.run([
            sys.executable,
            "scripts/evaluate_predictions.py",
            "--predictions", str(partial_path),
            "--metrics-output", str(partial_metrics),
            "--errors-output", str(partial_errors),
            "--evaluated-output", str(partial_evaluated),
        ], check=True)
        print("partial_metrics:")
        print(partial_metrics.read_text(encoding="utf-8"))
    except Exception:
        print("partial evaluation failed:")
        traceback.print_exc()
else:
    print("No readable local predictions; cannot do partial evaluation.")

/content/qwen25vl-chartqa-qlora
=== file existence / size ===
local_predictions outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl {'exists': False}
local_metrics outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920_metrics.json {'exists': False}
local_errors outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920_errors.jsonl {'exists': False}
local_evaluated outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920_evaluated.jsonl {'exists': False}
local_analysis outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920_error_analysis.json {'exists': False}
drive_predictions /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl {'exists': False}
drive_metrics /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3

In [ ]:
# 20.3b 抢救式继续 standard_numeric_final：增量写入 + 断点续跑 + 进度条 + 定期同步 Drive
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from dataclasses import asdict
import json
import shutil
import subprocess
import sys
import time
import traceback

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
    "tqdm",
], check=True)

from PIL import Image
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eval_chartqa import exact_match
from src.infer import DEFAULT_MODEL_ID, GenerationConfig, InferenceConfig, load_model_and_processor

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
FULL_VAL_JSONL = Path("data/processed/chartqa_val_full_sft_1920.jsonl")
FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

ADAPTER_NAME = "chartqa_qlora_train1k_steps100"
LOCAL_ADAPTER = Path("outputs/adapters") / ADAPTER_NAME
DRIVE_ADAPTER = PROJECT_DRIVE / "outputs" / "adapters" / ADAPTER_NAME

RUN_NAME = "chartqa_val_full_3b_standard_steps100_numeric_final_1920"
PREDICTIONS = FULL_VAL_DIR / f"{RUN_NAME}.jsonl"
METRICS = FULL_VAL_DIR / f"{RUN_NAME}_metrics.json"
ERRORS = FULL_VAL_DIR / f"{RUN_NAME}_errors.jsonl"
EVALUATED = FULL_VAL_DIR / f"{RUN_NAME}_evaluated.jsonl"
ANALYSIS = FULL_VAL_DIR / f"{RUN_NAME}_error_analysis.json"

EXPECTED_TOTAL = 1920
SAVE_EVERY = 25
FORCE_RESTART = False

PROMPT_TEMPLATE = (
    "Answer the chart question with only the final answer. "
    "If the answer is numeric, keep the chart scale and unit consistent with the question and axis labels. "
    "Do any counting or arithmetic internally, but do not output reasoning.\n"
    "Question: {question}"
)

FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path):
    rows = []
    bad = []
    if not path.exists():
        return rows, bad
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                bad.append((line_no, repr(exc), line[:300]))
                break
    return rows, bad

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

def copy_to_drive(path: Path):
    if path.exists():
        DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, DRIVE_FULL_VAL_DIR / path.name)
        print("Synced to Drive:", DRIVE_FULL_VAL_DIR / path.name)

def restore_from_drive(path: Path):
    drive_path = DRIVE_FULL_VAL_DIR / path.name
    if not path.exists() and drive_path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, path)
        print("Restored from Drive:", drive_path)

def adapter_complete(path: Path):
    return (
        (path / "adapter_config.json").exists()
        and ((path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists())
    )

def build_messages(image, question: str):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT_TEMPLATE.format(question=question)},
        ],
    }]

def predict_one(image, question, model, processor, generation_config):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = build_messages(image, question)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generate_kwargs = asdict(generation_config)
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)
    latency = time.perf_counter() - start

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids, strict=True)
    ]
    answer = processor.batch_decode(
        generated_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return answer, round(latency, 4), generate_kwargs, PROMPT_TEMPLATE.format(question=question)

if not FULL_VAL_JSONL.exists():
    raise FileNotFoundError(f"Missing {FULL_VAL_JSONL}. Run 20.1 first.")

rows, bad_rows = read_jsonl(FULL_VAL_JSONL)
if bad_rows:
    raise RuntimeError(f"Bad full-val JSONL line: {bad_rows[:1]}")
if len(rows) != EXPECTED_TOTAL:
    raise RuntimeError(f"Expected {EXPECTED_TOTAL} full-val rows, got {len(rows)}")

if not adapter_complete(LOCAL_ADAPTER) and adapter_complete(DRIVE_ADAPTER):
    LOCAL_ADAPTER.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_ADAPTER, LOCAL_ADAPTER, dirs_exist_ok=True)
    print("Restored adapter from Drive:", DRIVE_ADAPTER)

if not adapter_complete(LOCAL_ADAPTER):
    raise FileNotFoundError(f"Missing adapter: {LOCAL_ADAPTER} or {DRIVE_ADAPTER}")

if FORCE_RESTART and PREDICTIONS.exists():
    backup = PREDICTIONS.with_suffix(f".backup_{int(time.time())}.jsonl")
    shutil.copy2(PREDICTIONS, backup)
    print("Backed up existing predictions:", backup)
    PREDICTIONS.unlink()

restore_from_drive(PREDICTIONS)

existing, bad_existing = read_jsonl(PREDICTIONS)
if bad_existing:
    corrupt_backup = PREDICTIONS.with_suffix(f".corrupt_backup_{int(time.time())}.jsonl")
    shutil.copy2(PREDICTIONS, corrupt_backup)
    print("Existing prediction has bad JSON; backed up:", corrupt_backup)
    print("Bad line:", bad_existing[:1])
    write_jsonl(PREDICTIONS, existing)
    copy_to_drive(PREDICTIONS)

done = len(existing)
if done > EXPECTED_TOTAL:
    raise RuntimeError(f"Existing prediction has too many rows: {done}")

print({
    "run_name": RUN_NAME,
    "existing_rows": done,
    "expected_total": EXPECTED_TOTAL,
    "remaining": EXPECTED_TOTAL - done,
    "predictions": str(PREDICTIONS),
    "adapter": str(LOCAL_ADAPTER),
})

if done >= EXPECTED_TOTAL:
    print("Prediction already complete; skip inference.")
else:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Switch Colab runtime to GPU.")

    inference_config = InferenceConfig(
        model_id=DEFAULT_MODEL_ID,
        adapter_path=str(LOCAL_ADAPTER),
        load_in_4bit=True,
        device_map="auto",
        torch_dtype="auto",
        min_pixels=50176,
        max_pixels=401408,
    )
    generation_config = GenerationConfig(
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
        do_sample=False,
    )

    print("CUDA:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0))
    print("Loading model and adapter...")
    model, processor = load_model_and_processor(inference_config)
    model.eval()

    base_dir = FULL_VAL_JSONL.parent
    started = time.perf_counter()

    try:
        with PREDICTIONS.open("a", encoding="utf-8") as handle:
            progress = tqdm(
                range(done, EXPECTED_TOTAL),
                initial=done,
                total=EXPECTED_TOTAL,
                desc="standard_numeric_final full-val",
            )

            for position in progress:
                row = rows[position]
                image_path = base_dir / row["image"]
                image = Image.open(image_path).convert("RGB")
                question = row["query"]
                reference_answer = row.get("answer")

                answer, latency, generation, prompt_text = predict_one(
                    image=image,
                    question=question,
                    model=model,
                    processor=processor,
                    generation_config=generation_config,
                )

                is_exact = exact_match(answer, reference_answer)
                out_row = {
                    "sample_index": row.get("sample_index", position),
                    "selected_index": row.get("selected_index", position),
                    "question": question,
                    "answer": answer,
                    "reference_answer": reference_answer,
                    "all_labels": row.get("all_labels"),
                    "human_or_machine": row.get("human_or_machine"),
                    "split": row.get("split"),
                    "exact_match": is_exact,
                    "model_id": DEFAULT_MODEL_ID,
                    "adapter_path": str(LOCAL_ADAPTER),
                    "prompt_name": "numeric_final",
                    "prompt_text": prompt_text,
                    "image_path": str(image_path),
                    "latency_seconds": latency,
                    "generation": generation,
                    "min_pixels": 50176,
                    "max_pixels": 401408,
                    "load_in_4bit": True,
                }

                handle.write(json.dumps(out_row, ensure_ascii=False) + "\n")
                handle.flush()

                if hasattr(os, "fsync"):
                    os.fsync(handle.fileno())

                completed = position + 1
                progress.set_postfix({
                    "last_latency": latency,
                    "last_exact": is_exact,
                })

                if completed % SAVE_EVERY == 0:
                    copy_to_drive(PREDICTIONS)

    except Exception:
        print("Inference failed; syncing partial predictions before raising.")
        copy_to_drive(PREDICTIONS)
        traceback.print_exc()
        raise
    finally:
        copy_to_drive(PREDICTIONS)

    elapsed = time.perf_counter() - started
    print(f"Incremental inference finished/paused. elapsed_seconds={elapsed:.2f}")

final_rows, final_bad = read_jsonl(PREDICTIONS)
print({
    "final_readable_rows": len(final_rows),
    "bad_json_lines": final_bad,
    "complete": len(final_rows) == EXPECTED_TOTAL,
})

if len(final_rows) == EXPECTED_TOTAL and not final_bad:
    print("Running full evaluation and error analysis...")

    subprocess.run([
        sys.executable,
        "scripts/evaluate_predictions.py",
        "--predictions", str(PREDICTIONS),
        "--metrics-output", str(METRICS),
        "--errors-output", str(ERRORS),
        "--evaluated-output", str(EVALUATED),
    ], check=True)

    subprocess.run([
        sys.executable,
        "scripts/analyze_chartqa_errors.py",
        "--errors", str(ERRORS),
        "--output", str(ANALYSIS),
    ], check=True)

    for path in [PREDICTIONS, METRICS, ERRORS, EVALUATED, ANALYSIS]:
        copy_to_drive(path)

    print("metrics:")
    print(METRICS.read_text(encoding="utf-8"))
else:
    print("Prediction is still incomplete. Re-run this same 20.3b cell to continue from the saved row count.")

/content/qwen25vl-chartqa-qlora
{'run_name': 'chartqa_val_full_3b_standard_steps100_numeric_final_1920', 'existing_rows': 0, 'expected_total': 1920, 'remaining': 1920, 'predictions': 'outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl', 'adapter': 'outputs/adapters/chartqa_qlora_train1k_steps100'}
CUDA: True
GPU: Tesla T4
Loading model and adapter...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


standard_numeric_final full-val:   0%|          | 0/1920 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Synced to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chart

In [ ]:
# 20.3c 搜索 standard_steps100 adapter：诊断 adapter 实际位置与命名
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import json
import os
from tqdm.auto import tqdm

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
LOCAL_ROOT = Path(".")
EXPECTED_NAME = "chartqa_qlora_train1k_steps100"

SEARCH_ROOTS = [
    Path("outputs/adapters"),
    Path("outputs"),
    PROJECT_DRIVE / "outputs" / "adapters",
    PROJECT_DRIVE / "outputs",
]

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def adapter_info(path: Path) -> dict:
    info = {
        "path": str(path),
        "name": path.name,
        "complete": adapter_complete(path),
        "has_config": (path / "adapter_config.json").exists(),
        "has_safetensors": (path / "adapter_model.safetensors").exists(),
        "has_bin": (path / "adapter_model.bin").exists(),
        "files": [],
    }
    try:
        info["files"] = sorted(p.name for p in path.iterdir())[:20]
    except Exception as exc:
        info["files_error"] = repr(exc)

    cfg = path / "adapter_config.json"
    if cfg.exists():
        try:
            payload = json.loads(cfg.read_text(encoding="utf-8"))
            info["base_model_name_or_path"] = payload.get("base_model_name_or_path")
            info["r"] = payload.get("r")
            info["lora_alpha"] = payload.get("lora_alpha")
            info["target_modules"] = payload.get("target_modules")
        except Exception as exc:
            info["config_error"] = repr(exc)
    return info

def safe_walk_adapter_dirs(root: Path, max_dirs: int = 3000):
    found = []
    if not root.exists():
        return found

    stack = [root]
    seen = 0
    pbar = tqdm(desc=f"search {root}", total=max_dirs)

    while stack and seen < max_dirs:
        current = stack.pop()
        seen += 1
        pbar.update(1)

        if adapter_complete(current) or (current / "adapter_config.json").exists():
            found.append(current)

        try:
            children = [p for p in current.iterdir() if p.is_dir()]
        except Exception:
            children = []

        # 优先看名字像 adapter/checkpoint/steps 的目录，避免 Drive 深搜太慢。
        children.sort(
            key=lambda p: (
                0 if any(k in p.name.lower() for k in ["adapter", "checkpoint", "steps", "qlora", "chartqa"]) else 1,
                p.name.lower(),
            ),
            reverse=True,
        )
        stack.extend(children)

    pbar.close()
    return found

all_candidates = []
for root in SEARCH_ROOTS:
    print("\n=== search root ===")
    print(root, "exists:", root.exists())
    candidates = safe_walk_adapter_dirs(root)
    all_candidates.extend(candidates)

# 去重
unique = {}
for path in all_candidates:
    unique[str(path.resolve()) if path.exists() else str(path)] = path

infos = [adapter_info(path) for path in unique.values()]
complete_infos = [item for item in infos if item["complete"]]

def score_candidate(item: dict) -> int:
    text = item["path"].lower()
    score = 0
    if EXPECTED_NAME.lower() in text:
        score += 100
    if "train1k" in text:
        score += 30
    if "steps100" in text:
        score += 30
    if "standard" in text:
        score += 20
    if "calcnum" in text:
        score -= 25
    if "hardmix" in text:
        score -= 25
    if "steps200" in text:
        score -= 20
    if "steps250" in text:
        score -= 20
    if item.get("r") == 8:
        score += 3
    return score

ranked = sorted(complete_infos, key=score_candidate, reverse=True)

print("\n=== complete adapters found ===")
print("count:", len(complete_infos))
for item in ranked:
    print(json.dumps({
        "score": score_candidate(item),
        "path": item["path"],
        "name": item["name"],
        "r": item.get("r"),
        "lora_alpha": item.get("lora_alpha"),
        "base_model_name_or_path": item.get("base_model_name_or_path"),
        "files": item.get("files"),
    }, ensure_ascii=False, indent=2))

expected_local = Path("outputs/adapters") / EXPECTED_NAME
expected_drive = PROJECT_DRIVE / "outputs" / "adapters" / EXPECTED_NAME

print("\n=== expected exact paths ===")
for path in [expected_local, expected_drive]:
    print(path)
    print(json.dumps(adapter_info(path), ensure_ascii=False, indent=2))

print("\n=== recommendation ===")
if ranked:
    best = ranked[0]
    print("Best candidate:")
    print(best["path"])
    print("If this is the standard steps100 adapter, the next rescue cell should copy it to:")
    print(expected_local)
else:
    print("No complete adapter found under the searched project roots.")
    print("If Drive migration missed adapters, standard_numeric_final cannot run until the steps100 adapter is restored or retrained.")

/content/qwen25vl-chartqa-qlora

=== search root ===
outputs/adapters exists: True


search outputs/adapters:   0%|          | 0/3000 [00:00<?, ?it/s]


=== search root ===
outputs exists: True


search outputs:   0%|          | 0/3000 [00:00<?, ?it/s]


=== search root ===
/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters exists: False

=== search root ===
/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs exists: True


search /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs:   0%|          | 0/3000 [00:00<?, ?it/s]


=== complete adapters found ===
count: 0

=== expected exact paths ===
outputs/adapters/chartqa_qlora_train1k_steps100
{
  "path": "outputs/adapters/chartqa_qlora_train1k_steps100",
  "name": "chartqa_qlora_train1k_steps100",
  "complete": false,
  "has_config": false,
  "has_safetensors": false,
  "has_bin": false,
  "files": [],
  "files_error": "FileNotFoundError(2, 'No such file or directory')"
}
/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps100
{
  "path": "/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps100",
  "name": "chartqa_qlora_train1k_steps100",
  "complete": false,
  "has_config": false,
  "has_safetensors": false,
  "has_bin": false,
  "files": [],
  "files_error": "FileNotFoundError(2, 'No such file or directory')"
}

=== recommendation ===
No complete adapter found under the searched project roots.
If Drive migration missed adapters, standard_numeric_final cannot run until the steps1

In [ ]:
# 20.3d v3 从整个 Google Drive 拉取 standard steps100 adapter / train1k 数据
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import json
import shutil
import subprocess
import sys
from tqdm.auto import tqdm

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
    "tqdm",
], check=True)

DRIVE_ROOT = Path("/content/drive")
PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

ADAPTER_NAME = "chartqa_qlora_train1k_steps100"
LOCAL_ADAPTER = Path("outputs/adapters") / ADAPTER_NAME
STANDARD_DRIVE_ADAPTER = PROJECT_DRIVE / "outputs" / "adapters" / ADAPTER_NAME

LOCAL_PROCESSED = Path("data/processed")
TRAIN_JSONL = LOCAL_PROCESSED / "chartqa_train_sft_1000.jsonl"
TRAIN_IMAGE_DIR = LOCAL_PROCESSED / "chartqa_train_sft_1000_images"

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def copy_dir(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied dir:", src, "->", dst)

def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Copied file:", src, "->", dst)

def find_dirs_named(root: Path, name: str, max_hits=50):
    hits = []
    print(f"Searching dirs named {name} under {root} ...")
    for p in tqdm(root.rglob(name), desc=f"find {name}"):
        if p.is_dir():
            hits.append(p)
            print("found:", p, "complete:", adapter_complete(p))
            if len(hits) >= max_hits:
                break
    return hits

def find_files_named(root: Path, name: str, max_hits=50):
    hits = []
    print(f"Searching files named {name} under {root} ...")
    for p in tqdm(root.rglob(name), desc=f"find {name}"):
        if p.is_file():
            hits.append(p)
            print("found:", p)
            if len(hits) >= max_hits:
                break
    return hits

def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"count {path.name}"):
            if line.strip():
                n += 1
    return n

print("=== 1. restore adapter from Google Drive ===")
print("local adapter:", LOCAL_ADAPTER, "complete:", adapter_complete(LOCAL_ADAPTER))
print("standard drive adapter:", STANDARD_DRIVE_ADAPTER, "complete:", adapter_complete(STANDARD_DRIVE_ADAPTER))

if adapter_complete(LOCAL_ADAPTER):
    print("Local adapter already available.")
elif adapter_complete(STANDARD_DRIVE_ADAPTER):
    copy_dir(STANDARD_DRIVE_ADAPTER, LOCAL_ADAPTER)
else:
    adapter_hits = find_dirs_named(DRIVE_ROOT, ADAPTER_NAME)
    complete_hits = [p for p in adapter_hits if adapter_complete(p)]

    if complete_hits:
        src = complete_hits[0]
        copy_dir(src, LOCAL_ADAPTER)
        copy_dir(src, STANDARD_DRIVE_ADAPTER)
    else:
        print("No complete adapter found in Google Drive.")

print("adapter after restore:", LOCAL_ADAPTER, "complete:", adapter_complete(LOCAL_ADAPTER))

if adapter_complete(LOCAL_ADAPTER):
    print("\nREADY: adapter restored. Now rerun 20.3b.")
else:
    print("\n=== 2. adapter missing; restore train1k data from Google Drive ===")

    jsonl_hits = find_files_named(DRIVE_ROOT, "chartqa_train_sft_1000.jsonl")
    if jsonl_hits and not TRAIN_JSONL.exists():
        copy_file(jsonl_hits[0], TRAIN_JSONL)

    image_hits = find_dirs_named(DRIVE_ROOT, "chartqa_train_sft_1000_images")
    if image_hits and (not TRAIN_IMAGE_DIR.exists() or len(list(TRAIN_IMAGE_DIR.glob("*.png"))) < 1000):
        copy_dir(image_hits[0], TRAIN_IMAGE_DIR)

    train_count = count_jsonl(TRAIN_JSONL)
    image_count = len(list(TRAIN_IMAGE_DIR.glob("*.png"))) if TRAIN_IMAGE_DIR.exists() else 0

    print({
        "train_jsonl": str(TRAIN_JSONL),
        "train_jsonl_exists": TRAIN_JSONL.exists(),
        "train_count": train_count,
        "train_image_dir": str(TRAIN_IMAGE_DIR),
        "train_image_dir_exists": TRAIN_IMAGE_DIR.exists(),
        "train_image_count": image_count,
    })

    if train_count == 1000 and image_count >= 1000:
        print("\nTrain1k restored. If adapter really cannot be found, rerun/retrain standard steps100 from this restored data.")
        print("But first report this output back, especially whether adapter_hits/jsonl_hits/image_hits found anything.")
    else:
        raise RuntimeError(
            "Google Drive 中没有找到完整 standard steps100 adapter，也没有找到完整 train1k 数据。"
            "需要先把旧 Drive/旧账号里的 outputs/adapters 或 data/processed 迁移回来。"
        )

/content/qwen25vl-chartqa-qlora
=== 1. restore adapter from Google Drive ===
local adapter: outputs/adapters/chartqa_qlora_train1k_steps100 complete: False
standard drive adapter: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps100 complete: False
Searching dirs named chartqa_qlora_train1k_steps100 under /content/drive ...


find chartqa_qlora_train1k_steps100: 0it [00:00, ?it/s]

No complete adapter found in Google Drive.
adapter after restore: outputs/adapters/chartqa_qlora_train1k_steps100 complete: False

=== 2. adapter missing; restore train1k data from Google Drive ===
Searching files named chartqa_train_sft_1000.jsonl under /content/drive ...


find chartqa_train_sft_1000.jsonl: 0it [00:00, ?it/s]

found: /content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_train_sft_1000.jsonl
Searching dirs named chartqa_train_sft_1000_images under /content/drive ...


find chartqa_train_sft_1000_images: 0it [00:00, ?it/s]

found: /content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_train_sft_1000_images complete: False


count chartqa_train_sft_1000.jsonl: 0it [00:00, ?it/s]

{'train_jsonl': 'data/processed/chartqa_train_sft_1000.jsonl', 'train_jsonl_exists': True, 'train_count': 1000, 'train_image_dir': 'data/processed/chartqa_train_sft_1000_images', 'train_image_dir_exists': True, 'train_image_count': 1000}

Train1k restored. If adapter really cannot be found, rerun/retrain standard steps100 from this restored data.
But first report this output back, especially whether adapter_hits/jsonl_hits/image_hits found anything.


In [ ]:
# 20.3e 用已恢复 train1k 重训 standard steps100 adapter：不重新导出数据
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import os
import re
import shutil
import subprocess
import sys
import time
from tqdm.auto import tqdm

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
    "tqdm",
], check=True)

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

TRAIN_JSONL = Path("data/processed/chartqa_train_sft_1000.jsonl")
TRAIN_IMAGE_DIR = Path("data/processed/chartqa_train_sft_1000_images")

ADAPTER_NAME = "chartqa_qlora_train1k_steps100"
LOCAL_ADAPTER = Path("outputs/adapters") / ADAPTER_NAME
DRIVE_ADAPTER_DIR = PROJECT_DRIVE / "outputs" / "adapters"
DRIVE_ADAPTER = DRIVE_ADAPTER_DIR / ADAPTER_NAME

FORCE_RETRAIN = False
MAX_STEPS = 100

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def count_jsonl(path: Path) -> int:
    n = 0
    with path.open("r", encoding="utf-8") as handle:
        for line in tqdm(handle, desc=f"count {path.name}"):
            if line.strip():
                n += 1
    return n

def backup_incomplete_adapter(path: Path):
    if path.exists() and not adapter_complete(path):
        backup = path.with_name(f"{path.name}_incomplete_backup_{int(time.time())}")
        shutil.move(str(path), str(backup))
        print("Backed up incomplete adapter dir:", backup)

if not TRAIN_JSONL.exists():
    raise FileNotFoundError(f"Missing {TRAIN_JSONL}. Restore train1k first.")

if not TRAIN_IMAGE_DIR.exists():
    raise FileNotFoundError(f"Missing {TRAIN_IMAGE_DIR}. Restore train1k images first.")

train_count = count_jsonl(TRAIN_JSONL)
image_count = len(list(TRAIN_IMAGE_DIR.glob("*.png")))

print({
    "train_jsonl": str(TRAIN_JSONL),
    "train_count": train_count,
    "train_image_dir": str(TRAIN_IMAGE_DIR),
    "image_count": image_count,
    "local_adapter": str(LOCAL_ADAPTER),
    "drive_adapter": str(DRIVE_ADAPTER),
})

if train_count != 1000 or image_count < 1000:
    raise RuntimeError("train1k data is incomplete; do not train until data is restored.")

if adapter_complete(LOCAL_ADAPTER) and not FORCE_RETRAIN:
    print("Local adapter already complete; skip training:", LOCAL_ADAPTER)
elif adapter_complete(DRIVE_ADAPTER) and not FORCE_RETRAIN:
    LOCAL_ADAPTER.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_ADAPTER, LOCAL_ADAPTER, dirs_exist_ok=True)
    print("Restored adapter from Drive:", DRIVE_ADAPTER)
else:
    if FORCE_RETRAIN and LOCAL_ADAPTER.exists():
        backup = LOCAL_ADAPTER.with_name(f"{LOCAL_ADAPTER.name}_backup_{int(time.time())}")
        shutil.move(str(LOCAL_ADAPTER), str(backup))
        print("Backed up existing adapter before retrain:", backup)
    else:
        backup_incomplete_adapter(LOCAL_ADAPTER)

    LOCAL_ADAPTER.parent.mkdir(parents=True, exist_ok=True)
    DRIVE_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        "scripts/train_qwen25vl_qlora.py",
        "--train-jsonl", str(TRAIN_JSONL),
        "--output-dir", str(LOCAL_ADAPTER),
        "--drive-output-dir", str(DRIVE_ADAPTER_DIR),
        "--model-id", "Qwen/Qwen2.5-VL-3B-Instruct",
        "--max-steps", str(MAX_STEPS),
        "--per-device-train-batch-size", "1",
        "--gradient-accumulation-steps", "4",
        "--learning-rate", "2e-4",
        "--logging-steps", "1",
        "--save-steps", "0",
        "--seed", "42",
        "--min-pixels", "50176",
        "--max-pixels", "401408",
        "--lora-r", "8",
        "--lora-alpha", "16",
        "--lora-dropout", "0.05",
        "--load-in-4bit",
        "--gradient-checkpointing",
    ]

    print("command:", " ".join(command))

    env = os.environ.copy()
    env["WANDB_DISABLED"] = "true"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None

    progress = tqdm(total=MAX_STEPS, desc="train standard steps100")
    last_step = 0

    for line in process.stdout:
        print(line, end="")

        # Parse common Trainer/tqdm formats, e.g. " 12/100" or "{'step': 12}".
        step = None

        match = re.search(r"(\d+)\s*/\s*" + str(MAX_STEPS), line)
        if match:
            step = int(match.group(1))

        if step is None:
            match = re.search(r"['\"]step['\"]\s*:\s*(\d+)", line)
            if match:
                step = int(match.group(1))

        if step is not None and 0 <= step <= MAX_STEPS and step > last_step:
            progress.update(step - last_step)
            last_step = step

    returncode = process.wait()
    if last_step < MAX_STEPS and returncode == 0:
        progress.update(MAX_STEPS - last_step)
    progress.close()

    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

    if not adapter_complete(LOCAL_ADAPTER):
        raise RuntimeError(f"Training finished but adapter is incomplete: {LOCAL_ADAPTER}")

    DRIVE_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_ADAPTER, DRIVE_ADAPTER, dirs_exist_ok=True)
    print("Copied trained adapter to Drive:", DRIVE_ADAPTER)

print("\n=== final status ===")
print("local complete:", adapter_complete(LOCAL_ADAPTER), LOCAL_ADAPTER)
print("drive complete:", adapter_complete(DRIVE_ADAPTER), DRIVE_ADAPTER)

if not adapter_complete(LOCAL_ADAPTER):
    raise RuntimeError("Adapter is still incomplete.")

print("\nREADY: rerun 20.3b to continue standard_numeric_final full-val inference.")

/content/qwen25vl-chartqa-qlora


count chartqa_train_sft_1000.jsonl: 0it [00:00, ?it/s]

{'train_jsonl': 'data/processed/chartqa_train_sft_1000.jsonl', 'train_count': 1000, 'train_image_dir': 'data/processed/chartqa_train_sft_1000_images', 'image_count': 1000, 'local_adapter': 'outputs/adapters/chartqa_qlora_train1k_steps100', 'drive_adapter': '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps100'}
command: /usr/bin/python3 scripts/train_qwen25vl_qlora.py --train-jsonl data/processed/chartqa_train_sft_1000.jsonl --output-dir outputs/adapters/chartqa_qlora_train1k_steps100 --drive-output-dir /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters --model-id Qwen/Qwen2.5-VL-3B-Instruct --max-steps 100 --per-device-train-batch-size 1 --gradient-accumulation-steps 4 --learning-rate 2e-4 --logging-steps 1 --save-steps 0 --seed 42 --min-pixels 50176 --max-pixels 401408 --lora-r 8 --lora-alpha 16 --lora-dropout 0.05 --load-in-4bit --gradient-checkpointing


train standard steps100:   0%|          | 0/100 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-VL-3B-Instruct

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1847.71it/s]

Loading weights:   0%|          | 1/824 [00:02<32:59,  2.41s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 824/824 [00:29<00:00, 27.61it/s]
trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923
Starting QLoRA smoke training...

  0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior,

In [ ]:
# 20.4.0.1 恢复模块19 trained-adapter full-val 文件：供 20.4 完整汇总使用
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import shutil
from tqdm.auto import tqdm

DRIVE_ROOT = Path("/content/drive")
PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

LOCAL_FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

LOCAL_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAMES = {
    "standard_steps100": "chartqa_val_full_3b_standard_steps100_1920",
    "experiment_a_steps200": "chartqa_val_full_3b_steps200_1920",
    "experiment_b_calcnum": "chartqa_val_full_3b_calcnum1k_steps100_1920",
    "experiment_d_hardmix": "chartqa_val_full_3b_hardmix1k_steps100_1920",
    "experiment_f_steps250_r16a32": "chartqa_val_full_3b_steps250_r16a32_1920",
}

SUFFIXES = [
    ".jsonl",
    "_metrics.json",
    "_errors.jsonl",
    "_evaluated.jsonl",
    "_error_analysis.json",
]

EXPECTED_FILES = {
    label: [f"{run_name}{suffix}" for suffix in SUFFIXES]
    for label, run_name in RUN_NAMES.items()
}

def jsonl_count(path: Path) -> int:
    if not path.exists() or path.suffix != ".jsonl":
        return -1
    count = 0
    try:
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    count += 1
    except Exception:
        return -1
    return count

def file_ok(path: Path, filename: str) -> bool:
    if not path.exists():
        return False
    if filename.endswith("_metrics.json") or filename.endswith("_error_analysis.json"):
        return path.stat().st_size > 0
    if filename.endswith("_evaluated.jsonl"):
        return jsonl_count(path) == 1920
    if filename.endswith("_errors.jsonl"):
        # errors 数量不固定，只要求存在且可读；可能为 0 行但这里实际不会是 0。
        return jsonl_count(path) >= 0
    if filename.endswith(".jsonl"):
        return jsonl_count(path) == 1920
    return path.stat().st_size > 0

def find_file_in_drive(filename: str):
    # 优先标准项目目录，其次整个 Drive。
    priority_paths = [
        DRIVE_FULL_VAL_DIR / filename,
        PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val" / filename,
        PROJECT_DRIVE / "chartqa_3b_full_val" / filename,
        Path("/content/drive/MyDrive/outputs/chartqa_3b_full_val") / filename,
    ]

    for path in priority_paths:
        if file_ok(path, filename):
            return path

    hits = []
    for path in tqdm(DRIVE_ROOT.rglob(filename), desc=f"find {filename}"):
        if path.is_file() and file_ok(path, filename):
            hits.append(path)

    if not hits:
        return None

    # 偏好包含 chartqa_3b_full_val / qwen25vl-chartqa-qlora 的路径。
    def score(path: Path):
        text = str(path).lower()
        value = 0
        if "qwen25vl-chartqa-qlora" in text:
            value += 100
        if "chartqa_3b_full_val" in text:
            value += 50
        if "outputs" in text:
            value += 20
        if "trash" in text or ".shortcut-targets-by-id" in text:
            value -= 50
        return value

    hits.sort(key=score, reverse=True)
    return hits[0]

def restore_one(filename: str):
    local_path = LOCAL_FULL_VAL_DIR / filename
    drive_path = DRIVE_FULL_VAL_DIR / filename

    if file_ok(local_path, filename):
        if not file_ok(drive_path, filename):
            shutil.copy2(local_path, drive_path)
            print("Copied local -> standard Drive:", drive_path)
        return "local_ok", local_path

    if file_ok(drive_path, filename):
        shutil.copy2(drive_path, local_path)
        print("Restored standard Drive -> local:", drive_path)
        return "drive_ok", drive_path

    source = find_file_in_drive(filename)
    if source is None:
        return "missing", None

    shutil.copy2(source, local_path)
    shutil.copy2(source, drive_path)
    print("Restored found file:", source)
    print("  -> local:", local_path)
    print("  -> standard Drive:", drive_path)
    return "found", source

print("=== restore module19 trained-adapter full-val files ===")

summary = {}
for label, filenames in EXPECTED_FILES.items():
    print("\n---", label, "---")
    summary[label] = {}
    for filename in filenames:
        status, source = restore_one(filename)
        local_path = LOCAL_FULL_VAL_DIR / filename
        drive_path = DRIVE_FULL_VAL_DIR / filename
        summary[label][filename] = {
            "status": status,
            "source": str(source) if source else None,
            "local_ok": file_ok(local_path, filename),
            "drive_ok": file_ok(drive_path, filename),
            "local_count": jsonl_count(local_path) if filename.endswith(".jsonl") else None,
        }
        print(filename, summary[label][filename])

print("\n=== final availability for 20.4 ===")
missing = []
for label, filenames in EXPECTED_FILES.items():
    required_for_20_4 = [
        f"{RUN_NAMES[label]}_metrics.json",
        f"{RUN_NAMES[label]}_evaluated.jsonl",
    ]
    ok = True
    for filename in required_for_20_4:
        if not file_ok(LOCAL_FULL_VAL_DIR / filename, filename):
            ok = False
            missing.append((label, filename))
    print(label, "ready_for_20_4:", ok)

if missing:
    print("\nStill missing files needed by 20.4:")
    for item in missing:
        print(item)
    print("\nIf these remain missing, upload/copy the old module19 outputs folder into Drive and rerun this cell.")
else:
    print("\nREADY: all trained-adapter metrics/evaluated files are restored. Now rerun 20.4.")

/content/qwen25vl-chartqa-qlora
=== restore module19 trained-adapter full-val files ===

--- standard_steps100 ---


find chartqa_val_full_3b_standard_steps100_1920.jsonl: 0it [00:00, ?it/s]

chartqa_val_full_3b_standard_steps100_1920.jsonl {'status': 'missing', 'source': None, 'local_ok': False, 'drive_ok': False, 'local_count': -1}


find chartqa_val_full_3b_standard_steps100_1920_metrics.json: 0it [00:00, ?it/s]

chartqa_val_full_3b_standard_steps100_1920_metrics.json {'status': 'missing', 'source': None, 'local_ok': False, 'drive_ok': False, 'local_count': None}


...

chartqa_val_full_3b_steps250_r16a32_1920_errors.jsonl {'status': 'missing', 'source': None, 'local_ok': False, 'drive_ok': False, 'local_count': -1}


find chartqa_val_full_3b_steps250_r16a32_1920_evaluated.jsonl: 0it [00:00, ?it/s]

chartqa_val_full_3b_steps250_r16a32_1920_evaluated.jsonl {'status': 'missing', 'source': None, 'local_ok': False, 'drive_ok': False, 'local_count': -1}


find chartqa_val_full_3b_steps250_r16a32_1920_error_analysis.json: 0it [00:00, ?it/s]

chartqa_val_full_3b_steps250_r16a32_1920_error_analysis.json {'status': 'missing', 'source': None, 'local_ok': False, 'drive_ok': False, 'local_count': None}

=== final availability for 20.4 ===
standard_steps100 ready_for_20_4: False
experiment_a_steps200 ready_for_20_4: False
experiment_b_calcnum ready_for_20_4: False
experiment_d_hardmix ready_for_20_4: False
experiment_f_steps250_r16a32 ready_for_20_4: False

Still missing files needed by 20.4:
('standard_steps100', 'chartqa_val_full_3b_standard_steps100_1920_metrics.json')
('standard_steps100', 'chartqa_val_full_3b_standard_steps100_1920_evaluated.jsonl')
('experiment_a_steps200', 'chartqa_val_full_3b_steps200_1920_metrics.json')
('experiment_a_steps200', 'chartqa_val_full_3b_steps200_1920_evaluated.jsonl')
('experiment_b_calcnum', 'chartqa_val_full_3b_calcnum1k_steps100_1920_metrics.json')
('experiment_b_calcnum', 'chartqa_val_full_3b_calcnum1k_steps100_1920_evaluated.jsonl')
('experiment_d_hardmix', 'chartqa_val_full_3b_hardmix1

In [ ]:
# 20.4.0.2 重跑/恢复缺失的模块19 trained-adapter full-val 内容
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from dataclasses import asdict
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from tqdm.auto import tqdm

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
    "tqdm",
], check=True)

from PIL import Image

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eval_chartqa import exact_match
from src.infer import DEFAULT_MODEL_ID, GenerationConfig, InferenceConfig, load_model_and_processor

DRIVE_ROOT = Path("/content/drive")
PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

FULL_VAL_JSONL = Path("data/processed/chartqa_val_full_sft_1920.jsonl")
FULL_VAL_IMAGE_DIR = Path("data/processed/chartqa_val_full_sft_1920_images")
DRIVE_PROCESSED = PROJECT_DRIVE / "data" / "processed"

LOCAL_FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

LOCAL_ADAPTER_DIR = Path("outputs/adapters")
DRIVE_ADAPTER_DIR = PROJECT_DRIVE / "outputs" / "adapters"

EXPECTED_TOTAL = 1920
SAVE_EVERY = 25
FORCE_INFER = False

# 先建议只跑缺失项；如果要只跑某几个，填 label，例如 ["standard_steps100"]。
RUN_ONLY_LABELS = None

RUNS = {
    "standard_steps100": {
        "run_name": "chartqa_val_full_3b_standard_steps100_1920",
        "adapter": "chartqa_qlora_train1k_steps100",
    },
    "experiment_a_steps200": {
        "run_name": "chartqa_val_full_3b_steps200_1920",
        "adapter": "chartqa_qlora_train1k_steps200",
    },
    "experiment_b_calcnum": {
        "run_name": "chartqa_val_full_3b_calcnum1k_steps100_1920",
        "adapter": "chartqa_qlora_calcnum1k_steps100",
    },
    "experiment_d_hardmix": {
        "run_name": "chartqa_val_full_3b_hardmix1k_steps100_1920",
        "adapter": "chartqa_qlora_hardmix1k_steps100",
    },
    "experiment_f_steps250_r16a32": {
        "run_name": "chartqa_val_full_3b_steps250_r16a32_1920",
        "adapter": "chartqa_qlora_train1k_steps250_r16a32",
    },
}

PROMPT_TEMPLATE = (
    "Answer the chart question with a concise answer. "
    "If the answer is numeric, return only the number and unit when needed.\n"
    "Question: {question}"
)

LOCAL_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path):
    rows = []
    bad = []
    if not path.exists():
        return rows, bad
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                rows.append(json.loads(raw))
            except Exception as exc:
                bad.append((line_no, repr(exc), raw[:300]))
                break
    return rows, bad

def jsonl_count(path: Path) -> int:
    rows, bad = read_jsonl(path)
    if bad:
        return -1
    return len(rows)

def file_complete(path: Path, expected_total: int = EXPECTED_TOTAL) -> bool:
    return path.exists() and jsonl_count(path) == expected_total

def copy_to_drive(path: Path):
    if path.exists():
        DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, DRIVE_FULL_VAL_DIR / path.name)
        print("Copied to Drive:", DRIVE_FULL_VAL_DIR / path.name)

def restore_file_from_drive(path: Path):
    drive_path = DRIVE_FULL_VAL_DIR / path.name
    if not path.exists() and drive_path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, path)
        print("Restored file from standard Drive:", drive_path)

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def copy_adapter(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied adapter:", src, "->", dst)

def find_adapter_in_drive(adapter_name: str):
    exact_paths = [
        DRIVE_ADAPTER_DIR / adapter_name,
        PROJECT_DRIVE / "outputs" / "adapters" / adapter_name,
        PROJECT_DRIVE / "adapters" / adapter_name,
        Path("/content/drive/MyDrive") / adapter_name,
    ]

    for path in exact_paths:
        if adapter_complete(path):
            return path

    print("Searching adapter in Drive:", adapter_name)
    for path in tqdm(DRIVE_ROOT.rglob(adapter_name), desc=f"find {adapter_name}"):
        if path.is_dir() and adapter_complete(path):
            return path

    return None

def restore_adapter(adapter_name: str):
    local = LOCAL_ADAPTER_DIR / adapter_name
    standard_drive = DRIVE_ADAPTER_DIR / adapter_name

    if adapter_complete(local):
        return local

    if adapter_complete(standard_drive):
        copy_adapter(standard_drive, local)
        return local

    found = find_adapter_in_drive(adapter_name)
    if found is not None:
        copy_adapter(found, local)
        copy_adapter(found, standard_drive)
        return local

    return None

def restore_full_val_assets():
    drive_jsonl = DRIVE_PROCESSED / FULL_VAL_JSONL.name
    drive_image_dir = DRIVE_PROCESSED / FULL_VAL_IMAGE_DIR.name

    if not FULL_VAL_JSONL.exists() and drive_jsonl.exists():
        FULL_VAL_JSONL.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_jsonl, FULL_VAL_JSONL)
        print("Restored full-val JSONL:", drive_jsonl)

    if (
        not FULL_VAL_IMAGE_DIR.exists()
        or len(list(FULL_VAL_IMAGE_DIR.glob("*.png"))) < EXPECTED_TOTAL
    ) and drive_image_dir.exists():
        FULL_VAL_IMAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive_image_dir, FULL_VAL_IMAGE_DIR, dirs_exist_ok=True)
        print("Restored full-val images:", drive_image_dir)

    rows, bad = read_jsonl(FULL_VAL_JSONL)
    image_count = len(list(FULL_VAL_IMAGE_DIR.glob("*.png"))) if FULL_VAL_IMAGE_DIR.exists() else 0
    print({
        "full_val_jsonl": str(FULL_VAL_JSONL),
        "rows": len(rows),
        "bad": bad[:1],
        "image_dir": str(FULL_VAL_IMAGE_DIR),
        "image_count": image_count,
    })

    if bad or len(rows) != EXPECTED_TOTAL or image_count < EXPECTED_TOTAL:
        raise RuntimeError("Full-val assets are incomplete. Run 20.1 first.")
    return rows

def build_messages(image, question: str):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT_TEMPLATE.format(question=question)},
        ],
    }]

def predict_one(image, question, model, processor, generation_config):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = build_messages(image, question)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generate_kwargs = asdict(generation_config)
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)
    latency = time.perf_counter() - start

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids, strict=True)
    ]
    answer = processor.batch_decode(
        generated_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return answer, round(latency, 4), generate_kwargs, PROMPT_TEMPLATE.format(question=question)

def run_incremental_inference(label: str, cfg: dict, adapter_path: Path, rows: list[dict]):
    run_name = cfg["run_name"]
    predictions = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"

    restore_file_from_drive(predictions)

    existing, bad = read_jsonl(predictions)
    if bad:
        backup = predictions.with_name(f"{predictions.stem}_corrupt_backup_{int(time.time())}.jsonl")
        shutil.copy2(predictions, backup)
        print("Backed up corrupt predictions:", backup)
        existing = existing
        with predictions.open("w", encoding="utf-8") as handle:
            for row in existing:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    done = len(existing)
    if FORCE_INFER and predictions.exists():
        backup = predictions.with_name(f"{predictions.stem}_backup_{int(time.time())}.jsonl")
        shutil.copy2(predictions, backup)
        predictions.unlink()
        done = 0
        print("FORCE_INFER backup:", backup)

    print({
        "label": label,
        "run_name": run_name,
        "existing_rows": done,
        "remaining": EXPECTED_TOTAL - done,
        "predictions": str(predictions),
        "adapter": str(adapter_path),
    })

    if done >= EXPECTED_TOTAL:
        print("Prediction complete; skip inference:", predictions)
        copy_to_drive(predictions)
        return predictions

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Switch Colab runtime to GPU.")

    inference_config = InferenceConfig(
        model_id=DEFAULT_MODEL_ID,
        adapter_path=str(adapter_path),
        load_in_4bit=True,
        device_map="auto",
        torch_dtype="auto",
        min_pixels=50176,
        max_pixels=401408,
    )
    generation_config = GenerationConfig(
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
        do_sample=False,
    )

    print("Loading model for", label)
    print("GPU:", torch.cuda.get_device_name(0))
    model, processor = load_model_and_processor(inference_config)
    model.eval()

    base_dir = FULL_VAL_JSONL.parent

    try:
        with predictions.open("a", encoding="utf-8") as handle:
            progress = tqdm(
                range(done, EXPECTED_TOTAL),
                initial=done,
                total=EXPECTED_TOTAL,
                desc=f"{label} full-val",
            )

            for position in progress:
                row = rows[position]
                image_path = base_dir / row["image"]
                image = Image.open(image_path).convert("RGB")
                question = row["query"]
                reference_answer = row.get("answer")

                answer, latency, generation, prompt_text = predict_one(
                    image=image,
                    question=question,
                    model=model,
                    processor=processor,
                    generation_config=generation_config,
                )
                is_exact = exact_match(answer, reference_answer)

                out_row = {
                    "sample_index": row.get("sample_index", position),
                    "selected_index": row.get("selected_index", position),
                    "question": question,
                    "answer": answer,
                    "reference_answer": reference_answer,
                    "all_labels": row.get("all_labels"),
                    "human_or_machine": row.get("human_or_machine"),
                    "split": row.get("split"),
                    "exact_match": is_exact,
                    "model_id": DEFAULT_MODEL_ID,
                    "adapter_path": str(adapter_path),
                    "prompt_name": "default",
                    "prompt_text": prompt_text,
                    "image_path": str(image_path),
                    "latency_seconds": latency,
                    "generation": generation,
                    "min_pixels": 50176,
                    "max_pixels": 401408,
                    "load_in_4bit": True,
                }

                handle.write(json.dumps(out_row, ensure_ascii=False) + "\n")
                handle.flush()
                if hasattr(os, "fsync"):
                    os.fsync(handle.fileno())

                completed = position + 1
                progress.set_postfix({
                    "latency": latency,
                    "exact": is_exact,
                })

                if completed % SAVE_EVERY == 0:
                    copy_to_drive(predictions)

    except Exception:
        print("Inference failed; syncing partial predictions.")
        copy_to_drive(predictions)
        traceback.print_exc()
        raise
    finally:
        copy_to_drive(predictions)

    final_rows, final_bad = read_jsonl(predictions)
    if final_bad or len(final_rows) != EXPECTED_TOTAL:
        raise RuntimeError(f"Incomplete prediction after run: {predictions}, rows={len(final_rows)}, bad={final_bad[:1]}")

    return predictions

def evaluate_and_analyze(run_name: str):
    predictions = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"
    metrics = LOCAL_FULL_VAL_DIR / f"{run_name}_metrics.json"
    errors = LOCAL_FULL_VAL_DIR / f"{run_name}_errors.jsonl"
    evaluated = LOCAL_FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"
    analysis = LOCAL_FULL_VAL_DIR / f"{run_name}_error_analysis.json"

    for path in [metrics, errors, evaluated, analysis]:
        restore_file_from_drive(path)

    need_eval = (
        not metrics.exists()
        or not file_complete(evaluated, EXPECTED_TOTAL)
        or not errors.exists()
    )

    if need_eval:
        subprocess.run([
            sys.executable,
            "scripts/evaluate_predictions.py",
            "--predictions", str(predictions),
            "--metrics-output", str(metrics),
            "--errors-output", str(errors),
            "--evaluated-output", str(evaluated),
        ], check=True)
    else:
        print("Evaluation files already complete:", run_name)

    if not analysis.exists():
        subprocess.run([
            sys.executable,
            "scripts/analyze_chartqa_errors.py",
            "--errors", str(errors),
            "--output", str(analysis),
        ], check=True)
    else:
        print("Analysis file already exists:", analysis)

    for path in [predictions, metrics, errors, evaluated, analysis]:
        copy_to_drive(path)

    return {
        "metrics": metrics,
        "evaluated": evaluated,
        "analysis": analysis,
    }

rows = restore_full_val_assets()

labels = list(RUNS.keys())
if RUN_ONLY_LABELS is not None:
    labels = [label for label in labels if label in set(RUN_ONLY_LABELS)]

summary = {}
missing_adapters = []

for label in labels:
    cfg = RUNS[label]
    print("\n" + "=" * 80)
    print("RUN:", label, cfg)

    predictions = LOCAL_FULL_VAL_DIR / f"{cfg['run_name']}.jsonl"
    metrics = LOCAL_FULL_VAL_DIR / f"{cfg['run_name']}_metrics.json"
    evaluated = LOCAL_FULL_VAL_DIR / f"{cfg['run_name']}_evaluated.jsonl"

    restore_file_from_drive(predictions)
    restore_file_from_drive(metrics)
    restore_file_from_drive(evaluated)

    if file_complete(predictions) and metrics.exists() and file_complete(evaluated):
        print("Run already complete for 20.4:", label)
        summary[label] = "already_complete"
        continue

    adapter_path = restore_adapter(cfg["adapter"])
    if adapter_path is None:
        print("MISSING ADAPTER:", cfg["adapter"])
        missing_adapters.append((label, cfg["adapter"]))
        summary[label] = "missing_adapter"
        continue

    run_incremental_inference(label, cfg, adapter_path, rows)
    evaluate_and_analyze(cfg["run_name"])
    summary[label] = "completed"

print("\n=== 20.4.0.2 summary ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))

if missing_adapters:
    print("\nMissing adapters; these runs could not be rerun:")
    for label, adapter in missing_adapters:
        print(label, adapter)
    print("Restore/retrain these adapters, then rerun this same 20.4.0.2 cell.")
else:
    print("\nREADY: all available trained-adapter full-val files are restored/rerun. Now rerun 20.4.")

/content/qwen25vl-chartqa-qlora
Restored full-val JSONL: /content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_val_full_sft_1920.jsonl
Restored full-val images: /content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_val_full_sft_1920_images
{'full_val_jsonl': 'data/processed/chartqa_val_full_sft_1920.jsonl', 'rows': 1920, 'bad': [], 'image_dir': 'data/processed/chartqa_val_full_sft_1920_images', 'image_count': 1920}

RUN: standard_steps100 {'run_name': 'chartqa_val_full_3b_standard_steps100_1920', 'adapter': 'chartqa_qlora_train1k_steps100'}
Restored file from standard Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_1920.jsonl
Restored file from standard Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_1920_metrics.json
Restored file from standard Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_

In [ ]:
# 20.4.0.3 恢复或重训缺失 adapters：A/B/D/F
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from collections import Counter
import json
import os
import re
import shutil
import subprocess
import sys
import time
from tqdm.auto import tqdm

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1", "qwen-vl-utils", "tqdm"
], check=True)

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive")
LOCAL_PROCESSED = Path("data/processed")
DRIVE_PROCESSED = PROJECT_DRIVE / "data" / "processed"
LOCAL_ADAPTER_DIR = Path("outputs/adapters")
DRIVE_ADAPTER_DIR = PROJECT_DRIVE / "outputs" / "adapters"

LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)
DRIVE_PROCESSED.mkdir(parents=True, exist_ok=True)
LOCAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False
GENERATE_MISSING_DATASETS = True

MISSING_ADAPTER_RUNS = {
    "experiment_a_steps200": {
        "adapter": "chartqa_qlora_train1k_steps200",
        "train_jsonl": "chartqa_train_sft_1000.jsonl",
        "image_dir": "chartqa_train_sft_1000_images",
        "max_steps": 200,
        "lora_r": 8,
        "lora_alpha": 16,
    },
    "experiment_b_calcnum": {
        "adapter": "chartqa_qlora_calcnum1k_steps100",
        "train_jsonl": "chartqa_train_sft_calcnum_1000.jsonl",
        "image_dir": "chartqa_train_sft_calcnum_1000_images",
        "max_steps": 100,
        "lora_r": 8,
        "lora_alpha": 16,
    },
    "experiment_d_hardmix": {
        "adapter": "chartqa_qlora_hardmix1k_steps100",
        "train_jsonl": "chartqa_train_sft_hardmix_1000.jsonl",
        "image_dir": "chartqa_train_sft_hardmix_1000_images",
        "max_steps": 100,
        "lora_r": 8,
        "lora_alpha": 16,
    },
    "experiment_f_steps250_r16a32": {
        "adapter": "chartqa_qlora_train1k_steps250_r16a32",
        "train_jsonl": "chartqa_train_sft_1000.jsonl",
        "image_dir": "chartqa_train_sft_1000_images",
        "max_steps": 250,
        "lora_r": 16,
        "lora_alpha": 32,
    },
}

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    n = 0
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                n += 1
    return n

def image_count(path: Path) -> int:
    return len(list(path.glob("*.png"))) if path.exists() else 0

def copy_dir(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied dir:", src, "->", dst)

def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Copied file:", src, "->", dst)

def find_dir_in_drive(name: str):
    exact = [
        DRIVE_ADAPTER_DIR / name,
        PROJECT_DRIVE / "outputs" / "adapters" / name,
        PROJECT_DRIVE / "data" / "processed" / name,
        Path("/content/drive/MyDrive") / name,
    ]
    for path in exact:
        if path.exists():
            return path

    for path in tqdm(DRIVE_ROOT.rglob(name), desc=f"find dir {name}"):
        if path.is_dir():
            return path
    return None

def find_file_in_drive(name: str):
    exact = [
        DRIVE_PROCESSED / name,
        PROJECT_DRIVE / "data" / "processed" / name,
        Path("/content/drive/MyDrive") / name,
    ]
    for path in exact:
        if path.exists():
            return path

    for path in tqdm(DRIVE_ROOT.rglob(name), desc=f"find file {name}"):
        if path.is_file():
            return path
    return None

def restore_adapter(adapter_name: str) -> bool:
    local = LOCAL_ADAPTER_DIR / adapter_name
    drive = DRIVE_ADAPTER_DIR / adapter_name

    if adapter_complete(local):
        return True

    if adapter_complete(drive):
        copy_dir(drive, local)
        return True

    found = find_dir_in_drive(adapter_name)
    if found and adapter_complete(found):
        copy_dir(found, local)
        copy_dir(found, drive)
        return True

    return False

def restore_dataset(jsonl_name: str, image_dir_name: str) -> bool:
    local_jsonl = LOCAL_PROCESSED / jsonl_name
    local_images = LOCAL_PROCESSED / image_dir_name
    drive_jsonl = DRIVE_PROCESSED / jsonl_name
    drive_images = DRIVE_PROCESSED / image_dir_name

    if not local_jsonl.exists():
        src = drive_jsonl if drive_jsonl.exists() else find_file_in_drive(jsonl_name)
        if src:
            copy_file(src, local_jsonl)

    if image_count(local_images) < 1000:
        src = drive_images if drive_images.exists() else find_dir_in_drive(image_dir_name)
        if src:
            copy_dir(src, local_images)

    ok = count_jsonl(local_jsonl) == 1000 and image_count(local_images) >= 1000
    print({
        "jsonl": str(local_jsonl),
        "jsonl_count": count_jsonl(local_jsonl),
        "image_dir": str(local_images),
        "image_count": image_count(local_images),
        "ok": ok,
    })
    return ok

def generate_calcnum_dataset():
    from datasets import load_dataset

    dataset_name = "chartqa_train_sft_calcnum_1000"
    output_jsonl = LOCAL_PROCESSED / f"{dataset_name}.jsonl"
    image_dir = LOCAL_PROCESSED / f"{dataset_name}_images"

    target_count = 1000
    scan_limit = 12000
    keywords = [
        "sum", "total", "average", "mean", "median", "difference", "increase", "decrease",
        "changed", "change", "how much", "how many", "number of", "count", "times",
        "more than", "less than", "greater than", "lower than", "below", "above",
        "highest", "lowest", "maximum", "minimum", "peak", "percentage", "percent",
    ]

    def first_answer(labels):
        return str(labels[0]) if isinstance(labels, list) and labels else str(labels)

    def matched(question):
        q = str(question).lower()
        return [kw for kw in keywords if kw in q]

    def prompt(question):
        return (
            "Answer the chart question with a concise answer. "
            "If the answer is numeric, return only the number and unit when needed.\n"
            f"Question: {question}"
        )

    image_dir.mkdir(parents=True, exist_ok=True)
    records = []
    ds = load_dataset("HuggingFaceM4/ChartQA", split="train", streaming=True)

    for source_index, sample in tqdm(enumerate(ds), desc="generate calcnum", total=scan_limit):
        if source_index >= scan_limit or len(records) >= target_count:
            break
        reasons = matched(sample["query"])
        if not reasons:
            continue

        selected_index = len(records)
        image_name = f"train_calcnum_{selected_index:06d}.png"
        image_path = image_dir / image_name
        sample["image"].convert("RGB").save(image_path)
        image_rel = Path(image_dir.name) / image_name
        answer = first_answer(sample["label"])
        question = str(sample["query"])

        records.append({
            "sample_index": source_index,
            "selected_index": selected_index,
            "split": "train_calcnum_streaming",
            "human_or_machine": sample.get("human_or_machine"),
            "query": question,
            "answer": answer,
            "all_labels": sample["label"],
            "target_keywords": reasons,
            "image": image_rel.as_posix(),
            "messages": [
                {"role": "user", "content": [{"type": "image", "image": image_rel.as_posix()}, {"type": "text", "text": prompt(question)}]},
                {"role": "assistant", "content": [{"type": "text", "text": answer}]},
            ],
        })

    if len(records) != target_count:
        raise RuntimeError(f"calcnum generated {len(records)}, expected {target_count}")

    with output_jsonl.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    copy_file(output_jsonl, DRIVE_PROCESSED / output_jsonl.name)
    copy_dir(image_dir, DRIVE_PROCESSED / image_dir.name)

def generate_hardmix_dataset():
    from datasets import load_dataset

    dataset_name = "chartqa_train_sft_hardmix_1000"
    output_jsonl = LOCAL_PROCESSED / f"{dataset_name}.jsonl"
    image_dir = LOCAL_PROCESSED / f"{dataset_name}_images"

    scan_limit = 25000
    quotas = {
        "calculation": 400,
        "numeric_scale": 250,
        "color_legend": 200,
        "date_axis": 150,
    }
    keywords = {
        "calculation": [
            "sum", "total", "average", "mean", "median", "difference", "increase", "decrease",
            "how much", "how many", "number of", "count", "times", "greater than", "lower than",
            "more than", "less than", "below", "above", "highest", "lowest", "maximum", "minimum", "peak",
        ],
        "numeric_scale": [
            "value", "values", "percent", "percentage", "%", "rate", "ratio", "million", "billion",
            "dependents", "population", "amount", "score",
        ],
        "color_legend": [
            "color", "colour", "legend", "represent", "line", "bar", "segment", "graph", "dark", "light",
            "blue", "green", "red", "yellow", "grey", "gray",
        ],
        "date_axis": [
            "year", "month", "date", "when", "jan", "feb", "mar", "apr", "may", "jun", "jul",
            "aug", "sep", "sept", "oct", "nov", "dec", "axis",
        ],
    }

    def first_answer(labels):
        return str(labels[0]) if isinstance(labels, list) and labels else str(labels)

    def matched_buckets(question):
        q = str(question).lower()
        result = []
        for bucket, kws in keywords.items():
            hits = [kw for kw in kws if kw in q]
            if hits:
                result.append((bucket, hits))
        return result

    def choose_bucket(matches, counts):
        candidates = []
        for bucket, hits in matches:
            if counts[bucket] < quotas[bucket]:
                candidates.append((counts[bucket] / quotas[bucket], bucket, hits))
        if not candidates:
            return None, []
        _, bucket, hits = sorted(candidates, key=lambda x: (x[0], x[1]))[0]
        return bucket, hits

    def prompt(question):
        return (
            "Answer the chart question with a concise answer. "
            "If the answer is numeric, return only the number and unit when needed.\n"
            f"Question: {question}"
        )

    image_dir.mkdir(parents=True, exist_ok=True)
    records = []
    counts = Counter()
    ds = load_dataset("HuggingFaceM4/ChartQA", split="train", streaming=True)

    for source_index, sample in tqdm(enumerate(ds), desc="generate hardmix", total=scan_limit):
        if source_index >= scan_limit or all(counts[b] >= q for b, q in quotas.items()):
            break
        matches = matched_buckets(sample["query"])
        if not matches:
            continue

        bucket, hits = choose_bucket(matches, counts)
        if bucket is None:
            continue

        selected_index = len(records)
        image_name = f"train_hardmix_{selected_index:06d}.png"
        image_path = image_dir / image_name
        sample["image"].convert("RGB").save(image_path)
        image_rel = Path(image_dir.name) / image_name
        answer = first_answer(sample["label"])
        question = str(sample["query"])

        records.append({
            "sample_index": source_index,
            "selected_index": selected_index,
            "split": "train_hardmix_streaming",
            "human_or_machine": sample.get("human_or_machine"),
            "hardmix_bucket": bucket,
            "hardmix_keywords": hits,
            "query": question,
            "answer": answer,
            "all_labels": sample["label"],
            "image": image_rel.as_posix(),
            "messages": [
                {"role": "user", "content": [{"type": "image", "image": image_rel.as_posix()}, {"type": "text", "text": prompt(question)}]},
                {"role": "assistant", "content": [{"type": "text", "text": answer}]},
            ],
        })
        counts[bucket] += 1

    missing = {b: q - counts[b] for b, q in quotas.items() if counts[b] < q}
    if missing:
        raise RuntimeError(f"hardmix quotas not filled: {missing}")

    with output_jsonl.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    copy_file(output_jsonl, DRIVE_PROCESSED / output_jsonl.name)
    copy_dir(image_dir, DRIVE_PROCESSED / image_dir.name)

def ensure_dataset_for_run(label: str, cfg: dict):
    ok = restore_dataset(cfg["train_jsonl"], cfg["image_dir"])
    if ok:
        return

    if not GENERATE_MISSING_DATASETS:
        raise RuntimeError(f"Missing dataset for {label}: {cfg['train_jsonl']}")

    if label == "experiment_b_calcnum":
        generate_calcnum_dataset()
    elif label == "experiment_d_hardmix":
        generate_hardmix_dataset()
    else:
        raise RuntimeError(f"Missing required dataset for {label}: {cfg['train_jsonl']}")

    ok = restore_dataset(cfg["train_jsonl"], cfg["image_dir"])
    if not ok:
        raise RuntimeError(f"Dataset still incomplete after generation: {label}")

def train_adapter(label: str, cfg: dict):
    adapter_name = cfg["adapter"]
    local_adapter = LOCAL_ADAPTER_DIR / adapter_name
    drive_adapter = DRIVE_ADAPTER_DIR / adapter_name

    if adapter_complete(local_adapter) and not FORCE_RETRAIN:
        print("Adapter already local:", local_adapter)
        if not adapter_complete(drive_adapter):
            copy_dir(local_adapter, drive_adapter)
        return

    if adapter_complete(drive_adapter) and not FORCE_RETRAIN:
        copy_dir(drive_adapter, local_adapter)
        return

    found = find_dir_in_drive(adapter_name)
    if found and adapter_complete(found) and not FORCE_RETRAIN:
        copy_dir(found, local_adapter)
        copy_dir(found, drive_adapter)
        return

    ensure_dataset_for_run(label, cfg)

    if local_adapter.exists() and not adapter_complete(local_adapter):
        backup = local_adapter.with_name(f"{local_adapter.name}_incomplete_backup_{int(time.time())}")
        shutil.move(str(local_adapter), str(backup))
        print("Backed up incomplete adapter:", backup)

    command = [
        sys.executable,
        "scripts/train_qwen25vl_qlora.py",
        "--train-jsonl", str(LOCAL_PROCESSED / cfg["train_jsonl"]),
        "--output-dir", str(local_adapter),
        "--drive-output-dir", str(DRIVE_ADAPTER_DIR),
        "--model-id", "Qwen/Qwen2.5-VL-3B-Instruct",
        "--max-steps", str(cfg["max_steps"]),
        "--per-device-train-batch-size", "1",
        "--gradient-accumulation-steps", "4",
        "--learning-rate", "2e-4",
        "--logging-steps", "1",
        "--save-steps", "0",
        "--seed", "42",
        "--min-pixels", "50176",
        "--max-pixels", "401408",
        "--lora-r", str(cfg["lora_r"]),
        "--lora-alpha", str(cfg["lora_alpha"]),
        "--lora-dropout", "0.05",
        "--load-in-4bit",
        "--gradient-checkpointing",
    ]

    print("\nTRAIN:", label)
    print("command:", " ".join(command))

    env = os.environ.copy()
    env["WANDB_DISABLED"] = "true"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None

    pbar = tqdm(total=cfg["max_steps"], desc=f"train {label}")
    last_step = 0

    for line in process.stdout:
        print(line, end="")

        step = None
        m = re.search(r"(\\d+)\\s*/\\s*" + str(cfg["max_steps"]), line)
        if m:
            step = int(m.group(1))
        if step is None:
            m = re.search(r"['\\\"]step['\\\"]\\s*:\\s*(\\d+)", line)
            if m:
                step = int(m.group(1))

        if step is not None and 0 <= step <= cfg["max_steps"] and step > last_step:
            pbar.update(step - last_step)
            last_step = step

    returncode = process.wait()
    if returncode == 0 and last_step < cfg["max_steps"]:
        pbar.update(cfg["max_steps"] - last_step)
    pbar.close()

    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

    if not adapter_complete(local_adapter):
        raise RuntimeError(f"Training finished but adapter incomplete: {local_adapter}")

    copy_dir(local_adapter, drive_adapter)

labels = list(MISSING_ADAPTER_RUNS.keys())

for label in labels:
    print("\n" + "=" * 80)
    print("ADAPTER:", label, MISSING_ADAPTER_RUNS[label])
    train_adapter(label, MISSING_ADAPTER_RUNS[label])

print("\n=== final adapter status ===")
all_ok = True
for label, cfg in MISSING_ADAPTER_RUNS.items():
    local = LOCAL_ADAPTER_DIR / cfg["adapter"]
    drive = DRIVE_ADAPTER_DIR / cfg["adapter"]
    ok = adapter_complete(local) and adapter_complete(drive)
    all_ok = all_ok and ok
    print(label, {"local": str(local), "local_ok": adapter_complete(local), "drive": str(drive), "drive_ok": adapter_complete(drive)})

if not all_ok:
    raise RuntimeError("Some adapters are still missing.")
print("\nREADY: rerun 20.4.0.2 to generate missing full-val predictions/metrics/evaluated.")

/content/qwen25vl-chartqa-qlora

ADAPTER: experiment_a_steps200 {'adapter': 'chartqa_qlora_train1k_steps200', 'train_jsonl': 'chartqa_train_sft_1000.jsonl', 'image_dir': 'chartqa_train_sft_1000_images', 'max_steps': 200, 'lora_r': 8, 'lora_alpha': 16}
Copied dir: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps200 -> outputs/adapters/chartqa_qlora_train1k_steps200

ADAPTER: experiment_b_calcnum {'adapter': 'chartqa_qlora_calcnum1k_steps100', 'train_jsonl': 'chartqa_train_sft_calcnum_1000.jsonl', 'image_dir': 'chartqa_train_sft_calcnum_1000_images', 'max_steps': 100, 'lora_r': 8, 'lora_alpha': 16}
Copied dir: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_calcnum1k_steps100 -> outputs/adapters/chartqa_qlora_calcnum1k_steps100

ADAPTER: experiment_d_hardmix {'adapter': 'chartqa_qlora_hardmix1k_steps100', 'train_jsonl': 'chartqa_train_sft_hardmix_1000.jsonl', 'image_dir': 'chartqa_train_sft_hardmix_1000_images', 'max_

In [ ]:
# 20.4.0.4 恢复 baseline/control full-val 文件：供 20.4 完整汇总
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
import shutil
import subprocess
import sys
from tqdm.auto import tqdm

DRIVE_ROOT = Path("/content/drive")
PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

LOCAL_FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

LOCAL_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    "baseline_default": "chartqa_val_full_3b_baseline_default_1920",
    "standard_numeric_final": "chartqa_val_full_3b_standard_steps100_numeric_final_1920",
}

SUFFIXES = [
    ".jsonl",
    "_metrics.json",
    "_errors.jsonl",
    "_evaluated.jsonl",
    "_error_analysis.json",
]

def jsonl_count(path: Path) -> int:
    if not path.exists() or not path.suffix == ".jsonl":
        return -1
    count = 0
    try:
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    count += 1
    except Exception:
        return -1
    return count

def file_ok(path: Path, filename: str) -> bool:
    if not path.exists():
        return False
    if filename.endswith(".jsonl"):
        if filename.endswith("_errors.jsonl"):
            return jsonl_count(path) >= 0
        return jsonl_count(path) == 1920
    return path.stat().st_size > 0

def find_file(filename: str):
    priority = [
        DRIVE_FULL_VAL_DIR / filename,
        PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val" / filename,
        Path("/content/drive/MyDrive") / "chartqa_3b_full_val" / filename,
    ]
    for path in priority:
        if file_ok(path, filename):
            return path

    hits = []
    for path in tqdm(DRIVE_ROOT.rglob(filename), desc=f"find {filename}"):
        if path.is_file() and file_ok(path, filename):
            hits.append(path)

    if not hits:
        return None

    def score(path: Path):
        text = str(path).lower()
        s = 0
        if "qwen25vl-chartqa-qlora" in text:
            s += 100
        if "chartqa_3b_full_val" in text:
            s += 50
        if "outputs" in text:
            s += 20
        return s

    hits.sort(key=score, reverse=True)
    return hits[0]

def restore_one(filename: str):
    local = LOCAL_FULL_VAL_DIR / filename
    drive = DRIVE_FULL_VAL_DIR / filename

    if file_ok(local, filename):
        if not file_ok(drive, filename):
            shutil.copy2(local, drive)
            print("Copied local -> Drive:", drive)
        return True

    if file_ok(drive, filename):
        shutil.copy2(drive, local)
        print("Restored Drive -> local:", drive)
        return True

    source = find_file(filename)
    if source:
        shutil.copy2(source, local)
        shutil.copy2(source, drive)
        print("Restored found file:", source)
        return True

    return False

def ensure_eval(run_name: str):
    pred = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"
    metrics = LOCAL_FULL_VAL_DIR / f"{run_name}_metrics.json"
    errors = LOCAL_FULL_VAL_DIR / f"{run_name}_errors.jsonl"
    evaluated = LOCAL_FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"
    analysis = LOCAL_FULL_VAL_DIR / f"{run_name}_error_analysis.json"

    if not file_ok(pred, pred.name):
        return False

    if not metrics.exists() or not file_ok(evaluated, evaluated.name) or not errors.exists():
        subprocess.run([
            sys.executable,
            "scripts/evaluate_predictions.py",
            "--predictions", str(pred),
            "--metrics-output", str(metrics),
            "--errors-output", str(errors),
            "--evaluated-output", str(evaluated),
        ], check=True)

    if not analysis.exists():
        subprocess.run([
            sys.executable,
            "scripts/analyze_chartqa_errors.py",
            "--errors", str(errors),
            "--output", str(analysis),
        ], check=True)

    for path in [pred, metrics, errors, evaluated, analysis]:
        if path.exists():
            shutil.copy2(path, DRIVE_FULL_VAL_DIR / path.name)
            print("Synced:", DRIVE_FULL_VAL_DIR / path.name)

    return True

print("=== restore baseline/control files ===")
missing = []

for label, run_name in RUNS.items():
    print("\n---", label, run_name, "---")
    for suffix in SUFFIXES:
        filename = f"{run_name}{suffix}"
        ok = restore_one(filename)
        print(filename, "ok:", ok)
    if not ensure_eval(run_name):
        missing.append(label)

print("\n=== final check ===")
for label, run_name in RUNS.items():
    metrics = LOCAL_FULL_VAL_DIR / f"{run_name}_metrics.json"
    evaluated = LOCAL_FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"
    pred = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"
    ready = metrics.exists() and file_ok(evaluated, evaluated.name) and file_ok(pred, pred.name)
    print(label, {
        "prediction_rows": jsonl_count(pred),
        "metrics_exists": metrics.exists(),
        "evaluated_rows": jsonl_count(evaluated),
        "ready_for_20_4": ready,
    })

if missing:
    print("\nStill missing prediction files for:", missing)
    print("If baseline_default is missing, rerun 20.2.")
    print("If standard_numeric_final is missing, rerun 20.3b.")
else:
    print("\nREADY: baseline/control restored. Now rerun 20.4.")

/content/qwen25vl-chartqa-qlora
=== restore baseline/control files ===

--- baseline_default chartqa_val_full_3b_baseline_default_1920 ---


find chartqa_val_full_3b_baseline_default_1920.jsonl: 0it [00:00, ?it/s]

chartqa_val_full_3b_baseline_default_1920.jsonl ok: False


find chartqa_val_full_3b_baseline_default_1920_metrics.json: 0it [00:00, ?it/s]

chartqa_val_full_3b_baseline_default_1920_metrics.json ok: False


...

chartqa_val_full_3b_standard_steps100_numeric_final_1920_errors.jsonl ok: False


find chartqa_val_full_3b_standard_steps100_numeric_final_1920_evaluated.jsonl: 0it [00:00, ?it/s]

chartqa_val_full_3b_standard_steps100_numeric_final_1920_evaluated.jsonl ok: False


find chartqa_val_full_3b_standard_steps100_numeric_final_1920_error_analysis.json: 0it [00:00, ?it/s]

chartqa_val_full_3b_standard_steps100_numeric_final_1920_error_analysis.json ok: False

=== final check ===
baseline_default {'prediction_rows': -1, 'metrics_exists': False, 'evaluated_rows': -1, 'ready_for_20_4': False}
standard_numeric_final {'prediction_rows': -1, 'metrics_exists': False, 'evaluated_rows': -1, 'ready_for_20_4': False}

Still missing prediction files for: ['baseline_default', 'standard_numeric_final']
If baseline_default is missing, rerun 20.2.
If standard_numeric_final is missing, rerun 20.3b.


In [ ]:
# 20.4.0.5 重跑缺失 baseline/control：增量写入 + 断点续跑 + Drive 备份
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from dataclasses import asdict
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from tqdm.auto import tqdm

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "bitsandbytes>=0.46.1",
    "qwen-vl-utils",
    "tqdm",
], check=True)

from PIL import Image

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eval_chartqa import exact_match
from src.infer import DEFAULT_MODEL_ID, GenerationConfig, InferenceConfig, load_model_and_processor

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive")

FULL_VAL_JSONL = Path("data/processed/chartqa_val_full_sft_1920.jsonl")
FULL_VAL_IMAGE_DIR = Path("data/processed/chartqa_val_full_sft_1920_images")
DRIVE_PROCESSED = PROJECT_DRIVE / "data" / "processed"

LOCAL_FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

LOCAL_ADAPTER_DIR = Path("outputs/adapters")
DRIVE_ADAPTER_DIR = PROJECT_DRIVE / "outputs" / "adapters"

EXPECTED_TOTAL = 1920
SAVE_EVERY = 25
FORCE_INFER = False

# 如只想跑其中一个，可改成 ["baseline_default"] 或 ["standard_numeric_final"]
RUN_ONLY_LABELS = None

RUNS = {
    "baseline_default": {
        "run_name": "chartqa_val_full_3b_baseline_default_1920",
        "adapter": None,
        "prompt_name": "default",
        "prompt_template": (
            "Answer the chart question with a concise answer. "
            "If the answer is numeric, return only the number and unit when needed.\n"
            "Question: {question}"
        ),
    },
    "standard_numeric_final": {
        "run_name": "chartqa_val_full_3b_standard_steps100_numeric_final_1920",
        "adapter": "chartqa_qlora_train1k_steps100",
        "prompt_name": "numeric_final",
        "prompt_template": (
            "Answer the chart question with only the final answer. "
            "If the answer is numeric, keep the chart scale and unit consistent with the question and axis labels. "
            "Do any counting or arithmetic internally, but do not output reasoning.\n"
            "Question: {question}"
        ),
    },
}

LOCAL_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path):
    rows = []
    bad = []
    if not path.exists():
        return rows, bad
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                rows.append(json.loads(raw))
            except Exception as exc:
                bad.append((line_no, repr(exc), raw[:300]))
                break
    return rows, bad

def jsonl_count(path: Path) -> int:
    rows, bad = read_jsonl(path)
    if bad:
        return -1
    return len(rows)

def file_complete(path: Path, expected_total=EXPECTED_TOTAL) -> bool:
    return path.exists() and jsonl_count(path) == expected_total

def copy_to_drive(path: Path):
    if path.exists():
        DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, DRIVE_FULL_VAL_DIR / path.name)
        print("Copied to Drive:", DRIVE_FULL_VAL_DIR / path.name)

def restore_file_from_drive(path: Path):
    drive_path = DRIVE_FULL_VAL_DIR / path.name
    if not path.exists() and drive_path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, path)
        print("Restored from Drive:", drive_path)

def adapter_complete(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "adapter_config.json").exists()
        and (
            (path / "adapter_model.safetensors").exists()
            or (path / "adapter_model.bin").exists()
        )
    )

def copy_adapter(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied adapter:", src, "->", dst)

def find_adapter_in_drive(adapter_name: str):
    exact = [
        DRIVE_ADAPTER_DIR / adapter_name,
        PROJECT_DRIVE / "outputs" / "adapters" / adapter_name,
        Path("/content/drive/MyDrive") / adapter_name,
    ]
    for path in exact:
        if adapter_complete(path):
            return path

    for path in tqdm(DRIVE_ROOT.rglob(adapter_name), desc=f"find {adapter_name}"):
        if path.is_dir() and adapter_complete(path):
            return path
    return None

def restore_adapter(adapter_name: str | None):
    if adapter_name is None:
        return None

    local = LOCAL_ADAPTER_DIR / adapter_name
    drive = DRIVE_ADAPTER_DIR / adapter_name

    if adapter_complete(local):
        return local

    if adapter_complete(drive):
        copy_adapter(drive, local)
        return local

    found = find_adapter_in_drive(adapter_name)
    if found is not None:
        copy_adapter(found, local)
        copy_adapter(found, drive)
        return local

    raise FileNotFoundError(f"Missing adapter: {adapter_name}")

def restore_full_val_assets():
    drive_jsonl = DRIVE_PROCESSED / FULL_VAL_JSONL.name
    drive_image_dir = DRIVE_PROCESSED / FULL_VAL_IMAGE_DIR.name

    if not FULL_VAL_JSONL.exists() and drive_jsonl.exists():
        FULL_VAL_JSONL.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_jsonl, FULL_VAL_JSONL)
        print("Restored full-val JSONL:", drive_jsonl)

    if (
        not FULL_VAL_IMAGE_DIR.exists()
        or len(list(FULL_VAL_IMAGE_DIR.glob("*.png"))) < EXPECTED_TOTAL
    ) and drive_image_dir.exists():
        FULL_VAL_IMAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive_image_dir, FULL_VAL_IMAGE_DIR, dirs_exist_ok=True)
        print("Restored full-val images:", drive_image_dir)

    rows, bad = read_jsonl(FULL_VAL_JSONL)
    image_count = len(list(FULL_VAL_IMAGE_DIR.glob("*.png"))) if FULL_VAL_IMAGE_DIR.exists() else 0

    print({
        "full_val_jsonl": str(FULL_VAL_JSONL),
        "rows": len(rows),
        "bad": bad[:1],
        "image_dir": str(FULL_VAL_IMAGE_DIR),
        "image_count": image_count,
    })

    if bad or len(rows) != EXPECTED_TOTAL or image_count < EXPECTED_TOTAL:
        raise RuntimeError("Full-val assets incomplete. Run 20.1 first.")

    return rows

def build_messages(image, question: str, prompt_template: str):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_template.format(question=question)},
        ],
    }]

def predict_one(image, question, model, processor, generation_config, prompt_template):
    import torch
    from qwen_vl_utils import process_vision_info

    messages = build_messages(image, question, prompt_template)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generate_kwargs = asdict(generation_config)
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)
    latency = time.perf_counter() - start

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids, strict=True)
    ]
    answer = processor.batch_decode(
        generated_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return answer, round(latency, 4), generate_kwargs, prompt_template.format(question=question)

def run_incremental(label: str, cfg: dict, rows: list[dict]):
    run_name = cfg["run_name"]
    predictions = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"

    restore_file_from_drive(predictions)
    existing, bad = read_jsonl(predictions)

    if bad:
        backup = predictions.with_name(f"{predictions.stem}_corrupt_backup_{int(time.time())}.jsonl")
        shutil.copy2(predictions, backup)
        print("Backed up corrupt prediction:", backup)
        with predictions.open("w", encoding="utf-8") as handle:
            for row in existing:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    done = len(existing)

    if FORCE_INFER and predictions.exists():
        backup = predictions.with_name(f"{predictions.stem}_backup_{int(time.time())}.jsonl")
        shutil.copy2(predictions, backup)
        predictions.unlink()
        done = 0
        print("FORCE_INFER backup:", backup)

    print({
        "label": label,
        "run_name": run_name,
        "existing_rows": done,
        "remaining": EXPECTED_TOTAL - done,
        "predictions": str(predictions),
        "adapter": cfg["adapter"],
    })

    if done >= EXPECTED_TOTAL:
        print("Prediction already complete:", predictions)
        copy_to_drive(predictions)
        return predictions

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Switch Colab runtime to GPU.")

    adapter_path = restore_adapter(cfg["adapter"])

    inference_config = InferenceConfig(
        model_id=DEFAULT_MODEL_ID,
        adapter_path=str(adapter_path) if adapter_path is not None else None,
        load_in_4bit=True,
        device_map="auto",
        torch_dtype="auto",
        min_pixels=50176,
        max_pixels=401408,
    )
    generation_config = GenerationConfig(
        max_new_tokens=64,
        temperature=0.0,
        top_p=1.0,
        do_sample=False,
    )

    print("Loading model:", label)
    print("GPU:", torch.cuda.get_device_name(0))
    model, processor = load_model_and_processor(inference_config)
    model.eval()

    base_dir = FULL_VAL_JSONL.parent

    try:
        with predictions.open("a", encoding="utf-8") as handle:
            progress = tqdm(
                range(done, EXPECTED_TOTAL),
                initial=done,
                total=EXPECTED_TOTAL,
                desc=f"{label} full-val",
            )

            for position in progress:
                row = rows[position]
                image_path = base_dir / row["image"]
                image = Image.open(image_path).convert("RGB")
                question = row["query"]
                reference_answer = row.get("answer")

                answer, latency, generation, prompt_text = predict_one(
                    image=image,
                    question=question,
                    model=model,
                    processor=processor,
                    generation_config=generation_config,
                    prompt_template=cfg["prompt_template"],
                )
                is_exact = exact_match(answer, reference_answer)

                out_row = {
                    "sample_index": row.get("sample_index", position),
                    "selected_index": row.get("selected_index", position),
                    "question": question,
                    "answer": answer,
                    "reference_answer": reference_answer,
                    "all_labels": row.get("all_labels"),
                    "human_or_machine": row.get("human_or_machine"),
                    "split": row.get("split"),
                    "exact_match": is_exact,
                    "model_id": DEFAULT_MODEL_ID,
                    "adapter_path": str(adapter_path) if adapter_path is not None else None,
                    "prompt_name": cfg["prompt_name"],
                    "prompt_text": prompt_text,
                    "image_path": str(image_path),
                    "latency_seconds": latency,
                    "generation": generation,
                    "min_pixels": 50176,
                    "max_pixels": 401408,
                    "load_in_4bit": True,
                }

                handle.write(json.dumps(out_row, ensure_ascii=False) + "\n")
                handle.flush()
                if hasattr(os, "fsync"):
                    os.fsync(handle.fileno())

                completed = position + 1
                progress.set_postfix({"latency": latency, "exact": is_exact})

                if completed % SAVE_EVERY == 0:
                    copy_to_drive(predictions)

    except Exception:
        print("Inference failed; syncing partial prediction.")
        copy_to_drive(predictions)
        traceback.print_exc()
        raise
    finally:
        copy_to_drive(predictions)

    final_rows, final_bad = read_jsonl(predictions)
    if final_bad or len(final_rows) != EXPECTED_TOTAL:
        raise RuntimeError(f"Incomplete prediction: rows={len(final_rows)}, bad={final_bad[:1]}")

    return predictions

def evaluate_and_analyze(run_name: str):
    predictions = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"
    metrics = LOCAL_FULL_VAL_DIR / f"{run_name}_metrics.json"
    errors = LOCAL_FULL_VAL_DIR / f"{run_name}_errors.jsonl"
    evaluated = LOCAL_FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"
    analysis = LOCAL_FULL_VAL_DIR / f"{run_name}_error_analysis.json"

    if not metrics.exists() or not file_complete(evaluated) or not errors.exists():
        subprocess.run([
            sys.executable,
            "scripts/evaluate_predictions.py",
            "--predictions", str(predictions),
            "--metrics-output", str(metrics),
            "--errors-output", str(errors),
            "--evaluated-output", str(evaluated),
        ], check=True)

    if not analysis.exists():
        subprocess.run([
            sys.executable,
            "scripts/analyze_chartqa_errors.py",
            "--errors", str(errors),
            "--output", str(analysis),
        ], check=True)

    for path in [predictions, metrics, errors, evaluated, analysis]:
        if path.exists():
            copy_to_drive(path)

rows = restore_full_val_assets()

labels = list(RUNS)
if RUN_ONLY_LABELS is not None:
    labels = [x for x in labels if x in set(RUN_ONLY_LABELS)]

summary = {}

for label in labels:
    print("\n" + "=" * 80)
    print("RUN:", label)

    cfg = RUNS[label]
    run_name = cfg["run_name"]
    pred = LOCAL_FULL_VAL_DIR / f"{run_name}.jsonl"
    metrics = LOCAL_FULL_VAL_DIR / f"{run_name}_metrics.json"
    evaluated = LOCAL_FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"

    restore_file_from_drive(pred)
    restore_file_from_drive(metrics)
    restore_file_from_drive(evaluated)

    if file_complete(pred) and metrics.exists() and file_complete(evaluated):
        print("Already complete:", label)
        summary[label] = "already_complete"
        continue

    run_incremental(label, cfg, rows)
    evaluate_and_analyze(run_name)
    summary[label] = "completed"

print("\n=== 20.4.0.5 summary ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nREADY: rerun 20.4.")

/content/qwen25vl-chartqa-qlora
{'full_val_jsonl': 'data/processed/chartqa_val_full_sft_1920.jsonl', 'rows': 1920, 'bad': [], 'image_dir': 'data/processed/chartqa_val_full_sft_1920_images', 'image_count': 1920}

RUN: baseline_default
{'label': 'baseline_default', 'run_name': 'chartqa_val_full_3b_baseline_default_1920', 'existing_rows': 0, 'remaining': 1920, 'predictions': 'outputs/chartqa_3b_full_val/chartqa_val_full_3b_baseline_default_1920.jsonl', 'adapter': None}
Loading model: baseline_default
GPU: Tesla T4


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chart

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_standard_steps100_numeric_final_1920.jsonl
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chart

In [ ]:
# 20.4 汇总 full-val baseline/control/adapters，并重算 oracle。
%cd /content/qwen25vl-chartqa-qlora

from pathlib import Path
from collections import Counter
import csv
import json
import shutil

PROJECT_DRIVE = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
FULL_VAL_DIR = Path("outputs/chartqa_3b_full_val")
DRIVE_FULL_VAL_DIR = PROJECT_DRIVE / "outputs" / "chartqa_3b_full_val"

RUNS = {
    "baseline_default": "chartqa_val_full_3b_baseline_default_1920",
    "standard_steps100": "chartqa_val_full_3b_standard_steps100_1920",
    "standard_numeric_final": "chartqa_val_full_3b_standard_steps100_numeric_final_1920",
    "experiment_a_steps200": "chartqa_val_full_3b_steps200_1920",
    "experiment_b_calcnum": "chartqa_val_full_3b_calcnum1k_steps100_1920",
    "experiment_d_hardmix": "chartqa_val_full_3b_hardmix1k_steps100_1920",
    "experiment_f_steps250_r16a32": "chartqa_val_full_3b_steps250_r16a32_1920",
}

SUMMARY_JSON = FULL_VAL_DIR / "chartqa_val_full_3b_with_controls_summary.json"
SUMMARY_CSV = FULL_VAL_DIR / "chartqa_val_full_3b_with_controls_summary.csv"
ORACLE_JSON = FULL_VAL_DIR / "chartqa_val_full_3b_with_controls_oracle.json"

FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FULL_VAL_DIR.mkdir(parents=True, exist_ok=True)


def restore_from_drive(path: Path) -> None:
    drive_path = DRIVE_FULL_VAL_DIR / path.name
    if not path.exists() and drive_path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, path)
        print("Restored from Drive:", drive_path)


def load_json(path: Path):
    restore_from_drive(path)
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path: Path) -> list[dict]:
    restore_from_drive(path)
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def record_key(record: dict, fallback_index: int) -> str:
    if record.get("sample_index") is not None:
        return str(record["sample_index"])
    return str(fallback_index)


summary_rows = []
missing = []
evaluated_by_label = {}

for label, run_name in RUNS.items():
    metrics_path = FULL_VAL_DIR / f"{run_name}_metrics.json"
    evaluated_path = FULL_VAL_DIR / f"{run_name}_evaluated.jsonl"
    restore_from_drive(metrics_path)
    restore_from_drive(evaluated_path)
    if not metrics_path.exists() or not evaluated_path.exists():
        missing.append(label)
        continue

    metrics = load_json(metrics_path)
    evaluated = load_jsonl(evaluated_path)
    evaluated_by_label[label] = evaluated

    by_group = metrics.get("by_human_or_machine", {})
    summary_rows.append({
        "label": label,
        "run_name": run_name,
        "total": metrics.get("total"),
        "exact_match": metrics.get("exact_match"),
        "exact_accuracy": metrics.get("exact_accuracy"),
        "relaxed_correct": metrics.get("relaxed_correct"),
        "relaxed_accuracy": metrics.get("relaxed_accuracy"),
        "numeric_reference_total": metrics.get("numeric_reference_total"),
        "relaxed_numeric_match": metrics.get("relaxed_numeric_match"),
        "relaxed_numeric_accuracy_on_numeric": metrics.get("relaxed_numeric_accuracy_on_numeric"),
        "human_relaxed_accuracy": (by_group.get("0") or {}).get("relaxed_accuracy"),
        "machine_relaxed_accuracy": (by_group.get("1") or {}).get("relaxed_accuracy"),
        "errors": len(evaluated) - int(metrics.get("relaxed_correct", 0)),
    })

baseline = next((row for row in summary_rows if row["label"] == "baseline_default"), None)
for row in summary_rows:
    if baseline:
        row["relaxed_delta_vs_baseline"] = row["relaxed_accuracy"] - baseline["relaxed_accuracy"]
        row["exact_delta_vs_baseline"] = row["exact_accuracy"] - baseline["exact_accuracy"]
    else:
        row["relaxed_delta_vs_baseline"] = None
        row["exact_delta_vs_baseline"] = None

summary_rows = sorted(summary_rows, key=lambda row: (row["relaxed_accuracy"], row["exact_accuracy"]), reverse=True)

payload = {
    "scope": "full ChartQA val1920, 3B baseline/control/trained adapters when available",
    "expected_total": 1920,
    "available_runs": [row["label"] for row in summary_rows],
    "missing_runs": missing,
    "summary": summary_rows,
}

SUMMARY_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
with SUMMARY_CSV.open("w", encoding="utf-8", newline="") as handle:
    fieldnames = [
        "label",
        "run_name",
        "total",
        "exact_match",
        "exact_accuracy",
        "relaxed_correct",
        "relaxed_accuracy",
        "relaxed_delta_vs_baseline",
        "exact_delta_vs_baseline",
        "numeric_reference_total",
        "relaxed_numeric_match",
        "relaxed_numeric_accuracy_on_numeric",
        "human_relaxed_accuracy",
        "machine_relaxed_accuracy",
        "errors",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

oracle_payload = None
if evaluated_by_label:
    first_label = next(iter(evaluated_by_label))
    base_keys = [record_key(record, index) for index, record in enumerate(evaluated_by_label[first_label])]
    by_key = {
        label: {
            record_key(record, index): record
            for index, record in enumerate(records)
        }
        for label, records in evaluated_by_label.items()
    }

    oracle_correct = 0
    all_wrong = []
    unique_correct_counts = Counter()
    for key in base_keys:
        correct_labels = [
            label
            for label, records in by_key.items()
            if key in records and bool(records[key].get("eval_relaxed_correct"))
        ]
        if correct_labels:
            oracle_correct += 1
            if len(correct_labels) == 1:
                unique_correct_counts[correct_labels[0]] += 1
        else:
            first = by_key[first_label].get(key, {})
            all_wrong.append({
                "key": key,
                "sample_index": first.get("sample_index"),
                "question": first.get("question"),
                "reference": first.get("eval_reference"),
                "predictions": {
                    label: records.get(key, {}).get("eval_prediction")
                    for label, records in by_key.items()
                },
            })

    oracle_payload = {
        "runs": list(evaluated_by_label.keys()),
        "total": len(base_keys),
        "oracle_relaxed_correct": oracle_correct,
        "oracle_relaxed_accuracy": oracle_correct / len(base_keys) if base_keys else 0.0,
        "all_runs_wrong_count": len(all_wrong),
        "unique_correct_counts": dict(unique_correct_counts),
        "all_runs_wrong_preview": all_wrong[:30],
    }
    ORACLE_JSON.write_text(json.dumps(oracle_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for path in [SUMMARY_JSON, SUMMARY_CSV, ORACLE_JSON]:
    if path.exists():
        shutil.copy2(path, DRIVE_FULL_VAL_DIR / path.name)
        print("Copied to Drive:", DRIVE_FULL_VAL_DIR / path.name)

print("full_val_with_controls_summary:")
for row in summary_rows:
    print(row["label"], {
        "exact": f"{row['exact_match']}/{row['total']} = {row['exact_accuracy']:.2%}",
        "relaxed": f"{row['relaxed_correct']}/{row['total']} = {row['relaxed_accuracy']:.2%}",
        "relaxed_delta_vs_baseline": None if row["relaxed_delta_vs_baseline"] is None else f"{row['relaxed_delta_vs_baseline']:+.2%}",
    })

if missing:
    print("missing_runs:", missing)
    print("Run 20.2 for baseline_default and 20.3 for standard_numeric_final; module 19 outputs are needed for trained adapters.")

if oracle_payload:
    print("oracle_with_available_runs:", {
        "runs": oracle_payload["runs"],
        "oracle_relaxed": f"{oracle_payload['oracle_relaxed_correct']}/{oracle_payload['total']} = {oracle_payload['oracle_relaxed_accuracy']:.2%}",
        "all_runs_wrong_count": oracle_payload["all_runs_wrong_count"],
        "unique_correct_counts": oracle_payload["unique_correct_counts"],
    })

print("summary_json:", SUMMARY_JSON)
print("summary_csv:", SUMMARY_CSV)
print("oracle_json:", ORACLE_JSON if ORACLE_JSON.exists() else "skipped")


/content/qwen25vl-chartqa-qlora
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_with_controls_summary.json
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_with_controls_summary.csv
Copied to Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_3b_full_val/chartqa_val_full_3b_with_controls_oracle.json
full_val_with_controls_summary:
experiment_d_hardmix {'exact': '1331/1920 = 69.32%', 'relaxed': '1495/1920 = 77.86%', 'relaxed_delta_vs_baseline': '+1.93%'}
experiment_f_steps250_r16a32 {'exact': '1334/1920 = 69.48%', 'relaxed': '1491/1920 = 77.66%', 'relaxed_delta_vs_baseline': '+1.72%'}
experiment_a_steps200 {'exact': '1325/1920 = 69.01%', 'relaxed': '1489/1920 = 77.55%', 'relaxed_delta_vs_baseline': '+1.61%'}
experiment_b_calcnum {'exact': '1323/1920 = 68.91%', 'relaxed': '1487/1920 = 77.45%', 'relaxed_delta_vs_baseline': '+1.51%'}
standard_steps1

# Module 21 - ChartQA all-wrong diagnostic subset ablation

This module is designed for the active 3B Colab notebook. It does **not** run full validation. It only runs the recommended all-wrong diagnostic subset from the manual audit.

## Dependencies and minimal run order

Run after a fresh Colab restart:


```text
1.1 -> 1.3 -> 1.4 -> 21.1 -> 21.2 -> 21.3 -> 21.4 -> 21.5 -> 21.6
```


Assumptions:

- Google Drive is mounted at `/content/drive/MyDrive`.
- Repo path is `/content/qwen25vl-chartqa-qlora`.
- Drive project root is `/content/drive/MyDrive/qwen25vl-chartqa-qlora`.
- Module 21 helper scripts are either already in `scripts/` after `git pull`, or backed up in Drive under `scripts_module21/`.
- Manual audit inputs are backed up in Drive under `outputs/chartqa_all_wrong_diagnostics/inputs/`.
- This module writes local outputs under `outputs/chartqa_all_wrong_diagnostics/` and persists all important outputs to Drive.

Long-running cells use progress bars and are safely rerunnable. Prediction/extraction JSONL files are appended incrementally and skip completed `sample_index` rows on rerun.

## 21.1 Restore Module 21 inputs, scripts, and adapters


In [ ]:
# Module 21.1 - restore inputs, helper scripts, and required adapters
from pathlib import Path
import json
import shutil
import subprocess

from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_INPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/inputs"
DRIVE_SCRIPT_DIR = DRIVE_ROOT / "scripts_module21"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics"

LOCAL_OUTPUT_DIR = REPO / "outputs/chartqa_all_wrong_diagnostics"
LOCAL_DATA_DIR = REPO / "data/diagnostics"
LOCAL_SCRIPT_DIR = REPO / "scripts"

MODULE21_SCRIPTS = [
    "prepare_chartqa_all_wrong_subset.py",
    "run_chartqa_subset_ablation.py",
    "run_chartqa_structured_extraction_diagnostic.py",
    "summarize_chartqa_all_wrong_diagnostics.py",
]

REQUIRED_INPUTS = [
    "chartqa_all_wrong_manual_audit_report.md",
    "chartqa_all_wrong_manual_audit_table.csv",
    "chartqa_all_wrong_manual_audit_table.json",
    "chartqa_all_wrong_recommended_diagnostic_subset.csv",
]

FULL_VAL_SFT = DRIVE_ROOT / "data/processed/chartqa_val_full_sft_1920.jsonl"
FULL_VAL_IMAGE_ROOT = DRIVE_ROOT / "data/processed/chartqa_val_full_sft_1920_images"

HARDMIX_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
F_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_train1k_steps250_r16a32"
DRIVE_HARDMIX_ADAPTER = DRIVE_ROOT / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
DRIVE_F_ADAPTER = DRIVE_ROOT / "outputs/adapters/chartqa_qlora_train1k_steps250_r16a32"

for path in [REPO, DRIVE_ROOT, DRIVE_INPUT_DIR, FULL_VAL_SFT, FULL_VAL_IMAGE_ROOT]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path: {path}")

%cd /content/qwen25vl-chartqa-qlora

# If this notebook is running before the new scripts are in GitHub, restore them from Drive.
LOCAL_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
for script_name in MODULE21_SCRIPTS:
    local_script = LOCAL_SCRIPT_DIR / script_name
    drive_script = DRIVE_SCRIPT_DIR / script_name
    if local_script.exists():
        print(f"Script already present: {local_script}")
    elif drive_script.exists():
        shutil.copy2(drive_script, local_script)
        print(f"Restored script from Drive: {drive_script} -> {local_script}")
    else:
        raise FileNotFoundError(f"Missing Module 21 script in repo and Drive: {script_name}")

for file_name in REQUIRED_INPUTS:
    path = DRIVE_INPUT_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Missing manual audit input: {path}")
    print("Input OK:", path)

def copytree_with_progress(src: Path, dst: Path, desc: str) -> None:
    if dst.exists():
        print(f"Adapter already restored: {dst}")
        return
    files = [p for p in src.rglob("*") if p.is_file()]
    if not files:
        raise FileNotFoundError(f"No files found in adapter source: {src}")
    dst.mkdir(parents=True, exist_ok=True)
    for file_path in tqdm(files, desc=desc, unit="files"):
        rel = file_path.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, target)

copytree_with_progress(DRIVE_HARDMIX_ADAPTER, HARDMIX_ADAPTER, "Restoring hardmix adapter")
copytree_with_progress(DRIVE_F_ADAPTER, F_ADAPTER, "Restoring F adapter")

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Module 21 restore complete.")
print("Drive output dir:", DRIVE_OUTPUT_DIR)
print("Hardmix adapter:", HARDMIX_ADAPTER)
print("F adapter:", F_ADAPTER)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/qwen25vl-chartqa-qlora
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module21/prepare_chartqa_all_wrong_subset.py -> /content/qwen25vl-chartqa-qlora/scripts/prepare_chartqa_all_wrong_subset.py
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module21/run_chartqa_subset_ablation.py -> /content/qwen25vl-chartqa-qlora/scripts/run_chartqa_subset_ablation.py
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module21/run_chartqa_structured_extraction_diagnostic.py -> /content/qwen25vl-chartqa-qlora/scripts/run_chartqa_structured_extraction_diagnostic.py
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module21/summarize_chartqa_all_wrong_diagnostics.py -> /content/qwen25vl-chartqa-qlora/scripts/summarize_chartqa_all_wrong_di

Restoring hardmix adapter:   0%|          | 0/7 [00:00<?, ?files/s]

Restoring F adapter:   0%|          | 0/7 [00:00<?, ?files/s]

Module 21 restore complete.
Drive output dir: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics
Hardmix adapter: /content/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_hardmix1k_steps100
F adapter: /content/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps250_r16a32


## 21.2 Prepare diagnostic subset


In [ ]:
# Module 21.2 - prepare the fixed all-wrong diagnostic subset
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_INPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/inputs"
DRIVE_DATA_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/data"

SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"
SUBSET_SUMMARY = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85_summary.json"

cmd = [
    "python", "scripts/prepare_chartqa_all_wrong_subset.py",
    "--subset-csv", str(DRIVE_INPUT_DIR / "chartqa_all_wrong_recommended_diagnostic_subset.csv"),
    "--manual-audit-csv", str(DRIVE_INPUT_DIR / "chartqa_all_wrong_manual_audit_table.csv"),
    "--full-val-sft-jsonl", str(DRIVE_ROOT / "data/processed/chartqa_val_full_sft_1920.jsonl"),
    "--image-root", str(DRIVE_ROOT / "data/processed/chartqa_val_full_sft_1920_images"),
    "--output-jsonl", str(SUBSET_JSONL),
    "--summary-output", str(SUBSET_SUMMARY),
    "--drive-output-dir", str(DRIVE_DATA_DIR),
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/prepare_chartqa_all_wrong_subset.py', '--subset-csv', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_recommended_diagnostic_subset.csv', '--manual-audit-csv', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_manual_audit_table.csv', '--full-val-sft-jsonl', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_val_full_sft_1920.jsonl', '--image-root', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/data/processed/chartqa_val_full_sft_1920_images', '--output-jsonl', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--summary-output', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85_summary.json', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/data'], returncode=0)

## 21.3 Snapshot original all-wrong predictions for the subset


In [ ]:
# Module 21.3 - summarize existing original predictions from the manual audit
from pathlib import Path
import csv
import json
import shutil
from collections import Counter

from tqdm.auto import tqdm

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_INPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/inputs"
DRIVE_SUMMARY_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/summaries"
LOCAL_SUMMARY_DIR = REPO / "outputs/chartqa_all_wrong_diagnostics/summaries"
LOCAL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

manual_csv = DRIVE_INPUT_DIR / "chartqa_all_wrong_recommended_diagnostic_subset.csv"
rows = []
with manual_csv.open("r", encoding="utf-8-sig", newline="") as handle:
    for row in tqdm(csv.DictReader(handle), desc="Reading diagnostic subset rows", unit="rows"):
        rows.append(row)

summary = {
    "total": len(rows),
    "reviewed_primary_counts": dict(Counter(row["reviewed_primary"] for row in rows)),
    "review_flag_counts": dict(Counter(flag for row in rows for flag in row["review_flags"].split(";") if flag)),
    "note": "These rows are the fixed all-runs-wrong diagnostic subset. Original seven full-val runs are all relaxed-wrong by construction.",
}

json_path = LOCAL_SUMMARY_DIR / "original_all_wrong_subset_snapshot.json"
json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copy2(json_path, DRIVE_SUMMARY_DIR / json_path.name)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Copied snapshot to:", DRIVE_SUMMARY_DIR / json_path.name)


Reading diagnostic subset rows: 0rows [00:00, ?rows/s]

{
  "total": 85,
  "reviewed_primary_counts": {
    "counting_or_category_count": 8,
    "data_or_evaluator_issue": 2,
    "date_axis_reading": 9,
    "date_serial_or_label_format": 2,
    "extreme_value_or_ranking": 9,
    "multi_step_calculation": 9,
    "numeric_value_or_scale": 10,
    "text_label_lookup": 16,
    "visual_mapping_or_legend": 10,
    "yes_no_or_boolean": 10
  },
  "review_flag_counts": {
    "needs_legend_color_mapping": 44,
    "needs_table_or_value_extraction": 67,
    "resolution_sensitive": 66,
    "shared_wrong_consensus": 79,
    "data_or_evaluator_issue": 16,
    "needs_axis_date_grounding": 27,
    "list_answer_format": 7,
    "date_serial_reference": 2,
    "calculation_after_extraction": 6,
    "unstable_across_runs": 1,
    "boolean_or_trend": 10
  },
  "note": "These rows are the fixed all-runs-wrong diagnostic subset. Original seven full-val runs are all relaxed-wrong by construction."
}
Copied snapshot to: /content/drive/MyDrive/qwen25vl-chartqa-qlora/

## 21.4 High-resolution and prompt ablation


In [ ]:
# Module 21.4 - run high-resolution/prompt ablation on the fixed subset
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"
HARDMIX_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
F_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_train1k_steps250_r16a32"

cmd = [
    "python", "scripts/run_chartqa_subset_ablation.py",
    "--subset-jsonl", str(SUBSET_JSONL),
    "--output-dir", "outputs/chartqa_all_wrong_diagnostics",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/ablation_runs"),
    "--hardmix-adapter-path", str(HARDMIX_ADAPTER),
    "--f-adapter-path", str(F_ADAPTER),
    "--runs",
    "baseline_maxpix_802816",
    "hardmix_maxpix_602112",
    "hardmix_maxpix_802816",
    "f_maxpix_802816",
    "hardmix_axis_legend_prompt_802816",
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/run_chartqa_subset_ablation.py', '--subset-jsonl', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--output-dir', 'outputs/chartqa_all_wrong_diagnostics', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/ablation_runs', '--hardmix-adapter-path', '/content/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_hardmix1k_steps100', '--f-adapter-path', '/content/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_train1k_steps250_r16a32', '--runs', 'baseline_maxpix_802816', 'hardmix_maxpix_602112', 'hardmix_maxpix_802816', 'f_maxpix_802816', 'hardmix_axis_legend_prompt_802816'], returncode=0)

## 21.5 Structured extraction diagnostic


In [ ]:
# Module 21.5 - chart-to-JSON extraction plus table-assisted QA
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"

# Use the base 3B model first. The QA adapters were not trained for chart-to-JSON extraction.
cmd = [
    "python", "scripts/run_chartqa_structured_extraction_diagnostic.py",
    "--subset-jsonl", str(SUBSET_JSONL),
    "--output-dir", "outputs/chartqa_all_wrong_diagnostics",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/structured_extraction"),
    "--max-pixels", "802816",
    "--extract-max-new-tokens", "768",
    "--qa-max-new-tokens", "128",
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/run_chartqa_structured_extraction_diagnostic.py', '--subset-jsonl', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--output-dir', 'outputs/chartqa_all_wrong_diagnostics', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/structured_extraction', '--max-pixels', '802816', '--extract-max-new-tokens', '768', '--qa-max-new-tokens', '128'], returncode=0)

## 21.6 Summarize diagnostic results


In [ ]:
# Module 21.6 - summarize recovered samples across diagnostic runs
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"

cmd = [
    "python", "scripts/summarize_chartqa_all_wrong_diagnostics.py",
    "--subset-jsonl", str(SUBSET_JSONL),
    "--output-dir", "outputs/chartqa_all_wrong_diagnostics",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/summaries"),
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/summarize_chartqa_all_wrong_diagnostics.py', '--subset-jsonl', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--output-dir', 'outputs/chartqa_all_wrong_diagnostics', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/summaries'], returncode=0)

## 21.7 中文结果解读与下一步判断

### 本轮运行状态

Module 21 已完整跑完，`21.1 -> 21.6` 均成功返回，未发现 notebook cell 报错。

本轮只跑推荐的 all-wrong diagnostic subset，共 85 条；没有跑 full val，也没有训练新 LoRA。这个子集中的样本按定义都是原始 7 个 full-val run 全部 relaxed-wrong 的样本。

### 核心结果

| run | relaxed | exact | 备注 |
|---|---:|---:|---|
| `table_json_only` | 14/85 = 16.47% | 9/85 = 10.59% | 单 run 最好，结构化抽取方向有信号 |
| `hardmix_maxpix_602112` | 13/85 = 15.29% | 12/85 = 14.12% | image-only 最好之一 |
| `hardmix_maxpix_802816` | 13/85 = 15.29% | 12/85 = 14.12% | 与 602112 持平 |
| `image_plus_table_json` | 13/85 = 15.29% | 11/85 = 12.94% | 与 hardmix high-res 持平 |
| `f_maxpix_802816` | 12/85 = 14.12% | 12/85 = 14.12% | exact 不错，但 relaxed 不占优 |
| `baseline_maxpix_802816` | 10/85 = 11.76% | 9/85 = 10.59% | 纯高分辨率 baseline 有小幅追回 |
| `hardmix_axis_legend_prompt_802816` | 10/85 = 11.76% | 9/85 = 10.59% | 显式 axis/legend prompt 未带来提升 |

全部 Module 21 方法合并后的 oracle 为：

```text
26/85 = 30.59%
```

仍然全部错误：

```text
59/85 = 69.41%
```

chart-to-JSON extraction 共生成 85 条，其中 66 条能被解析为合法 JSON。

### 关键观察

1. **高分辨率有帮助，但不是充分解法。**

`baseline_maxpix_802816` 追回 10 条，说明部分错误确实和分辨率/可读性有关。但 hardmix 从 `602112` 提到 `802816` 后结果仍是 13/85，没有继续提升。

2. **hardmix 仍是 image-only 路线中更稳的 adapter。**

`hardmix_maxpix_602112` 和 `hardmix_maxpix_802816` 均为 13/85，高于 `f_maxpix_802816` 的 12/85，也高于 baseline 的 10/85。

3. **当前 axis/legend 显式 prompt 不值得作为主线。**

`hardmix_axis_legend_prompt_802816` 只有 10/85，低于普通 hardmix high-res。说明这个 prompt 没有稳定改善 grounding，后续如果继续做，应改成更细的分步抽取，而不是单句提示。

4. **table / structured extraction 是最值得继续的方向。**

`table_json_only` 是单 run 最好，并且有 5 个独有追回样本。这说明结构化读图能救回一部分 image-only 救不回的样本。

5. **但当前一次性 chart-to-JSON 还不够强。**

虽然 `table_json_only` 最好，但也只有 14/85。说明下一步不能只把这版 extraction 直接扩到 full val；应该先改进 extraction 质量，尤其是轴、图例、数据点、日期刻度和值的结构化还原。

### 按类别的 oracle 追回情况

| reviewed_primary | recovered / total |
|---|---:|
| `counting_or_category_count` | 5/8 |
| `text_label_lookup` | 7/16 |
| `yes_no_or_boolean` | 4/10 |
| `multi_step_calculation` | 3/9 |
| `numeric_value_or_scale` | 2/10 |
| `date_axis_reading` | 2/9 |
| `extreme_value_or_ranking` | 2/9 |
| `visual_mapping_or_legend` | 1/10 |
| `data_or_evaluator_issue` | 0/2 |
| `date_serial_or_label_format` | 0/2 |

比较容易被当前诊断追回的是 counting、text lookup、boolean/trend；仍然困难的是 date-axis、numeric scale、visual mapping。

### 当前判断

本轮结果进一步支持之前的判断：

> 不建议现在继续做新的 LoRA steps/rank 变体。下一步应继续围绕结构化读图、表格抽取、日期轴 grounding、图例颜色映射和 evaluator/data cleanup 展开。

推荐下一步：

```text
Module 22A: data/evaluator cleanup list
Module 22B: staged chart-to-table extraction on the same 85-sample subset
```

`Module 22B` 不应一次性要求模型输出完整 JSON，而应拆成：

1. chart type / title;
2. x-axis / y-axis / tick labels;
3. legend-color mapping;
4. visible text and data labels;
5. data_points table;
6. final QA 或 calculator-assisted QA。

这样能判断错误到底发生在 extraction 哪一步，而不是只看到最终 QA 仍然错误。


# Module 22A - evaluator/data cleanup list

This module does not run model inference and does not run full validation. It only turns the manual all-wrong audit into a reproducible evaluator/data cleanup list.

## Dependencies and minimal run order

Run after a fresh Colab restart:


```text
1.1 -> 1.3 -> 1.4 -> 22A.1 -> 22A.2 -> 22A.3
```


Inputs must already exist in Drive:

- `/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_manual_audit_table.csv`
- `/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_recommended_diagnostic_subset.csv`
- Module 22A helper script in either repo `scripts/` or Drive `scripts_module22a/`.

Outputs are written locally and copied to Drive:


```text
outputs/chartqa_evaluator_cleanup/
/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/
```


## 22A.1 Restore inputs and helper script


In [ ]:
# Module 22A.1 - 恢复输入和 helper 脚本；不跑模型，不跑 full-val
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_INPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/inputs"
DRIVE_SCRIPT_DIR = DRIVE_ROOT / "scripts_module22a"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_evaluator_cleanup"
LOCAL_OUTPUT_DIR = REPO / "outputs/chartqa_evaluator_cleanup"

SCRIPT_NAME = "prepare_chartqa_evaluator_cleanup.py"
LOCAL_SCRIPT = REPO / "scripts" / SCRIPT_NAME
DRIVE_SCRIPT = DRIVE_SCRIPT_DIR / SCRIPT_NAME

MANUAL_AUDIT_CSV = DRIVE_INPUT_DIR / "chartqa_all_wrong_manual_audit_table.csv"
SUBSET_CSV = DRIVE_INPUT_DIR / "chartqa_all_wrong_recommended_diagnostic_subset.csv"

for path in [REPO, DRIVE_ROOT, DRIVE_INPUT_DIR, MANUAL_AUDIT_CSV, SUBSET_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path: {path}")

%cd /content/qwen25vl-chartqa-qlora

# 如果 GitHub 里的 repo 还没包含 22A 脚本，就从 Drive 备份恢复。
LOCAL_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_SCRIPT.exists():
    print("Script already present:", LOCAL_SCRIPT)
elif DRIVE_SCRIPT.exists():
    shutil.copy2(DRIVE_SCRIPT, LOCAL_SCRIPT)
    print("Restored script from Drive:", DRIVE_SCRIPT, "->", LOCAL_SCRIPT)
else:
    raise FileNotFoundError(f"Missing helper script in repo and Drive: {SCRIPT_NAME}")

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Module 22A restore complete.")
print("Manual audit CSV:", MANUAL_AUDIT_CSV)
print("Subset CSV:", SUBSET_CSV)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/qwen25vl-chartqa-qlora
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module22a/prepare_chartqa_evaluator_cleanup.py -> /content/qwen25vl-chartqa-qlora/scripts/prepare_chartqa_evaluator_cleanup.py
Module 22A restore complete.
Manual audit CSV: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_manual_audit_table.csv
Subset CSV: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_recommended_diagnostic_subset.csv
Drive output dir: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup


## 22A.2 Generate cleanup candidates


In [ ]:
# Module 22A.2 - 生成 evaluator/data cleanup 清单
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_INPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/inputs"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_evaluator_cleanup"

cmd = [
    "python", "scripts/prepare_chartqa_evaluator_cleanup.py",
    "--manual-audit-csv", str(DRIVE_INPUT_DIR / "chartqa_all_wrong_manual_audit_table.csv"),
    "--subset-csv", str(DRIVE_INPUT_DIR / "chartqa_all_wrong_recommended_diagnostic_subset.csv"),
    "--output-dir", "outputs/chartqa_evaluator_cleanup",
    "--drive-output-dir", str(DRIVE_OUTPUT_DIR),
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/prepare_chartqa_evaluator_cleanup.py', '--manual-audit-csv', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_manual_audit_table.csv', '--subset-csv', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/inputs/chartqa_all_wrong_recommended_diagnostic_subset.csv', '--output-dir', 'outputs/chartqa_evaluator_cleanup', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup'], returncode=0)

## 22A.3 Read the cleanup report and decision notes


In [ ]:
# Module 22A.3 - 读取清理报告，给出下一步判断
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")
SUMMARY_JSON = REPO / "outputs/chartqa_evaluator_cleanup/chartqa_evaluator_cleanup_summary.json"
REPORT_MD = REPO / "outputs/chartqa_evaluator_cleanup/chartqa_evaluator_cleanup_report.md"

summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
subset = summary["recommended_subset"]

print("推荐 85 条 subset 样本数:", subset["total"])
print("cleanup candidates:", subset["cleanup_candidate_count"])
print("若仅剔除/修正 exclude_or_fix_reference，有效模型错误数:", subset["effective_model_error_count_if_excluded"])
print("\ncleanup policy counts:")
print(json.dumps(subset["cleanup_policy_counts"], ensure_ascii=False, indent=2))
print("\nissue type counts:")
print(json.dumps(subset["issue_type_counts"], ensure_ascii=False, indent=2))

print("\n报告位置:", REPORT_MD)
print(REPORT_MD.read_text(encoding="utf-8")[:4000])


推荐 85 条 subset 样本数: 85
cleanup candidates: 21
若仅剔除/修正 exclude_or_fix_reference，有效模型错误数: 77

cleanup policy counts:
{
  "exclude_or_fix_reference": 8,
  "normalization_candidate": 6,
  "answer_format_manual_review": 7
}

issue type counts:
{
  "date_serial_reference": 2,
  "color_granularity": 1,
  "list_answer_format": 7,
  "scale_normalization": 1,
  "general_data_or_evaluator_issue": 2,
  "answer_type_or_reference_mismatch": 4,
  "semantic_equivalence": 1,
  "ocr_spelling_near_miss": 3
}

报告位置: /content/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/chartqa_evaluator_cleanup_report.md
# ChartQA evaluator/data cleanup candidates - Module 22A

## 目的

本模块只整理评测口径、标注、reference、答案格式和 normalization 问题，不重新跑模型，不修改历史 full-val 指标。

## 汇总

- all-wrong 全量审计表样本数：325
- 全量 cleanup candidates：21
- 推荐 85 条 subset 样本数：85
- subset cleanup candidates：21
- 若仅剔除 `exclude_or_fix_reference` 类，subset 有效模型错误数：77

## subset cleanup policy 分布

| policy | count |
|---|---:|
| `answer_format_manual_revie

## 22A.4 中文结论

本模块只建立清理清单，不直接改 evaluator，也不重算历史 full-val。

使用口径：

- `exclude_or_fix_reference`：高优先级。先从模型能力增益判断中单独剥离，后续修 reference 或建立 ignore list。
- `normalization_candidate`：中优先级。需要设计 evaluator normalization 规则，并人工抽查。
- `answer_format_manual_review`：中优先级。先确认题目要求 list、sum、顺序无关 list，不能直接当模型失败。

下一步建议：


```text
Module 22B: staged chart-to-table extraction on the same 85-sample subset
```


22B 之前不要扩到 full-val，也不要继续训练新 LoRA。


## 22A.5 最新运行结果分析

### 运行状态

Module 22A 已成功运行完成：

- `22A.1` 成功从 Drive 恢复输入和 helper 脚本；
- `22A.2` 成功生成 evaluator/data cleanup candidates；
- `22A.3` 成功读取 summary 和 markdown report；
- 未发现 notebook cell 报错。

本模块没有跑模型，没有跑 full-val，也没有修改历史指标。

### 22A 清理结果

推荐 85 条 diagnostic subset 中，识别出：

```text
cleanup candidates: 21/85
exclude_or_fix_reference: 8
normalization_candidate: 6
answer_format_manual_review: 7
```

如果只把最高优先级的 `exclude_or_fix_reference` 从模型能力判断中剥离，则有效模型错误样本数从 85 变为：

```text
77
```

### issue type 分布

| issue_type | count |
|---|---:|
| `answer_type_or_reference_mismatch` | 4 |
| `date_serial_reference` | 2 |
| `general_data_or_evaluator_issue` | 2 |
| `list_answer_format` | 7 |
| `ocr_spelling_near_miss` | 3 |
| `color_granularity` | 1 |
| `scale_normalization` | 1 |
| `semantic_equivalence` | 1 |

### 高优先级剔除/修正样本

这些样本不应直接用于判断模型能力，应先修 reference 或单独统计：

```text
12, 14, 105, 158, 470, 918, 1351, 1561
```

原因包括：

- Excel serial date reference；
- 问题问数值但 reference 是 label；
- 问题问 label 但 reference 是数值；
- 明显 annotation/reference mismatch。

### normalization / 格式复核样本

normalization candidates：

```text
18, 46, 648, 1114, 1327, 1810
```

典型问题：

- `Light blue` vs `Blue`
- `0.0034` vs `0.34`
- `increasing` vs `increase`
- OCR spelling near miss

answer-format manual review：

```text
29, 362, 408, 455, 667, 688, 797
```

典型问题：

- list-vs-sum；
- list 顺序；
- list 部分正确；
- 题目到底要求单值还是多个值不够清晰。

### 对 Module 21 结论的影响

Module 21 原始 oracle：

```text
26/85 = 30.59%
```

剔除 8 个 `exclude_or_fix_reference` 后，有效 denominator 为 77。Module 21 oracle 追回样本中有 2 个属于 `exclude_or_fix_reference`：

```text
158, 1351
```

因此有效口径下的 Module 21 oracle 为：

```text
24/77 = 31.17%
```

这个变化很小，说明 Module 21 的主要结论没有被 evaluator/data issue 推翻：

> 高分辨率和一次性 chart-to-JSON 有一定帮助，但仍不足以解决大多数 hard subset；下一步仍应做 staged chart-to-table extraction，而不是继续 LoRA rank/steps。

### 当前判断

22A 的价值是把“模型真的不会”和“评测/标注/格式口径不干净”拆开了。

下一步建议仍然是：

```text
Module 22B: staged chart-to-table extraction on the same 85-sample subset
```

22B 应优先避开或单独标记 8 个 `exclude_or_fix_reference` 样本，防止错误 reference 污染 extraction 诊断。


# Module 22B - staged chart-to-table extraction diagnostic

This module runs staged chart-to-table extraction on the same all-wrong diagnostic subset. It does **not** run full validation and does **not** train a new adapter.

Compared with Module 21.5, this module avoids one-shot chart-to-JSON. It splits the extraction into:

1. chart overview;
2. axes / ticks / legend / color mapping;
3. question-relevant data table;
4. table-only QA;
5. image + staged table QA.

By default it skips the 8 high-priority `exclude_or_fix_reference` samples produced by Module 22A, leaving 77 valid samples for model-capability diagnostics.

## Dependencies and minimal run order

Run after a fresh Colab restart:


```text
1.1 -> 1.3 -> 1.4 -> 22B.1 -> 22B.2 -> 22B.3
```


Required Drive inputs:

- `/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/data/chartqa_all_wrong_diagnostic_subset_85.jsonl`
- `/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv`
- helper script in repo `scripts/` or Drive `scripts_module22b/`.

Long-running steps use progress bars. Stage outputs are append-only JSONL files and are written to both local Colab and Drive, so reruns skip completed `sample_index` rows.

## 22B.1 Restore inputs and helper script


In [ ]:
# Module 22B.1 - 恢复 staged extraction 输入和 helper 脚本
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_SCRIPT_DIR = DRIVE_ROOT / "scripts_module22b"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/staged_extraction"

DRIVE_SUBSET_JSONL = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/data/chartqa_all_wrong_diagnostic_subset_85.jsonl"
DRIVE_EXCLUDE_CSV = DRIVE_ROOT / "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv"

LOCAL_SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"
LOCAL_EXCLUDE_CSV = REPO / "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv"

SCRIPT_NAME = "run_chartqa_staged_extraction_diagnostic.py"
LOCAL_SCRIPT = REPO / "scripts" / SCRIPT_NAME
DRIVE_SCRIPT = DRIVE_SCRIPT_DIR / SCRIPT_NAME

for path in [REPO, DRIVE_ROOT, DRIVE_SUBSET_JSONL, DRIVE_EXCLUDE_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path: {path}")

%cd /content/qwen25vl-chartqa-qlora

LOCAL_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_SCRIPT.exists():
    print("Script already present:", LOCAL_SCRIPT)
elif DRIVE_SCRIPT.exists():
    shutil.copy2(DRIVE_SCRIPT, LOCAL_SCRIPT)
    print("Restored script from Drive:", DRIVE_SCRIPT, "->", LOCAL_SCRIPT)
else:
    raise FileNotFoundError(f"Missing helper script in repo and Drive: {SCRIPT_NAME}")

LOCAL_SUBSET_JSONL.parent.mkdir(parents=True, exist_ok=True)
LOCAL_EXCLUDE_CSV.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_SUBSET_JSONL.exists():
    shutil.copy2(DRIVE_SUBSET_JSONL, LOCAL_SUBSET_JSONL)
    print("Restored subset JSONL:", LOCAL_SUBSET_JSONL)
if not LOCAL_EXCLUDE_CSV.exists():
    shutil.copy2(DRIVE_EXCLUDE_CSV, LOCAL_EXCLUDE_CSV)
    print("Restored exclude list:", LOCAL_EXCLUDE_CSV)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Module 22B restore complete.")
print("Subset JSONL:", LOCAL_SUBSET_JSONL)
print("Exclude list:", LOCAL_EXCLUDE_CSV)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/qwen25vl-chartqa-qlora
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module22b/run_chartqa_staged_extraction_diagnostic.py -> /content/qwen25vl-chartqa-qlora/scripts/run_chartqa_staged_extraction_diagnostic.py
Restored subset JSONL: /content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl
Restored exclude list: /content/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv
Module 22B restore complete.
Subset JSONL: /content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl
Exclude list: /content/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv
Drive output dir: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_d

## 22B.2 Run staged extraction and QA


In [ ]:
# Module 22B.2 - 分步 chart-to-table extraction + QA
from pathlib import Path
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"
EXCLUDE_CSV = REPO / "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/staged_extraction"

cmd = [
    "python", "scripts/run_chartqa_staged_extraction_diagnostic.py",
    "--subset-jsonl", str(SUBSET_JSONL),
    "--exclude-list-csv", str(EXCLUDE_CSV),
    "--output-dir", "outputs/chartqa_all_wrong_diagnostics/staged_extraction",
    "--drive-output-dir", str(DRIVE_OUTPUT_DIR),
    "--max-pixels", "802816",
    "--stage-max-new-tokens", "768",
    "--qa-max-new-tokens", "128",
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/run_chartqa_staged_extraction_diagnostic.py', '--subset-jsonl', '/content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--exclude-list-csv', '/content/qwen25vl-chartqa-qlora/outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv', '--output-dir', 'outputs/chartqa_all_wrong_diagnostics/staged_extraction', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/staged_extraction', '--max-pixels', '802816', '--stage-max-new-tokens', '768', '--qa-max-new-tokens', '128'], returncode=0)

## 22B.3 Read staged extraction summary


In [ ]:
# Module 22B.3 - 读取 staged extraction 结果
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")
SUMMARY_JSON = REPO / "outputs/chartqa_all_wrong_diagnostics/staged_extraction/staged_extraction_summary.json"
REPORT_MD = REPO / "outputs/chartqa_all_wrong_diagnostics/staged_extraction/staged_extraction_report.md"

summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))

print("有效样本数（剔除 reference/evaluator 高风险样本后）:", summary["subset_total_after_excluding_reference_issues"])
print("跳过样本:", summary["skipped_exclude_or_fix_reference_indices"])
print("stage JSON validity:")
print(json.dumps(summary["stage_valid_json"], ensure_ascii=False, indent=2))
print("QA runs:")
for run in summary["runs"]:
    print(
        run["run_name"],
        f"{run['relaxed_correct']}/{run['total']} = {run['relaxed_accuracy']:.2%}",
        "recovered:",
        run["recovered_indices"],
    )
print("oracle:", f"{summary['oracle_recovered_count']}/{summary['subset_total_after_excluding_reference_issues']} = {summary['oracle_recovered_accuracy']:.2%}")
print("still wrong:", summary["still_wrong_count"])
print("\nReport:", REPORT_MD)
print(REPORT_MD.read_text(encoding="utf-8")[:4000])


有效样本数（剔除 reference/evaluator 高风险样本后）: 77
跳过样本: [12, 14, 105, 158, 470, 918, 1351, 1561]
stage JSON validity:
{
  "overview": 77,
  "axes_legend": 75,
  "data_table": 74
}
QA runs:
staged_table_json_only 15/77 = 19.48% recovered: [132, 169, 175, 186, 220, 368, 397, 419, 441, 623, 677, 700, 800, 1327, 1810]
staged_image_plus_table_json 16/77 = 20.78% recovered: [132, 162, 169, 175, 220, 233, 245, 269, 368, 426, 623, 832, 974, 1114, 1327, 1810]
oracle: 23/77 = 29.87%
still wrong: 54

Report: /content/qwen25vl-chartqa-qlora/outputs/chartqa_all_wrong_diagnostics/staged_extraction/staged_extraction_report.md
# Module 22B staged chart-to-table extraction report

Valid subset size after excluding reference issues: 77
Skipped exclude/fix-reference samples: [12, 14, 105, 158, 470, 918, 1351, 1561]

## Stage JSON validity

| stage | valid JSON |
|---|---:|
| `overview` | 77 |
| `axes_legend` | 75 |
| `data_table` | 74 |

## QA runs

| run | relaxed |
|---|---:|
| `staged_table_json_only` | 15/77 

## 22B.4 中文说明

本模块验证的是“分步结构化读图”是否比 Module 21 的一次性 chart-to-JSON 更稳。

判断口径：

- 如果 `overview/axes_legend/data_table` 的 JSON 合法率明显高于 Module 21 的一次性 extraction，说明分步 schema 更可控。
- 如果 `staged_table_json_only` 追回更多样本，说明结构化抽取本身有效。
- 如果 `staged_image_plus_table_json` 更好，说明 extraction 仍不完整，图像二次校验有帮助。
- 如果二者都不如 Module 21 的 one-shot extraction，则应先人工审查 staged prompts，而不是扩到 full-val。

本模块仍然不建议训练新 LoRA，也不建议 full-val 扩展。只有在 77 条有效 subset 上看到清晰收益后，才进入更大规模验证。


# ChartQA Module 22B staged extraction results - 2026-07-02

## 运行状态

Module 22B 已完整运行完成：

- `22B.1` 成功恢复 subset、exclude list 和 helper script；
- `22B.2` 成功完成 staged extraction 和两种 QA；
- `22B.3` 成功读取 summary；
- notebook cell 无报错，`returncode=0`。

本模块没有跑 full-val，也没有训练新 LoRA。

## 数据口径

原始 recommended diagnostic subset 有 85 条。

Module 22A 标出的 8 条 `exclude_or_fix_reference` 样本被默认跳过：

```text
12, 14, 105, 158, 470, 918, 1351, 1561
```

所以 Module 22B 的有效诊断样本数为：

```text
77
```

## Staged extraction JSON 合法率

| stage | valid JSON |
|---|---:|
| `overview` | 77/77 |
| `axes_legend` | 75/77 |
| `data_table` | 74/77 |

无效 JSON 样本：

```text
axes_legend: 1190, 1114
data_table: 13, 1327, 233
```

解释：

- 分步 schema 明显提升了 JSON 可解析率。
- Module 21 one-shot chart-to-JSON 是 66/85；22B 的关键 `data_table` 阶段是 74/77。
- 但 JSON 合法并不等于数值抽取正确。后续要看 QA 是否提升。

## 22B QA 结果

| run | relaxed |
|---|---:|
| `staged_table_json_only` | 15/77 = 19.48% |
| `staged_image_plus_table_json` | 16/77 = 20.78% |
| 22B oracle | 23/77 = 29.87% |

22B 追回样本：

```text
132, 162, 169, 175, 186, 220, 233, 245, 269, 368, 397, 419,
426, 441, 623, 677, 700, 800, 832, 974, 1114, 1327, 1810
```

仍错误样本数：

```text
54/77
```

## 与 Module 21 对比

使用同样的 77 条有效样本口径：

| method group | oracle |
|---|---:|
| Module 21 high-res / one-shot table diagnostics | 24/77 = 31.17% |
| Module 22B staged extraction diagnostics | 23/77 = 29.87% |
| Module 21 + Module 22B combined oracle | 30/77 = 38.96% |

结论：

- 22B 单独没有超过 Module 21 的整体 oracle。
- 但 22B 追回了 6 个 Module 21 没追回的新样本。
- 组合后从 24/77 提升到 30/77，说明 staged extraction 有互补性，但还不是更强的单一路线。

## 22B 独有追回样本

22B 相对 Module 21 独有追回：

```text
186, 245, 419, 441, 677, 832
```

类别分布：

| reviewed_primary | count |
|---|---:|
| `date_axis_reading` | 2 |
| `counting_or_category_count` | 1 |
| `text_label_lookup` | 1 |
| `visual_mapping_or_legend` | 1 |
| `yes_no_or_boolean` | 1 |

解释：

- staged extraction 对部分 date-axis 和 visual/text/boolean 样本确实有新增价值。
- 但新增样本不集中在 numeric/calculation 主瓶颈上。

## Module 21 独有、22B 没追回的样本

Module 21 相对 22B 独有追回：

```text
13, 24, 190, 255, 362, 369, 882
```

类别分布：

| reviewed_primary | count |
|---|---:|
| `counting_or_category_count` | 2 |
| `date_axis_reading` | 1 |
| `multi_step_calculation` | 1 |
| `numeric_value_or_scale` | 1 |
| `text_label_lookup` | 1 |
| `yes_no_or_boolean` | 1 |

解释：

- staged extraction 在一些原本 one-shot/table 或 high-res 能解决的样本上发生回退。
- 这说明分步过程虽然更可解析，但可能在 stage 间传递时丢信息，或者 data_table 阶段没有保留足够数值。

## Combined oracle by category

Module 21 + 22B 合并后，按类别追回：

| reviewed_primary | recovered / total |
|---|---:|
| `counting_or_category_count` | 6/8 |
| `yes_no_or_boolean` | 5/9 |
| `text_label_lookup` | 7/15 |
| `date_axis_reading` | 4/9 |
| `multi_step_calculation` | 3/9 |
| `visual_mapping_or_legend` | 2/9 |
| `numeric_value_or_scale` | 2/10 |
| `extreme_value_or_ranking` | 1/8 |

仍然最难的是：

- numeric value / scale;
- extreme/ranking;
- visual mapping / legend;
- multi-step calculation。

## 当前技术判断

1. 分步抽取提高了 JSON 格式稳定性。

`overview`、`axes_legend`、`data_table` 的合法 JSON 率都高于 one-shot extraction。这说明 staged schema 是更可控的工程方向。

2. 但分步抽取没有带来足够的 QA 增益。

22B oracle 是 23/77，略低于 Module 21 的 24/77。说明问题不只是 JSON 格式，而是抽取内容的正确性，尤其是 data values、axis/date alignment、legend mapping 是否真的抽对。

3. image + staged table 略好于 table-only。

`staged_image_plus_table_json` 是 16/77，高于 `staged_table_json_only` 的 15/77。这说明 extraction 还不完整，模型仍需要回看图像。

4. 22B 有互补性，但不能直接替代 Module 21。

组合 oracle 提升到 30/77，说明 staged extraction 值得保留为诊断路线，但不能说它已经是更强 pipeline。

5. 现在仍不建议继续训练 LoRA 或扩 full-val。

剩余 47/77 连 Module 21 + 22B 都没救回。继续 full-val 只会放大成本，不能解释失败原因。

## 下一步建议

下一步不应直接扩展 22B，而应先做 22C：抽取质量审计。

建议：

```text
Module 22C: staged extraction quality audit
```

目标：

1. 对 77 条的 `overview / axes_legend / data_table` 做抽取质量打分；
2. 标出失败发生在哪个阶段：
   - axis/tick/date 错；
   - legend/color mapping 错；
   - data point 缺失；
   - 数值尺度错；
   - table 对了但 QA 算错；
3. 对 22B 独有追回和 Module 21 独有追回分别抽样看原因；
4. 再决定是否做 crop-based extraction。

推荐 22C 输出：

```text
staged_extraction_quality_audit.csv
staged_extraction_quality_audit_summary.json
```

目前最有价值的问题不是“能不能多跑一点”，而是：

> 为什么 JSON 合法率很高，但 QA 只追回 20% 左右？

这个问题答清楚之后，才值得进入 crop、OCR、或更细 schema。


# ChartQA Module 22C staged extraction quality audit - 2026-07-03

## 运行口径

本模块已按要求作为纯本地审计完成：没有加载模型，没有调用 GPU，没有跑 full-val，也没有训练 LoRA。

输入来自已经落盘的 Module 21 / 22A / 22B 结果镜像：

- Module 22B staged extraction：`outputs/chartqa_all_wrong_diagnostics_from_drive/staged_extraction`
- Module 21 evaluated runs：`outputs/chartqa_all_wrong_diagnostics_from_drive/evaluated`
- Module 22A reference/evaluator 排除表：`outputs/chartqa_evaluator_cleanup`
- diagnostic subset：`data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl`

22A 排除 8 个高优先级 reference/evaluator 问题样本后，本轮有效样本仍为：

```text
77
```

## 总体结果

| item | count |
|---|---:|
| Module 21 oracle | 24/77 |
| Module 22B oracle | 23/77 |
| Module 21 + 22B combined oracle | 30/77 = 38.96% |
| combined still wrong | 47/77 |

22B 独有追回：

```text
186, 245, 419, 441, 677, 832
```

Module 21 独有追回：

```text
13, 24, 190, 255, 362, 369, 882
```

两者都没追回：

```text
18, 28, 29, 46, 138, 154, 189, 229, 241, 250, 251, 281, 290, 291, 310, 312, 317, 326, 344, 381, 408, 424, 455, 467, 523, 529, 571, 648, 667, 675, 688, 778, 779, 781, 797, 804, 816, 877, 901, 946, 976, 977, 978, 987, 1055, 1065, 1190
```

## 自动归因分层

| relation / suspected layer | count |
|---|---:|
| `22b_unique_recovery` | 6 |
| `axis_legend_json_failure_but_table_parsed` | 1 |
| `both_recovered` | 17 |
| `likely_reasoning_or_aggregation_error:extreme_value_or_ranking` | 3 |
| `likely_reasoning_or_aggregation_error:multi_step_calculation` | 6 |
| `likely_visual_extraction_error:date_axis_reading` | 1 |
| `likely_visual_extraction_error:numeric_value_or_scale` | 5 |
| `likely_visual_extraction_error:visual_mapping_or_legend` | 4 |
| `module21_unique_recovery` | 7 |
| `still_wrong_needs_manual_visual_audit` | 7 |
| `table_may_contain_answer_but_qa_failed` | 20 |

说明：

- `22b_unique_recovery` 表示分阶段抽取相对 Module 21 有独立价值，适合看它到底补上了哪类视觉或语义线索。
- `module21_unique_recovery` 表示 22B 分阶段链路反而损失了信息，适合检查 staged table 是否遗漏或误改了原图线索。
- `table_may_contain_answer_but_qa_failed` 表示表格文本里启发式能找到 reference，但最终 QA 仍错，更像 QA 推理、格式化或答案选择失败。
- `stage_json_or_schema_failure` / `axis_legend_json_failure_but_table_parsed` 是格式或阶段 schema 层问题。
- `likely_visual_extraction_error:*` 与 `likely_reasoning_or_aggregation_error:*` 是按人工类别和输出状态做的初筛，仍需人工看图确认。

## 按人工主类别看恢复情况

| reviewed_primary | total | combined recovered | 22B unique | Module21 unique | both |
|---|---:|---:|---:|---:|---:|
| `counting_or_category_count` | 8 | 6 | 1 | 2 | 3 |
| `date_axis_reading` | 9 | 4 | 2 | 1 | 1 |
| `extreme_value_or_ranking` | 8 | 1 | 0 | 0 | 1 |
| `multi_step_calculation` | 9 | 3 | 0 | 1 | 2 |
| `numeric_value_or_scale` | 10 | 2 | 0 | 1 | 1 |
| `text_label_lookup` | 15 | 7 | 1 | 1 | 5 |
| `visual_mapping_or_legend` | 9 | 2 | 1 | 0 | 1 |
| `yes_no_or_boolean` | 9 | 5 | 1 | 1 | 3 |

## 高价值人工复核队列

已生成：

```text
outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_high_value_manual_review_queue.csv
```

优先看三类：

1. 22B 独有追回样本：确认 staged extraction 捕捉到了哪些 Module21 没捕捉到的线索。
2. Module21 独有追回样本：确认 22B 的 staged table 是否丢失信息或引入错误。
3. table 里疑似含 reference 但 QA 仍错的样本：判断下一步是否需要改 QA prompt/答案规范化，而不是继续改视觉抽取。

schema 或 JSON 层异常样本：

```text
1190
```

table 疑似含 reference 但 QA 仍错样本：

```text
28, 138, 154, 189, 229, 250, 281, 317, 326, 424, 529, 675, 778, 779, 781, 877, 946, 977, 987, 1065
```

## 当前判断

22C 的结论延续 22B：分阶段抽取让输出更可控，但不是单独更强的主线。它的价值在于提供互补样本和可审计中间态。

现在最值得继续的是小规模人工复核高价值队列，而不是马上 full-val 或继续 LoRA。复核目标不是重新判断总分，而是判断失败层级：

- 若 table 已有正确值但 QA 错，下一步优先做 QA prompt / answer normalization。
- 若 table 没有正确值，下一步优先做视觉读数、轴刻度、legend/color mapping 的专项提示或裁剪策略。
- 若 22B 相比 Module21 丢失正确样本，说明 staged extraction 有信息压缩损失，不能直接替代 image-only / one-shot 路线。

## 输出文件

- `outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_staged_extraction_quality_audit.csv`
- `outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_staged_extraction_quality_audit.json`
- `outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_high_value_manual_review_queue.csv`
- `outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_quality_audit_summary.json`
- `docs/experiments/chartqa_staged_extraction_quality_audit_22c_2026-07-03.md`


# ChartQA 22C Codex visual review - 2026-07-03

## Scope

This is a local, no-GPU visual review of the 34 high-value samples from Module 22C.
The review reads the mounted Google Drive images and existing Module 21 / 22B outputs.
No model inference, training, or full-val run was performed.

Source queue:

```text
outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_high_value_manual_review_queue.csv
```

Drive image source:

```text
G:\我的云端硬盘\qwen25vl-chartqa-qlora\data\processed\chartqa_val_full_sft_1920_images
```

## Executive Finding

The high-value queue is not a single failure mode. It splits into four useful groups:

| group | samples | interpretation |
|---|---:|---|
| clear 22B/staged-extraction useful recoveries | 4-6 | staged table helps on counting, color/line mapping, local comparison, and arithmetic when the table is trusted |
| true visual / semantic / arithmetic failures | ~17 | still need better visual grounding, strict threshold handling, range aggregation, spatial grounding, or QA computation |
| answer normalization / evaluator issues | ~4 | model output is essentially correct but evaluator rejects wording, punctuation, star suffix, or close numeric format |
| likely reference/question issues | ~6 | chart contradicts the reference or the question is ambiguous enough that the sample should be excluded or separately tagged |

The current conclusion remains: Module 22B should be kept as a diagnostic and complementary route, but it should not replace Module 21 style image-only / one-shot runs. The next improvement target is not more JSON formatting. It is failure-layer separation:

- value extraction / color mapping / axis grounding;
- strict comparison and range aggregation;
- QA prompt forcing short final answers;
- evaluator normalization and reference cleanup.

## Reviewed Samples

| sample | relation | visual review judgment |
|---:|---|---|
| 13 | Module21 unique | Reference 155 is visually correct: peaks are about 80, 68, and 7. 22B loses peak values or sum reasoning. True staged extraction / aggregation failure. |
| 24 | Module21 unique | Iraqi dependents is the 12% slice, while 22B picks 19% Iraqi principal applicants. True role/group disambiguation failure. |
| 186 | 22B unique | X-axis shows 9 labeled years. 22B table-only correctly answers 9; image+table drifts to all years. Staged table is useful here. |
| 190 | Module21 unique | Only Southern Asia is greater than 60%; Eastern Asia is approximately equal to 60%, not greater. 22B fails strict inequality. |
| 245 | 22B unique | Peak year is visually close/ambiguous between 1980 and 2000. Reference is 1980, but 2000 appears at least comparable. Mark as ambiguous peak/reference-sensitive. |
| 255 | Module21 unique | Only 1998 is clearly below 15 dollars/barrel. 22B counts near-threshold years. True strict threshold / visual precision failure. |
| 362 | Module21 unique | Chart countries are Czech Republic and New Zealand. 22B outputs the same labels but evaluator rejects list format. Normalization/evaluator issue. |
| 369 | Module21 unique | Bars are 1.2k and 4.5k; average is 2.85k, not greater than 3k. 22B says Yes, likely unit-scale reasoning failure. |
| 419 | 22B unique | Ukraine 34.5 is less than the sum of the other four countries, 36.6. 22B table-only answers No correctly; image+table is distracted by longest bar. |
| 441 | 22B unique | Red line is below blue only in 2009. 22B table-only gets this right. Useful staged color/line mapping case. |
| 677 | 22B unique | Poland 21.8 vs Austria 21.5 differs by 0.3. 22B table-only gets this right; image+table confuses nearby Sweden. |
| 832 | 22B unique | Light-blue shortest is 21 and dark-blue tallest is 38. Reference 17 treats subtraction as absolute difference; literal subtraction is -17. Mark as sign/wording ambiguity. |
| 882 | Module21 unique | Chart legend has 2028*, reference is 2028. 22B predicts 2028*. Normalization issue, not visual failure. |
| 1190 | axis/legend JSON issue | Reference 2008 is questionable: increases occur in several later years, while 2008 has no previous comparison. Mark as question/reference ambiguity. |
| 28 | table contains reference but QA failed | Dark grey segment corresponds to Both. 22B answers Don't know. True color/legend mapping failure. |
| 138 | table contains reference but QA failed | Smallest value is South Sudan 0.53%, not Angola 41.39%. Reference appears wrong; 22B answer is visually correct. |
| 154 | table contains reference but QA failed | Correct answer is orange. 22B image+table says the answer is orange inside a sentence, but evaluator rejects it. Final-answer normalization issue. |
| 189 | table contains reference but QA failed | Four true age groups are 15-49, 50-69, 70+, and 5-14. All ages / age-standardized should not count. 22B semantic filtering failure. |
| 229 | table contains reference but QA failed | Male series peak is 2018, slightly higher than 2016. 22B picks 2016. True near-peak date-axis reading failure. |
| 250 | table contains reference but QA failed | Agriculture series peaks in 1999; 2000 is slightly lower. 22B picks 2000. True neighboring-year visual precision failure. |
| 281 | table contains reference but QA failed | Highest navy blue is 47; median light blue is about 28; sum is 75, not greater than 100. QA arithmetic/median failure. |
| 317 | table contains reference but QA failed | All voters: Very=57 and Somewhat=34, so answer should be Yes. Reference says No. Reference error. |
| 326 | table contains reference but QA failed | Rightmost bar in the middle row is 50. 22B picks 63 from another row. Spatial grounding failure. |
| 424 | table contains reference but QA failed | Brown/orange bar is Northwest Europe marker price. 22B returns either value or wrong label. Color-to-label mapping failure. |
| 529 | table contains reference but QA failed | YellowPages is 12%; three times is 36%, matching Industry specific. 22B fails multiplicative target lookup. |
| 675 | table contains reference but QA failed | People over 55 means 55-64 plus 65+, 14+8=22. 22B only takes 65+. Range aggregation failure. |
| 778 | table contains reference but QA failed | Very likely is highest for 30-44 at 17%, slightly above 18-29 at 16%. 22B fails close-value max selection. |
| 779 | table contains reference but QA failed | Minimum difference between Not likely at all and Very likely is not 65+; 65+ is the largest gap. Reference likely wrong. |
| 781 | table contains reference but QA failed | Biggest gender wage gap is Some college or associate's degree, about 248. 22B chooses Advanced degree, likely using max value rather than max gap. |
| 877 | table contains reference but QA failed | Twitter has the smallest male/female gap, about 1 point. 22B picks Pinterest, likely using minimum single value instead of minimum difference. |
| 946 | table contains reference but QA failed | Grocers average is higher than General Merchandisers. Reference says General Merchandisers; reference likely wrong. |
| 977 | table contains reference but QA failed | Question asks for 2020 ICT sector value; chart value is 11.8 for ICT services. Reference 2022 is not consistent with the question. Reference/question mismatch. |
| 987 | table contains reference but QA failed | Question wording is ambiguous. Reference 2013 looks like a start year, while 22B picks 2019 as peak/latest increase. Needs reference/question cleanup. |
| 1065 | table contains reference but QA failed | Home furnishings is about 65.3%; 22B says 65 or 65%. This should be accepted under relaxed numeric matching. Evaluator/normalization issue. |

## Recommended Action

1. Add a cleanup tag list for samples that should not be counted as model failures:
   `138, 317, 362, 779, 882, 946, 977, 987, 1065, 1190`.
   Sample `245` and `832` should be marked ambiguous/reference-sensitive rather than clean failures.

2. Add answer normalization for:
   - list answers such as `Czech Republic, New Zealand`;
   - starred years such as `2028* -> 2028`;
   - sentence answers that contain the exact categorical answer, such as `... orange`;
   - percentage strings and close numeric values such as `65%` vs `65.3`.

3. For true model failures, prioritize targeted diagnostics:
   - strict inequality and near-threshold comparisons: `190, 255, 778`;
   - date-axis peak/neighbor precision: `229, 250`;
   - unit/scale reasoning: `369`;
   - range aggregation: `675`;
   - spatial grounding: `326`;
   - color/legend mapping: `28, 424`;
   - arithmetic/difference/median reasoning: `281, 529, 781, 877`.

4. Do not launch full-val yet. First convert this review into a cleaned denominator and a small prompt/evaluator ablation, otherwise metric changes will mix real model gains with dataset/evaluator cleanup.


# ChartQA Module 23A cleanup + normalization-only ablation - 2026-07-03

## 运行口径

Module 23A 已完成。它是纯本地后处理模块：

- 不加载模型；
- 不使用 GPU；
- 不改任何 prediction；
- 不跑 full-val；
- 只读取现有 Module 21 / 22B evaluated JSONL。

本模块分两步：

1. 应用 cleanup list：在 22A 原有 8 条 exclude 的基础上，加入 Codex 视觉复核标记的 10 条 reference/evaluator 问题样本。
2. 做 normalization-only 消融：在 22A 后的 77 条 subset 上，只改 answer normalization，观察能追回多少。

## Cleanup List

22A 原始排除：

```text
12, 14, 105, 158, 470, 918, 1351, 1561
```

23A 新增 Codex cleanup：

```text
138, 317, 362, 779, 882, 946, 977, 987, 1065, 1190
```

23A 扩展后 exclude 总数：

```text
18
```

因此 clean-after-23A denominator 为：

```text
67
```

另有两个样本只标记为 ambiguous/reference-sensitive，默认不加入 exclude：

```text
245, 832
```

## Normalization Rules

23A 新增 normalization 只覆盖以下规则：

- list answer format：例如 `Czech Republic, New Zealand` vs `[Czech Republic, New Zealand]`；
- star year：例如 `2028* -> 2028`；
- categorical answer contained in sentence：例如句子中明确包含 `orange`；
- percent / close numeric：例如 `65%` 或 `65` vs `65.3`。

没有加入会扩大语义边界的规则，例如把 reference 错误的样本强行判对。

## 77 条上的 normalization-only 消融

| metric | before | after | gain |
|---|---:|---:|---:|
| oracle on valid77 | 30/77 = 38.96% | 32/77 = 41.56% | +2 |

normalization-only 追回的 unique sample：

```text
154, 455
```

按规则分布：

| rule | unique samples | sample indices |
|---|---:|---|
| `categorical_answer_contained_in_sentence` | 5 | `154, 362, 426, 1114, 1327` |
| `exact_after_text_cleanup` | 3 | `362, 455, 882` |

## Clean Denominator 口径

应用 23A expanded cleanup 后：

| metric | before normalization | after normalization |
|---|---:|---:|
| oracle on clean-after-23A | 28/67 = 41.79% | 30/67 = 44.78% |

## Per-run Summary

| run | valid77 before | valid77 after | gain | clean before | clean after |
|---|---:|---:|---:|---:|---:|
| `baseline_maxpix_802816` | 9/77 | 11/77 | +2 | 9/67 | 9/67 |
| `f_maxpix_802816` | 11/77 | 11/77 | +0 | 9/67 | 9/67 |
| `hardmix_axis_legend_prompt_802816` | 9/77 | 10/77 | +1 | 9/67 | 9/67 |
| `hardmix_maxpix_602112` | 12/77 | 13/77 | +1 | 11/67 | 11/67 |
| `hardmix_maxpix_802816` | 12/77 | 13/77 | +1 | 11/67 | 11/67 |
| `image_plus_table_json` | 12/77 | 14/77 | +2 | 12/67 | 13/67 |
| `table_json_only` | 13/77 | 16/77 | +3 | 13/67 | 15/67 |
| `staged_table_json_only` | 15/77 | 19/77 | +4 | 15/67 | 17/67 |
| `staged_image_plus_table_json` | 16/77 | 19/77 | +3 | 16/67 | 17/67 |

## 当前判断

23A 把两件事分开了：

- normalization-only 的收益说明有多少“答案已经基本对，但 evaluator 没吃到”；
- expanded cleanup 的收益说明当前 subset 里有多少不适合作为模型失败统计的 reference/evaluator 问题。

如果后续要做硬失败定向诊断，建议使用 clean-after-23A denominator，同时保留 ambiguous 样本单独统计。这样下一轮 strict threshold、date-axis、range aggregation、spatial grounding 等诊断不会被 reference cleanup 和 answer formatting 干扰。

## 输出文件

- `outputs/chartqa_23a_cleanup_normalization/chartqa_23a_expanded_cleanup_exclude_list.csv`
- `outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv`
- `outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_recovered_predictions.csv`
- `outputs/chartqa_23a_cleanup_normalization/chartqa_23a_run_summary.csv`
- `outputs/chartqa_23a_cleanup_normalization/chartqa_23a_summary.json`
- `docs/experiments/chartqa_23a_cleanup_normalization_2026-07-03.md`


# ChartQA Module 23B Codex targeted hard-failure review - 2026-07-03

## Scope

This review completes the targeted diagnostics requested after Module 23A. It uses the clean-after-23A denominator and reviews only samples that remain wrong after answer normalization.

No GPU, model inference, training, or full-val run was used. Inputs are existing Module 21 / 22B / 23A outputs plus G-drive images.

## Starting Point

Module 23A clean denominator:

```text
67
```

Module 23A normalized oracle:

```text
30/67 = 44.78%
```

Remaining hard-failure queue before this targeted review:

```text
37
```

Hard-failure indices:

```text
18, 28, 29, 46, 189, 229, 241, 250, 251, 281, 290, 291, 310, 312, 326, 344, 381, 408, 424, 467, 523, 529, 571, 648, 667, 675, 688, 778, 781, 797, 804, 816, 877, 901, 976, 978, 1055
```

## Refined Diagnosis

The 37 remaining samples are not all true model hard failures.

| refined group | count | samples |
|---|---:|---|
| true hard failures | 28 | 28, 29, 189, 229, 250, 251, 281, 290, 291, 310, 312, 326, 344, 381, 408, 424, 467, 523, 529, 667, 675, 778, 781, 797, 804, 877, 901, 1055 |
| residual normalization candidates | 5 | 18, 241, 648, 688, 816 |
| residual reference candidates | 3 | 46, 571, 978 |
| residual question / answer-extraction candidate | 1 | 976 |

So the effective hard-failure set after 23B manual refinement is closer to:

```text
28/67
```

This is not a new official score yet; it is a diagnostic denominator recommendation.

## True Hard Failure Buckets

| bucket | samples | interpretation |
|---|---|---|
| legend / color / visual encoding binding | 28, 290, 312, 424 | Model fails to bind color or visual encoding to the right semantic label/value. |
| semantic filtering / counting / threshold count | 189, 251 | Model counts aggregate series or estimates instead of enumerating valid categories/years. |
| date-axis near-peak precision | 229, 250 | Model chooses a neighboring year when peaks are close. |
| spatial grounding | 326, 344 | Positional phrases like rightmost/middle/bottom are mapped to the wrong segment. |
| arithmetic / aggregation | 281, 291, 381, 467, 523 | Model needs to list operands first, then compute. |
| label after computation | 529, 667, 675, 778, 781, 877, 901 | Model must compute a target value/range/difference first, then return a label. |
| stacked bar total vs segment | 804 | Model reports a segment when the reference expects the stacked total. |
| multi-answer list | 29, 408, 797 | Model outputs one value/year instead of an exhaustive list. |
| crop/OCR visibility | 1055 | Label/value is truncated in the image; needs crop/zoom/OCR. |
| series vs total disambiguation | 310 | Model chooses total bar instead of a requested segment. |

## Residual Evaluator / Data Issues

These should be handled before using the 37 as a hard-failure benchmark:

| sample | issue |
|---:|---|
| 18 | prediction says blue / hex code while reference says Light blue; color-name normalization candidate |
| 46 | chart visibly shows 0.34 while reference is 0.0034; scale/reference mismatch |
| 241 | prediction sentence contains correct 900, but parser grabs the year 1990 first |
| 571 | chart shows Sudan (April 1983) as 150000, not reference 250000 |
| 648 | predictions say Increase/increases; reference is increasing |
| 688 | outputs often contain the correct two labels but order/plural/singular normalization misses them |
| 816 | prediction says blue / hex code while reference says light blue |
| 976 | question appears truncated; some output contains Danish citizens but with uncertainty |
| 978 | question asks software and ICT solutions in 2019, chart value is 7.7; reference 12.3 appears to be ICT services |

## Recommended Next Module

Do not train or full-val yet. The next useful module should be a targeted prompt/evaluator ablation over the 28 true hard failures:

1. `legend_table_prompt`: explicitly extract legend/color mapping before answering.
2. `operand_table_prompt`: force list of operands and computation expression before final answer.
3. `spatial_locator_prompt`: force row/column/segment localization for positional language.
4. `range_count_prompt`: force enumerate all categories/years satisfying a threshold/range condition.
5. `multi_answer_prompt`: force exhaustive list output when the reference is a list.

Separately, update evaluator normalization for:

- color synonyms and hex-to-color names;
- morphological trend words such as increase/increases/increasing;
- numeric answer extraction from sentences when the answer is near the end and the question contains years;
- list order plus singular/plural variants.

## Output Files

- `outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_hard_failure_queue.csv`
- `outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv`
- `outputs/chartqa_23b_hard_failure_diagnostics/contact_sheets/*.png`
- `docs/experiments/chartqa_23b_hard_failure_diagnostics_2026-07-03.md`
- `docs/experiments/chartqa_23b_codex_targeted_review_2026-07-03.md`


# Module 23C - normalization v2 and targeted prompt ablation

This module has two parts:

1. evaluator-only normalization v2 on existing predictions;
2. small targeted prompt ablation on the 28 true-hard samples from Module 23B.

It does **not** run full validation and does **not** train a new adapter. The prompt ablation is intentionally small: routed mode runs one targeted prompt for each true-hard sample, so the default run is 28 generations.

Long-running prediction outputs are append-only JSONL, restored from Drive when available, and skipped on rerun by `(sample_index, prompt_name)`.

## 23C.1 Restore scripts, inputs, and adapter


In [5]:
# Module 23C.1 - restore scripts, inputs, and adapter
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_SCRIPT_DIR = DRIVE_ROOT / "scripts_module23c"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_23c_targeted_prompt_ablation"
DRIVE_NORM_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_23c_normalization_v2"

SCRIPT_NAMES = [
    "run_chartqa_23c_normalization_v2.py",
    "run_chartqa_23c_targeted_prompt_ablation.py",
]

LOCAL_SUBSET_JSONL = REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl"
DRIVE_SUBSET_JSONL = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/data/chartqa_all_wrong_diagnostic_subset_85.jsonl"

LOCAL_23A_PER_PRED = REPO / "outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv"
DRIVE_23A_PER_PRED = DRIVE_ROOT / "outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv"

LOCAL_23B_REVIEW = REPO / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv"
DRIVE_23B_REVIEW = DRIVE_ROOT / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv"

HARDMIX_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
DRIVE_HARDMIX_ADAPTER = DRIVE_ROOT / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"

for path in [REPO, DRIVE_ROOT, DRIVE_SCRIPT_DIR, DRIVE_SUBSET_JSONL, DRIVE_23A_PER_PRED, DRIVE_23B_REVIEW]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path: {path}")

%cd /content/qwen25vl-chartqa-qlora

for script_name in SCRIPT_NAMES:
    local_script = REPO / "scripts" / script_name
    drive_script = DRIVE_SCRIPT_DIR / script_name
    local_script.parent.mkdir(parents=True, exist_ok=True)
    if local_script.exists():
        print("Script already present:", local_script)
    elif drive_script.exists():
        shutil.copy2(drive_script, local_script)
        print("Restored script from Drive:", drive_script, "->", local_script)
    else:
        raise FileNotFoundError(f"Missing helper script in repo and Drive: {script_name}")

for local_path, drive_path in [
    (LOCAL_SUBSET_JSONL, DRIVE_SUBSET_JSONL),
    (LOCAL_23A_PER_PRED, DRIVE_23A_PER_PRED),
    (LOCAL_23B_REVIEW, DRIVE_23B_REVIEW),
]:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if not local_path.exists():
        shutil.copy2(drive_path, local_path)
        print("Restored input:", local_path)

def copytree_if_needed(src: Path, dst: Path) -> None:
    if dst.exists():
        print("Adapter already present:", dst)
        return
    if not src.exists():
        raise FileNotFoundError(f"Missing adapter source: {src}")
    shutil.copytree(src, dst)
    print("Restored adapter:", dst)

copytree_if_needed(DRIVE_HARDMIX_ADAPTER, HARDMIX_ADAPTER)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_NORM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Module 23C restore complete.")
print("23A per-prediction:", LOCAL_23A_PER_PRED)
print("23B targeted review:", LOCAL_23B_REVIEW)
print("Hardmix adapter:", HARDMIX_ADAPTER)


Mounted at /content/drive
/content/qwen25vl-chartqa-qlora
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module23c/run_chartqa_23c_normalization_v2.py -> /content/qwen25vl-chartqa-qlora/scripts/run_chartqa_23c_normalization_v2.py
Restored script from Drive: /content/drive/MyDrive/qwen25vl-chartqa-qlora/scripts_module23c/run_chartqa_23c_targeted_prompt_ablation.py -> /content/qwen25vl-chartqa-qlora/scripts/run_chartqa_23c_targeted_prompt_ablation.py
Restored input: /content/qwen25vl-chartqa-qlora/data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl
Restored input: /content/qwen25vl-chartqa-qlora/outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv
Restored input: /content/qwen25vl-chartqa-qlora/outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv
Restored adapter: /content/qwen25vl-chartqa-qlora/outputs/adapters/chartqa_qlora_hardmix1k_steps100
Module 23C restore complete.
23A per

## 23C.2 Run normalization v2 evaluator-only


In [6]:
# Module 23C.2 - evaluator-only normalization v2
from pathlib import Path
import subprocess
import shutil

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

cmd = [
    "python", "scripts/run_chartqa_23c_normalization_v2.py",
    "--per-prediction-csv", "outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv",
    "--targeted-review-csv", "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
    "--output-dir", "outputs/chartqa_23c_normalization_v2",
    "--report-md", "docs/experiments/chartqa_23c_normalization_v2_2026-07-03.md",
]
subprocess.run(cmd, check=True)

drive_out = DRIVE_ROOT / "outputs/chartqa_23c_normalization_v2"
drive_out.mkdir(parents=True, exist_ok=True)
for path in (REPO / "outputs/chartqa_23c_normalization_v2").glob("*"):
    if path.is_file():
        shutil.copy2(path, drive_out / path.name)
report = REPO / "docs/experiments/chartqa_23c_normalization_v2_2026-07-03.md"
if report.exists():
    shutil.copy2(report, drive_out / report.name)
print("Normalization v2 outputs copied to:", drive_out)


Normalization v2 outputs copied to: /content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_23c_normalization_v2


## 23C.3 Run routed targeted prompt ablation


In [7]:
# Module 23C.3 - small routed targeted prompt ablation on 28 true-hard samples
from pathlib import Path
import subprocess
import torch

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the targeted prompt ablation cell.")

cmd = [
    "python", "scripts/run_chartqa_23c_targeted_prompt_ablation.py",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--targeted-review-csv", "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
    "--output-dir", "outputs/chartqa_23c_targeted_prompt_ablation",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_23c_targeted_prompt_ablation"),
    "--adapter-path", "outputs/adapters/chartqa_qlora_hardmix1k_steps100",
    "--adapter-name", "hardmix",
    "--prompt-policy", "routed",
    "--max-pixels", "802816",
    "--max-new-tokens", "96",
    "--sync-every", "1",
]
subprocess.run(cmd, check=True)


CompletedProcess(args=['python', 'scripts/run_chartqa_23c_targeted_prompt_ablation.py', '--subset-jsonl', 'data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl', '--targeted-review-csv', 'outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv', '--output-dir', 'outputs/chartqa_23c_targeted_prompt_ablation', '--drive-output-dir', '/content/drive/MyDrive/qwen25vl-chartqa-qlora/outputs/chartqa_23c_targeted_prompt_ablation', '--adapter-path', 'outputs/adapters/chartqa_qlora_hardmix1k_steps100', '--adapter-name', 'hardmix', '--prompt-policy', 'routed', '--max-pixels', '802816', '--max-new-tokens', '96', '--sync-every', '1'], returncode=0)

## 23C.4 Read summaries


In [8]:
# Module 23C.4 - read normalization and prompt-ablation summaries
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")

paths = [
    REPO / "outputs/chartqa_23c_normalization_v2/chartqa_23c_normalization_v2_summary.json",
    REPO / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed_metrics.json",
]

for path in paths:
    print("\n==", path, "==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)[:5000])
    else:
        print("Not found yet.")



== /content/qwen25vl-chartqa-qlora/outputs/chartqa_23c_normalization_v2/chartqa_23c_normalization_v2_summary.json ==
{
  "created_at": "2026-07-04T03:41:38",
  "gpu_or_model_used": false,
  "model_predictions_changed": false,
  "clean_after_23a_total": 67,
  "oracle_v1_count": 30,
  "oracle_v1_accuracy": 0.44776119402985076,
  "oracle_v2_count": 34,
  "oracle_v2_accuracy": 0.5074626865671642,
  "oracle_v2_gain": 4,
  "oracle_v2_recovered_indices": [
    18,
    241,
    648,
    816
  ],
  "normalization_v2_recovered_by_rule": {
    "color_synonym_or_hex": [
      18,
      816
    ],
    "numeric_answer_in_sentence": [
      241
    ],
    "trend_morphology": [
      648
    ]
  },
  "true_hard_before_v2_count": 28,
  "true_hard_after_v2_count": 28,
  "true_hard_after_v2_indices": [
    28,
    29,
    189,
    229,
    250,
    251,
    281,
    290,
    291,
    310,
    312,
    326,
    344,
    381,
    408,
    424,
    467,
    523,
    529,
    667,
    675,
    778,
    781,

# ChartQA Module 23C normalization v2 - 2026-07-03

## 运行口径

本模块只做 evaluator normalization v2：

- 不加载模型；
- 不使用 GPU；
- 不改 prediction；
- 不跑 full-val；
- 只读取 Module 23A 的 per-prediction 结果和 Module 23B 的诊断分组。

新增四类 normalization：

1. color synonyms and hex-to-color names；
2. trend morphology，例如 `increase / increases / increasing`；
3. numeric answer extraction from sentences when questions contain years；
4. list order plus singular/plural variants。

## 结果

| metric | before v2 | after v2 | gain |
|---|---:|---:|---:|
| oracle on clean-after-23A | 30/67 = 44.78% | 34/67 = 50.75% | +4 |

v2 oracle 追回样本：

```text
18, 241, 648, 816
```

按规则：

| rule | unique samples | indices |
|---|---:|---|
| `color_synonym_or_hex` | 2 | `18, 816` |
| `numeric_answer_in_sentence` | 1 | `241` |
| `trend_morphology` | 1 | `648` |

23B true-hard 口径变化：

```text
before v2: 28
after v2:  28
```

v2 后仍建议作为 targeted prompt ablation 的 true-hard 样本：

```text
28, 29, 189, 229, 250, 251, 281, 290, 291, 310, 312, 326, 344, 381, 408, 424, 467, 523, 529, 667, 675, 778, 781, 797, 804, 877, 901, 1055
```

## 解释

Normalization v2 主要用于剥离残留 evaluator 问题，避免 targeted prompt ablation 被颜色名、趋势词、句子数字抽取和列表格式问题污染。后续 prompt ablation 应默认使用 v2 后的 true-hard 样本。


# Module 23C-SUPPLEMENT - dependencies, reconnect, checkpoint, and latest results

This supplement fixes the missing operational details for Module 23C. It does not replace the executed 23C cells above.

## Latest 23C Results Read From The Notebook

The latest notebook run completed:


```text
23C.1 restore: executed
23C.2 normalization v2: executed
23C.3 routed targeted prompt ablation: returncode=0
23C.4 summary read: executed
```


Normalization v2:


```text
before v2: 30/67 = 44.78%
after v2:  34/67 = 50.75%
gain: +4
recovered: 18, 241, 648, 816
```


Routed targeted prompt ablation on 28 true-hard samples:


```text
total predictions: 28
unique samples: 28
oracle recovered: 1
recovered index: 344
```


By prompt:


```text
legend_table_prompt: 0/5
multi_answer_prompt: 0/3
operand_table_prompt: 0/15
range_count_prompt: 0/3
spatial_locator_prompt: 1/2, recovered 344
```


Interpretation:


```text
The first routed prompt ablation is mostly negative. The only clear gain is spatial_locator_prompt on sample 344.
This means simple prompt routing is not enough for most hard failures; the remaining bottleneck is structured extraction / localization / operand correctness, not just final-answer wording.
```


## Fresh Runtime Dependency Cell

Run this after reconnecting to a fresh Colab runtime if imports fail or if the runtime was reset.


In [ ]:
# Module 23C.0-install - dependencies for a fresh Colab runtime
# 中文说明：
# 1. 这个单元只安装依赖，不在同一个进程里 import peft。
# 2. Pillow 在 Colab 中原地升级后容易出现 PIL 新旧文件混用，例如：
#    ImportError: cannot import name '_Ink' from 'PIL._typing'
# 3. 因此本单元执行完成后，请 Runtime -> Restart runtime，再运行 23C.0-check。

%pip install -q --no-cache-dir --force-reinstall "pillow==10.4.0"
%pip install -q -U --no-cache-dir \
    "transformers>=4.51.0" \
    "accelerate>=1.2.0" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "qwen-vl-utils>=0.0.8" \
    "tqdm>=4.66.0" \
    "pandas>=2.2.0"

print("Dependency install finished.")
print("如果本单元刚刚安装/升级了 Pillow：请现在重启 runtime，然后从 23C.0-check 继续。")
print("Colab 菜单：Runtime -> Restart runtime")


In [ ]:
# Module 23C.0-check - post-restart import check
# 中文说明：这个单元必须在重启 runtime 之后运行，用来确认依赖环境干净可用。
import torch
import transformers
import peft
import qwen_vl_utils
from PIL import Image, ImageText

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("pillow:", Image.__version__)
print("qwen_vl_utils import: ok")


## Reconnect And Resume Rules

If Colab disconnects during `23C.3`:


```text
1. Reconnect to a GPU runtime.
2. Run 23C.0 if dependencies/imports fail.
3. Run 23C.1 to restore scripts, inputs, and adapter.
4. Run 23C.checkpoint to inspect completed rows.
5. Run 23C.3 again with the same arguments.
6. Run 23C.4 to read summaries.
```


Do not delete:


```text
outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed.jsonl
```


The script restores the JSONL from Drive if local output is missing and skips completed `(sample_index, prompt_name)` rows. A successful resume should print fewer pending tasks or `pending: 0`.

## Checkpoint Inspection Cell


In [ ]:
# Module 23C.checkpoint - inspect routed prompt-ablation progress
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

local_pred = REPO / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed.jsonl"
drive_pred = DRIVE_ROOT / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed.jsonl"
local_eval = REPO / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed_evaluated.jsonl"
metrics = REPO / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed_metrics.json"
drive_metrics = DRIVE_ROOT / "outputs/chartqa_23c_targeted_prompt_ablation/targeted_prompt_hardmix_routed_metrics.json"

for path in [local_pred, drive_pred, local_eval, metrics, drive_metrics]:
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else None)

if local_pred.exists():
    rows = [json.loads(line) for line in local_pred.read_text(encoding="utf-8").splitlines() if line.strip()]
    done = sorted({(int(row["sample_index"]), row["prompt_name"]) for row in rows})
    print("completed predictions:", len(done))
    print("completed sample indices:", sorted({idx for idx, _ in done}))

if metrics.exists():
    print("\nmetrics:")
    print(metrics.read_text(encoding="utf-8")[:5000])


## Exact Resume Command For Routed Run


In [ ]:
# Module 23C.resume - rerun routed targeted prompt ablation safely
from pathlib import Path
import subprocess
import torch

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this cell. Switch Colab runtime to GPU.")

cmd = [
    "python", "scripts/run_chartqa_23c_targeted_prompt_ablation.py",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--targeted-review-csv", "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
    "--output-dir", "outputs/chartqa_23c_targeted_prompt_ablation",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_23c_targeted_prompt_ablation"),
    "--adapter-path", "outputs/adapters/chartqa_qlora_hardmix1k_steps100",
    "--adapter-name", "hardmix",
    "--prompt-policy", "routed",
    "--max-pixels", "802816",
    "--max-new-tokens", "96",
    "--sync-every", "1",
]
subprocess.run(cmd, check=True)


## Next Diagnostic Recommendation

Do not expand to full-val from this routed prompt result. The gain is only `1/28`, and the only winning family is spatial localization.

Recommended next action:


```text
Run all-prompts only on a smaller priority subset, not all 28 at once:
326, 344, 281, 291, 467, 529, 675, 781, 877, 901
```


The goal should be to identify whether any prompt family helps a specific failure type before spending more GPU time.


# Module 22C/23A/23B - Colab reproduction for local audit modules

This module fills the reproducibility gap for the local-only audit steps that were previously represented mostly as text cells in the notebook.

It does not load the model, does not use GPU, does not train, and does not run full-val. It restores existing Module 21/22B/23A/23B artifacts from Drive, runs the local audit scripts, and writes outputs back to Drive.

Recommended order after reconnect:


```text
22C23.0 -> 22C23.1 -> 22C23.2 -> 22C23.3 -> 22C23.4
```


## 22C23.0 Restore scripts and artifact inputs


In [ ]:
# Module 22C23.0 - restore scripts and existing artifacts for local audit reproduction
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
%cd /content/qwen25vl-chartqa-qlora

SCRIPT_NAMES = [
    "run_chartqa_staged_extraction_quality_audit.py",
    "run_chartqa_23a_cleanup_normalization_ablation.py",
    "run_chartqa_23b_hard_failure_diagnostics.py",
]
SCRIPT_DRIVE_DIRS = [
    DRIVE_ROOT / "scripts_module22c",
    DRIVE_ROOT / "scripts_module23a",
    DRIVE_ROOT / "scripts_module23b",
    DRIVE_ROOT / "scripts_module23c",
]

for script_name in SCRIPT_NAMES:
    local_script = REPO / "scripts" / script_name
    if local_script.exists():
        print("Script already present:", local_script)
        continue
    for drive_dir in SCRIPT_DRIVE_DIRS:
        candidate = drive_dir / script_name
        if candidate.exists():
            local_script.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, local_script)
            print("Restored script:", candidate, "->", local_script)
            break
    else:
        raise FileNotFoundError(f"Missing script in repo and Drive: {script_name}")

def copy_if_missing(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print("Already present:", dst)
        return
    if not src.exists():
        raise FileNotFoundError(f"Missing required Drive file: {src}")
    shutil.copy2(src, dst)
    print("Restored:", src, "->", dst)

copy_if_missing(
    DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/data/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
)
copy_if_missing(
    DRIVE_ROOT / "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv",
    REPO / "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv",
)

# Restore 22B staged extraction files.
LOCAL_STAGED = REPO / "outputs/chartqa_all_wrong_diagnostics_from_drive/staged_extraction"
DRIVE_STAGED = DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/staged_extraction"
LOCAL_STAGED.mkdir(parents=True, exist_ok=True)
if not DRIVE_STAGED.exists():
    raise FileNotFoundError(f"Missing Drive staged extraction directory: {DRIVE_STAGED}")
for src in DRIVE_STAGED.glob("*"):
    if src.is_file():
        shutil.copy2(src, LOCAL_STAGED / src.name)

# Restore Module 21 evaluated JSONL files into the standard local mirror.
LOCAL_EVAL = REPO / "outputs/chartqa_all_wrong_diagnostics_from_drive/evaluated"
LOCAL_EVAL.mkdir(parents=True, exist_ok=True)
needed = {
    "baseline_maxpix_802816_evaluated.jsonl",
    "f_maxpix_802816_evaluated.jsonl",
    "hardmix_axis_legend_prompt_802816_evaluated.jsonl",
    "hardmix_maxpix_602112_evaluated.jsonl",
    "hardmix_maxpix_802816_evaluated.jsonl",
    "image_plus_table_json_evaluated.jsonl",
    "table_json_only_evaluated.jsonl",
}
found = {}
for candidate in (DRIVE_ROOT / "outputs").rglob("*_evaluated.jsonl"):
    if candidate.name in needed and candidate.name not in found:
        found[candidate.name] = candidate
for name in sorted(needed):
    if name not in found:
        raise FileNotFoundError(f"Could not find evaluated JSONL in Drive outputs: {name}")
    shutil.copy2(found[name], LOCAL_EVAL / name)
    print("Restored evaluated:", found[name], "->", LOCAL_EVAL / name)

print("22C/23A/23B reproduction restore complete.")


## 22C23.1 Re-run 22C quality audit


In [ ]:
# Module 22C23.1 - reproduce 22C staged extraction quality audit
import subprocess

cmd = [
    "python", "scripts/run_chartqa_staged_extraction_quality_audit.py",
    "--staged-dir", "outputs/chartqa_all_wrong_diagnostics_from_drive/staged_extraction",
    "--module21-eval-dir", "outputs/chartqa_all_wrong_diagnostics_from_drive/evaluated",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--exclude-csv", "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv",
    "--output-dir", "outputs/chartqa_staged_extraction_quality_audit_22c",
    "--report-md", "docs/experiments/chartqa_staged_extraction_quality_audit_22c_colab_repro.md",
]
subprocess.run(cmd, check=True)


## 22C23.2 Re-run 23A cleanup + normalization-only ablation


In [ ]:
# Module 22C23.2 - reproduce 23A cleanup + normalization-only ablation
import subprocess

cmd = [
    "python", "scripts/run_chartqa_23a_cleanup_normalization_ablation.py",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--exclude-22a-csv", "outputs/chartqa_evaluator_cleanup/chartqa_subset85_exclude_or_fix_reference_list.csv",
    "--module21-eval-dir", "outputs/chartqa_all_wrong_diagnostics_from_drive/evaluated",
    "--module22b-eval-dir", "outputs/chartqa_all_wrong_diagnostics_from_drive/staged_extraction",
    "--output-dir", "outputs/chartqa_23a_cleanup_normalization",
    "--report-md", "docs/experiments/chartqa_23a_cleanup_normalization_colab_repro.md",
]
subprocess.run(cmd, check=True)


## 22C23.3 Re-run 23B hard-failure queue and restore Codex targeted labels


In [ ]:
# Module 22C23.3 - reproduce 23B hard-failure queue and restore Codex targeted review labels
from pathlib import Path
import shutil
import subprocess

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

cmd = [
    "python", "scripts/run_chartqa_23b_hard_failure_diagnostics.py",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--per-prediction-csv", "outputs/chartqa_23a_cleanup_normalization/chartqa_23a_normalization_per_prediction.csv",
    "--output-dir", "outputs/chartqa_23b_hard_failure_diagnostics",
    "--report-md", "docs/experiments/chartqa_23b_hard_failure_diagnostics_colab_repro.md",
]
subprocess.run(cmd, check=True)

# The Codex targeted review is a manual/visual audit artifact. Restore it for downstream 23C/24A.
src = DRIVE_ROOT / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv"
dst = REPO / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv"
if not src.exists():
    raise FileNotFoundError(f"Missing Codex targeted review CSV on Drive: {src}")
shutil.copy2(src, dst)
print("Restored Codex targeted review:", dst)


## 22C23.4 Sync reproduced audit outputs to Drive and read summaries


In [ ]:
# Module 22C23.4 - sync reproduced local-audit outputs and read summaries
from pathlib import Path
import json
import shutil

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

for rel in [
    "outputs/chartqa_staged_extraction_quality_audit_22c",
    "outputs/chartqa_23a_cleanup_normalization",
    "outputs/chartqa_23b_hard_failure_diagnostics",
]:
    local_dir = REPO / rel
    drive_dir = DRIVE_ROOT / rel
    drive_dir.mkdir(parents=True, exist_ok=True)
    for src in local_dir.rglob("*"):
        if src.is_file():
            dst = drive_dir / src.relative_to(local_dir)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print("Synced:", local_dir, "->", drive_dir)

for path in [
    REPO / "outputs/chartqa_staged_extraction_quality_audit_22c/chartqa_22c_quality_audit_summary.json",
    REPO / "outputs/chartqa_23a_cleanup_normalization/chartqa_23a_summary.json",
    REPO / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_summary.json",
]:
    print("\n==", path, "==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)[:4000])
    else:
        print("missing")


# Module 24A - structured intermediate output ablation

Module 24A tests whether forcing a structured intermediate representation helps the 28 true-hard samples that remained after cleanup and normalization.

It does **not** run full-val and does **not** train a new adapter.

Default run:


```text
28 true-hard samples × 1 routed structured prompt = 28 generations
```


Optional `--prompt-policy all` run:


```text
28 true-hard samples × 5 structured prompts = 140 generations
```


## 24A.0 Dependencies and reconnect rules

Run `24A.0-install` only on a fresh Colab runtime, or after an import error.
Pillow is installed in a separate force-reinstall step because Colab can keep old
`PIL` modules in memory after an in-place upgrade. If this cell installs or
upgrades Pillow, restart the runtime before importing `peft` / `transformers`.


In [ ]:
# Module 24A.0-install - dependencies for a fresh Colab runtime
# 中文说明：
# 1. 这个单元只负责安装依赖，不在同一个 Python 进程里 import peft。
# 2. Pillow 在 Colab 中原地升级后容易出现 PIL 新旧文件混用，例如：
#    ImportError: cannot import name '_Ink' from 'PIL._typing'
# 3. 因此本单元执行完成后，请先 Runtime -> Restart runtime，再运行 24A.0-check。

%pip install -q --no-cache-dir --force-reinstall "pillow==10.4.0"
%pip install -q -U --no-cache-dir \
    "transformers>=4.51.0" \
    "accelerate>=1.2.0" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "qwen-vl-utils>=0.0.8" \
    "tqdm>=4.66.0" \
    "pandas>=2.2.0"

print("Dependency install finished.")
print("如果本单元刚刚安装/升级了 Pillow：请现在重启 runtime，然后从 24A.0-check 继续。")
print("Colab 菜单：Runtime -> Restart runtime")


In [ ]:
# Module 24A.0-check - post-restart import check
# 中文说明：这个单元必须在重启 runtime 之后运行，用来确认依赖环境干净可用。
import torch
import transformers
import peft
import qwen_vl_utils
from PIL import Image, ImageText

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("pillow:", Image.__version__)
print("qwen_vl_utils import: ok")


If Colab disconnects during `24A.2`, reconnect to GPU and run:


```text
24A.0-install only if dependencies are missing, then restart runtime
24A.0-check
24A.1 restore
24A.checkpoint inspect progress
24A.2 resume with same arguments
24A.3 read summary
```


The prediction JSONL is append-only and skips completed `(sample_index, prompt_name)` rows.

## 24A.1 Restore scripts, inputs, and adapter


In [ ]:
# Module 24A.1 - restore scripts, inputs, and hardmix adapter
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")
DRIVE_SCRIPT_DIR = DRIVE_ROOT / "scripts_module24a"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "outputs/chartqa_24a_structured_hard_ablation"

%cd /content/qwen25vl-chartqa-qlora

SCRIPT_NAME = "run_chartqa_24a_structured_hard_ablation.py"
LOCAL_SCRIPT = REPO / "scripts" / SCRIPT_NAME
DRIVE_SCRIPT = DRIVE_SCRIPT_DIR / SCRIPT_NAME
if LOCAL_SCRIPT.exists():
    print("Script already present:", LOCAL_SCRIPT)
elif DRIVE_SCRIPT.exists():
    LOCAL_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SCRIPT, LOCAL_SCRIPT)
    print("Restored script:", DRIVE_SCRIPT, "->", LOCAL_SCRIPT)
else:
    raise FileNotFoundError(f"Missing 24A script in repo and Drive: {SCRIPT_NAME}")

def copy_if_missing(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print("Already present:", dst)
        return
    if not src.exists():
        raise FileNotFoundError(f"Missing required Drive file: {src}")
    shutil.copy2(src, dst)
    print("Restored:", src, "->", dst)

copy_if_missing(
    DRIVE_ROOT / "outputs/chartqa_all_wrong_diagnostics/data/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    REPO / "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
)
copy_if_missing(
    DRIVE_ROOT / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
    REPO / "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
)

HARDMIX_ADAPTER = REPO / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
DRIVE_HARDMIX_ADAPTER = DRIVE_ROOT / "outputs/adapters/chartqa_qlora_hardmix1k_steps100"
if HARDMIX_ADAPTER.exists():
    print("Adapter already present:", HARDMIX_ADAPTER)
else:
    if not DRIVE_HARDMIX_ADAPTER.exists():
        raise FileNotFoundError(f"Missing hardmix adapter on Drive: {DRIVE_HARDMIX_ADAPTER}")
    shutil.copytree(DRIVE_HARDMIX_ADAPTER, HARDMIX_ADAPTER)
    print("Restored hardmix adapter:", HARDMIX_ADAPTER)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("24A restore complete.")


## 24A.checkpoint Inspect progress


In [ ]:
# Module 24A.checkpoint - inspect structured ablation checkpoint
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

local_pred = REPO / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed.jsonl"
drive_pred = DRIVE_ROOT / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed.jsonl"
local_eval = REPO / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed_evaluated.jsonl"
metrics = REPO / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed_metrics.json"

for path in [local_pred, drive_pred, local_eval, metrics]:
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else None)

if local_pred.exists():
    rows = [json.loads(line) for line in local_pred.read_text(encoding="utf-8").splitlines() if line.strip()]
    done = sorted({(int(row["sample_index"]), row["prompt_name"]) for row in rows})
    print("completed predictions:", len(done))
    print("completed sample indices:", sorted({idx for idx, _ in done}))

if metrics.exists():
    print(metrics.read_text(encoding="utf-8")[:5000])


## 24A.2 Run routed structured ablation


In [ ]:
# Module 24A.2 - run routed structured intermediate ablation on 28 true-hard samples
from pathlib import Path
import subprocess
import torch

REPO = Path("/content/qwen25vl-chartqa-qlora")
DRIVE_ROOT = Path("/content/drive/MyDrive/qwen25vl-chartqa-qlora")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for 24A. Switch Colab runtime to GPU.")

cmd = [
    "python", "scripts/run_chartqa_24a_structured_hard_ablation.py",
    "--subset-jsonl", "data/diagnostics/chartqa_all_wrong_diagnostic_subset_85.jsonl",
    "--targeted-review-csv", "outputs/chartqa_23b_hard_failure_diagnostics/chartqa_23b_codex_targeted_review.csv",
    "--output-dir", "outputs/chartqa_24a_structured_hard_ablation",
    "--drive-output-dir", str(DRIVE_ROOT / "outputs/chartqa_24a_structured_hard_ablation"),
    "--adapter-path", "outputs/adapters/chartqa_qlora_hardmix1k_steps100",
    "--adapter-name", "hardmix",
    "--prompt-policy", "routed",
    "--max-pixels", "802816",
    "--max-new-tokens", "256",
    "--sync-every", "1",
]
subprocess.run(cmd, check=True)


## 24A.3 Read summary


In [ ]:
# Module 24A.3 - read structured ablation summary
from pathlib import Path
import json

REPO = Path("/content/qwen25vl-chartqa-qlora")
metrics = REPO / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed_metrics.json"
evaluated = REPO / "outputs/chartqa_24a_structured_hard_ablation/structured_24a_hardmix_routed_evaluated.jsonl"

print("metrics:", metrics, "exists=", metrics.exists())
if metrics.exists():
    print(json.dumps(json.loads(metrics.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)[:6000])

if evaluated.exists():
    rows = [json.loads(line) for line in evaluated.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("evaluated rows:", len(rows))
    print("recovered:")
    for row in rows:
        if row.get("eval_normalization_v2_correct"):
            print(row["sample_index"], row["prompt_name"], "answer=", row.get("parsed_final_answer"))
